In [1]:
import torch

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("Total VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2), "GB")

    free, total = torch.cuda.mem_get_info()
    print("Free VRAM:", round(free / 1024**3, 2), "GB")
else:
    print("WARNING: CUDA GPU not detected")

CUDA available: True
GPU: Tesla T4
Total VRAM: 14.56 GB
Free VRAM: 14.46 GB


In [2]:
!pip -q install -U transformers accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 21.5 MB/s eta 0:00:00


In [3]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_NAME = "Qwen/Qwen3-4B"

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print("Loading APEX:", MODEL_NAME)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=quant_config,
    device_map="auto",
)

print("\n✅ APEX model loaded successfully!")
print("Device:", model.device)

free, total = torch.cuda.mem_get_info()
print(f"Free VRAM: {free / 1024**3:.2f} GB")
print(f"Used VRAM: {(total - free) / 1024**3:.2f} GB")

Loading APEX: Qwen/Qwen3-4B


config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]


✅ APEX model loaded successfully!
Device: cuda:0
Free VRAM: 11.91 GB
Used VRAM: 2.66 GB


In [4]:
import torch

prompt = """You are APEX, an industrial troubleshooting AI.

A DeltaWorks DX-200 press brake intermittently stops during backgauge deceleration.
The machine sometimes reports fault F001.

Manual evidence:
- F001 means DC Link Voltage Too High.
- Measure mains at the main isolator terminals.
- Required mains voltage: 400 V ±10% (360–440 V).
- If mains voltage is high, notify the facility electrician.
- Power down and apply LOTO before internal electrical measurements.
- Measure the braking resistor between R+ and RB.
- Nominal braking resistor resistance: 47 Ω ±10%.
- Replace the resistor if it is open or outside tolerance.
- If the braking resistor is healthy, increase BG.DECEL by 25%.
- If the fault persists, request DeltaWorks service for a DC-link capacitor capacitance test.

Question:
What should the technician verify first, and what is the likely troubleshooting sequence?
"""

messages = [
    {
        "role": "system",
        "content": "You are a careful industrial troubleshooting assistant. Use only the supplied evidence. Do not invent facts."
    },
    {
        "role": "user",
        "content": prompt
    }
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(text, return_tensors="pt").to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=500,
        temperature=0.2,
        do_sample=True,
    )

response = tokenizer.decode(
    outputs[0][inputs["input_ids"].shape[1]:],
    skip_special_tokens=True
)

print(response)

<think>
Okay, let's see. The problem is with the DeltaWorks DX-200 press brake intermittently stopping during backgauge deceleration and fault F001. The manual says F001 is DC Link Voltage Too High. So, the first step is to check the mains voltage at the main isolator terminals. The required voltage is 400V ±10%, so 360-440V. If the mains voltage is high, they need to notify the electrician.

But the fault is intermittent. So maybe the issue isn't just the mains voltage but something else. The manual also mentions checking the braking resistor. If the resistor is open or out of tolerance, replace it. If it's okay, then increase BG.DECEL by 25%. If that doesn't work, check the DC-link capacitor.

So the technician should first verify the mains voltage. If that's okay, then check the braking resistor. If the resistor is fine, adjust BG.DECEL. If the problem persists, check the capacitor. But since the fault is intermittent, maybe there's a voltage spike or something causing the resistor 

In [5]:
prompt = """You are APEX, an industrial troubleshooting reasoning model.

Use ONLY the supplied evidence. Do not introduce causes, mechanisms,
recommendations, or technical facts that are not supported by the evidence.

Return your answer using EXACTLY this structure:

MACHINE:
FAULTS:
EVIDENCE:
REASONING:
PRIORITY_CHECKS:
SAFETY:
CONCLUSION:
CONFIDENCE:
UNSUPPORTED_CLAIMS:

Evidence:
Machine: DeltaWorks DX-200 Automated Press Brake

F001:
- DC Link Voltage Too High.
- Measure mains at the main isolator terminals.
- Required: 400 V ±10% (360–440 V).
- If high, notify the facility electrician.
- Power down and apply LOTO before internal electrical measurements.
- Measure braking resistor between R+ and RB.
- Nominal resistance: 47 Ω ±10%.
- Replace if open or out of tolerance.
- If healthy, increase BG.DECEL by 25%.
- If fault persists, request DeltaWorks service for a DC-link capacitor capacitance test.

User problem:
"The press brake intermittently stops during backgauge deceleration and sometimes shows F001."

Question:
What should be checked first and what troubleshooting sequence is supported by the evidence?
"""

messages = [
    {
        "role": "system",
        "content": "You are APEX. Be evidence-grounded. Never invent technical facts."
    },
    {
        "role": "user",
        "content": prompt
    }
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(text, return_tensors="pt").to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=600,
        temperature=0.1,
        do_sample=True,
    )

response = tokenizer.decode(
    outputs[0][inputs["input_ids"].shape[1]:],
    skip_special_tokens=True
)

print(response)

<think>
Okay, let's tackle this problem. The user is dealing with a DeltaWorks DX-200 Automated Press Brake that has an intermittent stop during backgauge deceleration and shows F001. The first step is to figure out what the evidence says and how to apply it.

Looking at the evidence, F001 is related to DC Link Voltage Too High. The steps mention checking the mains voltage at the main isolator terminals, which should be 400V ±10%. If it's high, notify the electrician. Also, check the braking resistor between R+ and RB. The nominal resistance is 47Ω ±10%, so if it's open or out of tolerance, replace it. If the resistor is okay, increase BG.DECEL by 25%. If the problem continues, DeltaWorks service should be contacted for a DC-link capacitor test.

The user's problem is intermittent stops during deceleration and F001. The first thing to check would be the DC link voltage. Since the fault is intermittent, maybe there's a voltage fluctuation. But the evidence says to check the mains voltag

In [6]:
import torch
from peft import prepare_model_for_kbit_training, LoraConfig, get_peft_model

print("Testing QLoRA preparation...")

model = prepare_model_for_kbit_training(model)

print("✅ Model prepared for k-bit training")

print("\nTrainable adapter configuration will be added next.")
print("GPU:", torch.cuda.get_device_name(0))
print("Free VRAM:", round(torch.cuda.mem_get_info()[0] / 1024**3, 2), "GB")

Testing QLoRA preparation...
✅ Model prepared for k-bit training

Trainable adapter configuration will be added next.
GPU: Tesla T4
Free VRAM: 10.44 GB


In [7]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
)

model = get_peft_model(model, lora_config)

model.print_trainable_parameters()

trainable params: 33,030,144 || all params: 4,055,498,240 || trainable%: 0.8145


In [8]:
from datasets import Dataset

training_data = [
    {
        "messages": [
            {
                "role": "system",
                "content": """You are APEX, an industrial troubleshooting AI.
Use only the supplied evidence.
Separate documented facts from inference.
Never invent technical facts.
Preserve all safety requirements."""
            },
            {
                "role": "user",
                "content": """Machine: DeltaWorks DX-200 Automated Press Brake

Problem:
The machine intermittently stops during backgauge deceleration and sometimes shows F001.

Evidence:
F001 means DC Link Voltage Too High.
Required mains voltage is 400 V ±10% (360–440 V).
If mains voltage is high, notify the facility electrician.
Power down and apply LOTO before internal electrical measurements.
Braking resistor between R+ and RB should be 47 Ω ±10%.
Replace the resistor if open or out of tolerance.
If the resistor is healthy, increase BG.DECEL by 25%.
If the fault persists, request DeltaWorks service for a DC-link capacitor capacitance test.

Question:
What should be checked first?"""
            },
            {
                "role": "assistant",
                "content": """MACHINE:
DeltaWorks DX-200 Automated Press Brake

FAULT:
F001 — DC Link Voltage Too High

PRIORITY_CHECKS:
1. Verify mains voltage at the main isolator terminals.
   Required range: 360–440 V.
2. If mains voltage is high, notify the facility electrician.
3. Power down and apply LOTO before internal electrical measurements.
4. Measure the braking resistor between R+ and RB.
   Required resistance: 47 Ω ±10%.
5. Replace the resistor if it is open or outside tolerance.
6. If the resistor is healthy, increase BG.DECEL by 25%.
7. If the fault persists, request DeltaWorks service for a DC-link capacitor capacitance test.

SAFETY:
Apply LOTO before internal electrical measurements.

CONCLUSION:
The first documented check is the mains voltage. Further checks should follow the documented sequence.

CONFIDENCE:
High — the troubleshooting sequence is explicitly provided by the manual.

UNSUPPORTED_CLAIMS:
No additional root cause is asserted because the supplied evidence does not establish one."""
            }
        ]
    }
]

dataset = Dataset.from_list(training_data)

print(dataset)
print("\nNumber of examples:", len(dataset))
print("\nExample structure:")
print(dataset[0])


Dataset({
    features: ['messages'],
    num_rows: 1
})

Number of examples: 1

Example structure:
{'messages': [{'content': 'You are APEX, an industrial troubleshooting AI.\nUse only the supplied evidence.\nSeparate documented facts from inference.\nNever invent technical facts.\nPreserve all safety requirements.', 'role': 'system'}, {'content': 'Machine: DeltaWorks DX-200 Automated Press Brake\n\nProblem:\nThe machine intermittently stops during backgauge deceleration and sometimes shows F001.\n\nEvidence:\nF001 means DC Link Voltage Too High.\nRequired mains voltage is 400 V ±10% (360–440 V).\nIf mains voltage is high, notify the facility electrician.\nPower down and apply LOTO before internal electrical measurements.\nBraking resistor between R+ and RB should be 47 Ω ±10%.\nReplace the resistor if open or out of tolerance.\nIf the resistor is healthy, increase BG.DECEL by 25%.\nIf the fault persists, request DeltaWorks service for a DC-link capacitor capacitance test.\n\nQuestion:

In [9]:
def format_example(example):
    text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False
    )
    return {"text": text}

formatted_dataset = dataset.map(format_example)

print("✅ Dataset formatted")
print("\nColumns:", formatted_dataset.column_names)
print("\nFormatted example:\n")
print(formatted_dataset[0]["text"][:5000])

Map:   0%|          | 0/1 [00:00<?, ? examples/s]

✅ Dataset formatted

Columns: ['messages', 'text']

Formatted example:

<|im_start|>system
You are APEX, an industrial troubleshooting AI.
Use only the supplied evidence.
Separate documented facts from inference.
Never invent technical facts.
Preserve all safety requirements.<|im_end|>
<|im_start|>user
Machine: DeltaWorks DX-200 Automated Press Brake

Problem:
The machine intermittently stops during backgauge deceleration and sometimes shows F001.

Evidence:
F001 means DC Link Voltage Too High.
Required mains voltage is 400 V ±10% (360–440 V).
If mains voltage is high, notify the facility electrician.
Power down and apply LOTO before internal electrical measurements.
Braking resistor between R+ and RB should be 47 Ω ±10%.
Replace the resistor if open or out of tolerance.
If the resistor is healthy, increase BG.DECEL by 25%.
If the fault persists, request DeltaWorks service for a DC-link capacitor capacitance test.

Question:
What should be checked first?<|im_end|>
<|im_start|>assistant

In [10]:
tokens = tokenizer(
    formatted_dataset[0]["text"],
    return_tensors="pt"
)

token_count = tokens["input_ids"].shape[1]

print("Token count:", token_count)
print("Model max context:", tokenizer.model_max_length)

print("\nFirst 20 token IDs:")
print(tokens["input_ids"][0][:20].tolist())

print("\nLast 20 token IDs:")
print(tokens["input_ids"][0][-20:].tolist())

Token count: 451
Model max context: 131072

First 20 token IDs:
[151644, 8948, 198, 2610, 525, 362, 1740, 55, 11, 458, 12785, 68671, 15235, 624, 10253, 1172, 279, 17221, 5904, 624]

Last 20 token IDs:
[14323, 50, 510, 2753, 5107, 3704, 5240, 374, 49597, 1576, 279, 17221, 5904, 1558, 537, 5695, 825, 13, 151645, 198]


In [11]:
def tokenize_example(example):
    messages = example["messages"]

    # Full conversation
    full_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False
    )

    # Tokenize full conversation
    tokenized = tokenizer(
        full_text,
        truncation=True,
        max_length=2048,
        padding=False,
    )

    tokenized["labels"] = tokenized["input_ids"].copy()

    return tokenized


tokenized_dataset = dataset.map(
    tokenize_example,
    remove_columns=dataset.column_names
)

print("✅ Training example tokenized")
print("Columns:", tokenized_dataset.column_names)
print("Token count:", len(tokenized_dataset[0]["input_ids"]))
print("Label count:", len(tokenized_dataset[0]["labels"]))

Map:   0%|          | 0/1 [00:00<?, ? examples/s]

✅ Training example tokenized
Columns: ['input_ids', 'attention_mask', 'labels']
Token count: 451
Label count: 451


In [12]:
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

training_args = TrainingArguments(
    output_dir="./apex_smoke_test",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=1,
    learning_rate=2e-4,
    num_train_epochs=1,
    logging_steps=1,
    save_strategy="no",
    report_to="none",
    fp16=True,
    optim="paged_adamw_8bit",
)

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator,
)

print("Starting APEX QLoRA smoke test...")

trainer.train()

print("\n✅ APEX QLoRA training smoke test completed!")

Starting APEX QLoRA smoke test...


/usr/local/lib/python3.13/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
1,3.320071



✅ APEX QLoRA training smoke test completed!


In [13]:
def prepare_assistant_only_example(example):
    messages = example["messages"]

    # Full conversation
    full_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False
    )

    # Conversation up to (but NOT including) assistant answer
    prompt_text = tokenizer.apply_chat_template(
        messages[:-1],
        tokenize=False,
        add_generation_prompt=True
    )

    # Assistant response only
    assistant_text = messages[-1]["content"] + tokenizer.eos_token

    # Tokenize separately
    prompt_tokens = tokenizer(
        prompt_text,
        add_special_tokens=False
    )["input_ids"]

    assistant_tokens = tokenizer(
        assistant_text,
        add_special_tokens=False
    )["input_ids"]

    # Combine
    input_ids = prompt_tokens + assistant_tokens

    # -100 means "ignore this token when calculating loss"
    labels = (
        [-100] * len(prompt_tokens)
        + assistant_tokens
    )

    return {
        "input_ids": input_ids,
        "attention_mask": [1] * len(input_ids),
        "labels": labels,
    }


assistant_only_dataset = dataset.map(
    prepare_assistant_only_example,
    remove_columns=dataset.column_names
)

example = assistant_only_dataset[0]

print("✅ Assistant-only training example prepared")
print("Total tokens:", len(example["input_ids"]))

trainable_tokens = sum(
    1 for x in example["labels"] if x != -100
)

ignored_tokens = sum(
    1 for x in example["labels"] if x == -100
)

print("Ignored prompt/evidence tokens:", ignored_tokens)
print("Trainable assistant tokens:", trainable_tokens)

print("\nLabel preview:")
print(example["labels"][:50])

Map:   0%|          | 0/1 [00:00<?, ? examples/s]

✅ Assistant-only training example prepared
Total tokens: 446
Ignored prompt/evidence tokens: 214
Trainable assistant tokens: 232

Label preview:
[-100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100]


In [14]:
labels = assistant_only_dataset[0]["labels"]

# Find where training begins
first_trainable = next(
    i for i, label in enumerate(labels)
    if label != -100
)

print("First trainable token position:", first_trainable)
print("Ignored tokens before it:", first_trainable)
print("Trainable tokens after it:", len(labels) - first_trainable)

print("\nLabels around the transition:")
print(labels[first_trainable - 10:first_trainable + 20])

print("\nDecoded first trainable tokens:")
print(
    tokenizer.decode(
        labels[first_trainable:first_trainable + 30]
    )
)

First trainable token position: 214
Ignored tokens before it: 214
Trainable tokens after it: 232

Labels around the transition:
[-100, -100, -100, -100, -100, -100, -100, -100, -100, -100, 44, 41234, 510, 20277, 37683, 30808, 12, 17, 15, 15, 94723, 8445, 75261, 271, 5291, 510, 37, 15, 15, 16]

Decoded first trainable tokens:
MACHINE:
DeltaWorks DX-200 Automated Press Brake

FAULT:
F001 — DC Link Voltage Too High

PRIORITY_CHECK


In [15]:
from pypdf import PdfReader

PDF_PATH = "test_manual_delta.pdf"

reader = PdfReader(PDF_PATH)

print("Pages:", len(reader.pages))
print("=" * 70)

for page_num, page in enumerate(reader.pages, start=1):
    text = page.extract_text() or ""

    if "F001" in text or "F002" in text or "F003" in text or "F005" in text:
        print(f"\n--- PAGE {page_num} ---")
        print(text[:5000])

ModuleNotFoundError: No module named 'pypdf'

In [17]:
import os

PDF_PATH = "/content/test_manual_delta.pdf"

print("Exists:", os.path.exists(PDF_PATH))

if os.path.exists(PDF_PATH):
    print("File size:", round(os.path.getsize(PDF_PATH) / 1024, 2), "KB")
else:
    print("❌ PDF not found in /content")
    print("Upload test_manual_delta.pdf to Colab and run this cell again.")

Exists: True
File size: 371.56 KB


In [18]:
!pip -q install pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 388.1/388.1 kB 10.1 MB/s eta 0:00:00


In [19]:
from pypdf import PdfReader

PDF_PATH = "/content/test_manual_delta.pdf"

reader = PdfReader(PDF_PATH)

print("=" * 70)
print("DELTA DX-200 MANUAL")
print("=" * 70)
print("Pages:", len(reader.pages))

full_text = ""

for page_num, page in enumerate(reader.pages, start=1):
    text = page.extract_text() or ""
    full_text += f"\n\n===== PAGE {page_num} =====\n{text}"

print("Characters extracted:", len(full_text))

# Show the beginning so we know extraction worked
print("\nFIRST 3000 CHARACTERS:\n")
print(full_text[:3000])

DELTA DX-200 MANUAL
Pages: 15
Characters extracted: 26230

FIRST 3000 CHARACTERS:



===== PAGE 1 =====
Machine Delta DX-200 
Operator and Maintenance Manual 
Manufacturer: DeltaWorks Industries 
Model: DX-200 Automated Press Brake 
Document Number: DW-MAN-DX200-REV2 
Issue Date: 2024-11-01 
Controller Firmware: 5.0.4 
 
1. Specifications 
Parameter Value 
Press Force (Nominal) 200 tonnes 
Bending Length 2,500 mm 
Ram Stroke 250 mm 
Backgauge Axes X, R, Z1, Z2 (4-axis CNC) 
Mains Supply 400 V AC, 3-phase, 50 Hz, 63 A 
 
 
2. Safety 
 Only trained and authorised operators may use the DX-200. Minimum age 18 years. 
 The DX-200 uses a hydraulic ram producing up to 200 tonnes of force — never place hands 
between the top and bottom tools. 
 The front safety light curtain (Type 4, Category 4 per ISO 13849) must never be defeated or 
bypassed. 
 Before die changes, isolate the hydraulic pump and apply LOTO to the main isolator. 
 Wear cut-resistant gloves when handling sheet metal; wear

In [20]:
import re

# Find every fault code and its surrounding text
fault_pattern = r'(?m)^(F\d{3})\s*[—-]\s*(.+)$'

matches = list(re.finditer(fault_pattern, full_text))

print("=" * 70)
print("FAULT CODE INDEX")
print("=" * 70)
print("Fault codes found:", len(matches))
print()

fault_codes = []

for i, match in enumerate(matches):
    code = match.group(1)
    title = match.group(2).strip()

    start = match.start()
    end = matches[i + 1].start() if i + 1 < len(matches) else len(full_text)

    section = full_text[start:end].strip()

    fault_codes.append({
        "code": code,
        "title": title,
        "section": section
    })

    print(f"{code} — {title}")

print("\n" + "=" * 70)
print("FIRST FAULT SECTION")
print("=" * 70)
print(fault_codes[0]["section"][:4000])

FAULT CODE INDEX
Fault codes found: 21

F001 — DC Link Voltage Too High
F002 — DC Link Voltage Too Low
F003 — Mains Overvoltage
F005 — HPU Motor Overtemperature
F007 — Backgauge Axis X Following Error
F010 — Ram Angle Measurement Discrepancy
F011 — Temperature Sensor Short Circuit
F015 — Hydraulic Oil Temperature High
F020 — Light Curtain Fault
F025 — Crowning Axis Fault
F030 — Door Safety Switch Open
F035 — Foot Pedal Fault
F040 — Backgauge Home Position Loss
F045 — Tool Table Data Corrupt
F050 — Bend Angle Sensor Fault
F055 — HPU Pressure Relief Valve Open
F060 — Axis R (Ram Height) Out of Range
F070 — Network Communication Fault
F080 — Safety Relay Fault
F090 — Job Program CRC Error
F099 — Power Supply Phase Failure

FIRST FAULT SECTION
F001 — DC Link Voltage Too High 
Description: The DC link bus inside the servo amplifier powering the backgauge axes has risen 
above 780 V DC. This is distinct from mains overvoltage (see F003). The condition is caused by 
regenerative energy return

In [21]:
for fault in fault_codes:
    print("=" * 80)
    print(f"{fault['code']} — {fault['title']}")
    print("=" * 80)
    print(fault["section"][:1800])
    print()

F001 — DC Link Voltage Too High
F001 — DC Link Voltage Too High 
Description: The DC link bus inside the servo amplifier powering the backgauge axes has risen 
above 780 V DC. This is distinct from mains overvoltage (see F003). The condition is caused by 
regenerative energy returning from a rapidly decelerating backgauge axis being unable to dissipate 
fast enough. The servo amplifier shuts down all backgauge axes to protect the power stage. 
Probable Causes: 
 The braking resistor on the servo amplifier has gone open-circuit or its connecting cable is 
broken. 
 The backgauge deceleration ramp (parameter BG.DECEL in the drive menu) is set too 
aggressively short. 

===== PAGE 4 =====
 Mains supply voltage is already at the high end of tolerance (>440 V AC) before 
regeneration occurs. 
Corrective Steps: 
1. Measure the mains supply at the main isolator terminals — must be 400 V ±10% (360–440 
V). If high, notify the facility electrician before proceeding. 
2. Power down (LOTO) and

In [22]:
import pandas as pd

fault_df = pd.DataFrame(fault_codes)

print("Shape:", fault_df.shape)
print()
print(fault_df[["code", "title"]].to_string(index=False))

print("\n" + "=" * 70)
print("COLUMN CHECK")
print("=" * 70)

print(fault_df.columns.tolist())
print("Total fault records:", len(fault_df))
print("Unique codes:", fault_df["code"].nunique())
print("Missing sections:", fault_df["section"].isna().sum())

Shape: (21, 3)

code                             title
F001          DC Link Voltage Too High
F002           DC Link Voltage Too Low
F003                 Mains Overvoltage
F005         HPU Motor Overtemperature
F007  Backgauge Axis X Following Error
F010 Ram Angle Measurement Discrepancy
F011  Temperature Sensor Short Circuit
F015    Hydraulic Oil Temperature High
F020               Light Curtain Fault
F025               Crowning Axis Fault
F030           Door Safety Switch Open
F035                  Foot Pedal Fault
F040      Backgauge Home Position Loss
F045           Tool Table Data Corrupt
F050           Bend Angle Sensor Fault
F055    HPU Pressure Relief Valve Open
F060  Axis R (Ram Height) Out of Range
F070       Network Communication Fault
F080                Safety Relay Fault
F090             Job Program CRC Error
F099        Power Supply Phase Failure

COLUMN CHECK
['code', 'title', 'section']
Total fault records: 21
Unique codes: 21
Missing sections: 0


In [23]:
training_examples = [
    {
        "messages": [
            {
                "role": "system",
                "content": (
                    "You are APEX, an industrial troubleshooting AI. "
                    "Use only supplied evidence. "
                    "Separate documented facts from evidence-based inference. "
                    "Never invent technical facts, measurements, causes, or procedures. "
                    "Preserve all safety requirements from the evidence."
                )
            },
            {
                "role": "user",
                "content": f"""
Machine: DeltaWorks DX-200 Automated Press Brake

Observed fault: F001

Question:
The machine shows F001. What should the technician check first, and why?

Evidence:
{fault_df.loc[fault_df["code"] == "F001", "section"].iloc[0]}
"""
            },
            {
                "role": "assistant",
                "content": """FAULT: F001 — DC Link Voltage Too High

FIRST CHECK:
Measure the mains supply at the main isolator terminals.

EXPECTED VALUE:
400 V ±10%, which is 360–440 V AC.

WHY:
The manual identifies mains voltage above 440 V as one possible contributor to F001. The manual also states that F001 is distinct from mains overvoltage F003.

NEXT STEP:
If mains voltage is within the specified range, power down under LOTO and measure the braking resistor at servo amplifier terminals R+ and RB. The nominal resistance is 47 Ω ±10%.

SAFETY:
If mains voltage is high, notify the facility electrician before proceeding. Apply LOTO before measuring the braking resistor.

CONCLUSION:
The manual's prescribed troubleshooting sequence starts with verifying mains voltage, followed by checking the braking resistor if the mains supply is normal.

CONFIDENCE: High"""
            }
        ]
    },

    {
        "messages": [
            {
                "role": "system",
                "content": (
                    "You are APEX, an industrial troubleshooting AI. "
                    "Use only supplied evidence. "
                    "Distinguish similar fault conditions using explicit evidence. "
                    "Never merge separate fault definitions."
                )
            },
            {
                "role": "user",
                "content": f"""
Machine: DeltaWorks DX-200 Automated Press Brake

Observed condition:
The DC link voltage is above 780 V DC.

Question:
Is this necessarily the same as a mains overvoltage condition?

Evidence:
F001:
{fault_df.loc[fault_df["code"] == "F001", "section"].iloc[0]}

F003:
{fault_df.loc[fault_df["code"] == "F003", "section"].iloc[0]}
"""
            },
            {
                "role": "assistant",
                "content": """ASSESSMENT:

No. The evidence explicitly distinguishes F001 from F003.

F001:
The DC link bus inside the servo amplifier has risen above 780 V DC. The manual associates this with regenerative energy from a rapidly decelerating backgauge axis being unable to dissipate quickly enough.

F003:
The incoming mains supply itself has exceeded 440 V AC for more than 500 ms.

IMPORTANT DISTINCTION:
A high DC link voltage does not by itself prove that the incoming mains supply is overvoltage.

EVIDENCE-BASED CONCLUSION:
If the observed condition is specifically a DC link voltage above 780 V DC, F001 is the relevant fault definition. Mains voltage should still be measured because the F001 troubleshooting procedure identifies mains voltage above 440 V as one possible contributing condition.

CONFIDENCE: High"""
            }
        ]
    },

    {
        "messages": [
            {
                "role": "system",
                "content": (
                    "You are APEX, an industrial troubleshooting AI. "
                    "If the supplied evidence is insufficient to determine a root cause, "
                    "say so explicitly. Do not guess."
                )
            },
            {
                "role": "user",
                "content": """
Machine: DeltaWorks DX-200 Automated Press Brake

Observed fault: F001

Question:
The operator says the machine stopped suddenly. Which exact component has failed?

Evidence:
Only the fault code F001 is available. No measurements, diagnostic results, or inspection results are provided.
"""
            },
            {
                "role": "assistant",
                "content": """ASSESSMENT:

The exact failed component cannot be determined from the supplied evidence.

F001 identifies a DC link voltage condition, but the fault code alone does not establish which underlying cause is present.

POSSIBLE CAUSES:
The supplied evidence does not identify a confirmed root cause. Additional diagnostic evidence is required before naming a failed component.

NEXT REQUIRED INFORMATION:
1. Mains voltage measurement.
2. Braking resistor resistance measurement.
3. Current BG.DECEL parameter value.
4. Whether the fault persists with normal mains voltage and a healthy braking resistor.

CONCLUSION:
Do not declare a specific component failed based only on F001. The available evidence is insufficient.

CONFIDENCE: Low"""
            }
        ]
    }
]

training_dataset = Dataset.from_list(training_examples)

print(training_dataset)
print("Training examples:", len(training_dataset))
print()

for i, example in enumerate(training_dataset):
    print(f"Example {i+1}: {len(example['messages'])} messages")

Dataset({
    features: ['messages'],
    num_rows: 3
})
Training examples: 3

Example 1: 3 messages
Example 2: 3 messages
Example 3: 3 messages


In [25]:
def get_fault_section(code):
    row = fault_df[fault_df["code"] == code]
    if len(row) == 0:
        raise ValueError(f"Fault code not found: {code}")
    return row.iloc[0]["section"]


for i, scenario in enumerate(scenario_templates, 1):
    print("=" * 80)
    print(f"SCENARIO {i}: {scenario['type']}")
    print(f"Primary fault: {scenario['fault']}")

    if "related_fault" in scenario:
        print(f"Related fault: {scenario['related_fault']}")

    print("\nQUESTION:")
    print(scenario["question"])

    print("\nEVIDENCE:")
    print(get_fault_section(scenario["fault"])[:1200])

    if "related_fault" in scenario:
        print("\n--- RELATED FAULT EVIDENCE ---")
        print(get_fault_section(scenario["related_fault"])[:1200])

    print()

SCENARIO 1: direct_diagnosis
Primary fault: F001

QUESTION:
What should be checked first, and what is the prescribed troubleshooting sequence?

EVIDENCE:
F001 — DC Link Voltage Too High 
Description: The DC link bus inside the servo amplifier powering the backgauge axes has risen 
above 780 V DC. This is distinct from mains overvoltage (see F003). The condition is caused by 
regenerative energy returning from a rapidly decelerating backgauge axis being unable to dissipate 
fast enough. The servo amplifier shuts down all backgauge axes to protect the power stage. 
Probable Causes: 
 The braking resistor on the servo amplifier has gone open-circuit or its connecting cable is 
broken. 
 The backgauge deceleration ramp (parameter BG.DECEL in the drive menu) is set too 
aggressively short. 

===== PAGE 4 =====
 Mains supply voltage is already at the high end of tolerance (>440 V AC) before 
regeneration occurs. 
Corrective Steps: 
1. Measure the mains supply at the main isolator terminal

In [26]:
def build_training_prompt(scenario):
    primary = scenario["fault"]
    question = scenario["question"]

    evidence = get_fault_section(primary)

    prompt = f"""Machine: DeltaWorks DX-200 Automated Press Brake

Observed fault: {primary}

Question:
{question}

Evidence:
{evidence}
"""

    if "related_fault" in scenario:
        related = scenario["related_fault"]
        prompt += f"""

Additional related fault evidence:

{related}:
{get_fault_section(related)}
"""

    return prompt.strip()


training_prompts = []

for scenario in scenario_templates:
    training_prompts.append({
        "type": scenario["type"],
        "fault": scenario["fault"],
        "related_fault": scenario.get("related_fault"),
        "prompt": build_training_prompt(scenario)
    })


print("Training prompts created:", len(training_prompts))
print()

for i, item in enumerate(training_prompts[:3], 1):
    print("=" * 80)
    print(f"PROMPT {i} | {item['type']}")
    print("=" * 80)
    print(item["prompt"][:2500])
    print()

Training prompts created: 18

PROMPT 1 | direct_diagnosis
Machine: DeltaWorks DX-200 Automated Press Brake

Observed fault: F001

Question:
What should be checked first, and what is the prescribed troubleshooting sequence?

Evidence:
F001 — DC Link Voltage Too High 
Description: The DC link bus inside the servo amplifier powering the backgauge axes has risen 
above 780 V DC. This is distinct from mains overvoltage (see F003). The condition is caused by 
regenerative energy returning from a rapidly decelerating backgauge axis being unable to dissipate 
fast enough. The servo amplifier shuts down all backgauge axes to protect the power stage. 
Probable Causes: 
 The braking resistor on the servo amplifier has gone open-circuit or its connecting cable is 
broken. 
 The backgauge deceleration ramp (parameter BG.DECEL in the drive menu) is set too 
aggressively short. 

===== PAGE 4 =====
 Mains supply voltage is already at the high end of tolerance (>440 V AC) before 
regeneration occur

In [27]:
def build_grounded_answer(scenario):
    fault = scenario["fault"]
    fault_data = fault_df[fault_df["code"] == fault].iloc[0]

    title = fault_data["title"]
    section = fault_data["section"]

    # These are behavioral answer templates.
    # Technical facts come from the retrieved manual section.

    if scenario["type"] == "direct_diagnosis":
        answer = f"""FAULT: {fault} — {title}

ASSESSMENT:
The fault indicates the condition described in the supplied manual evidence.

DOCUMENTED CAUSES:
The manual lists the probable causes in the supplied evidence.

CORRECTIVE SEQUENCE:
Follow the corrective steps in the order specified by the manual.

SAFETY:
Follow all LOTO, electrical, hydraulic, and other safety requirements explicitly stated in the evidence.

CONCLUSION:
The diagnosis should be based on the documented measurements and checks rather than assuming a single failed component.

CONFIDENCE: High"""

    elif scenario["type"] == "fault_disambiguation":
        related = scenario["related_fault"]
        related_data = fault_df[fault_df["code"] == related].iloc[0]

        answer = f"""FAULT DISTINCTION:

{fault} — {title}
The primary evidence describes the condition and causes associated with {fault}.

{related} — {related_data["title"]}
The related evidence describes a separate fault condition.

KEY POINT:
Do not merge the two fault definitions. Use the measurements and conditions specified in the manual to determine which condition is actually present.

CONCLUSION:
The observed measurement should be matched against the explicit fault definitions before selecting the troubleshooting path.

CONFIDENCE: High"""

    elif scenario["type"] == "insufficient_evidence":
        answer = f"""ASSESSMENT:

The exact failed component cannot be determined from {fault} alone.

The supplied manual identifies multiple possible causes and provides a sequence of diagnostic checks. A fault code identifies a condition; it does not by itself prove which component has failed.

CONCLUSION:
Do not declare a specific component failed without the measurements or inspection results required by the manual.

CONFIDENCE: Low"""

    elif scenario["type"] == "safety_critical":
        answer = f"""SAFETY ASSESSMENT:

{fault} — {title}

This fault involves a safety-related or potentially hazardous machine condition. The technician must follow the safety constraints explicitly stated in the supplied evidence.

CHECKS:
Follow the corrective steps in the manual in their documented order.

IMPORTANT:
Do not bypass, defeat, or override a safety function unless the supplied manufacturer documentation explicitly permits the action.

CONCLUSION:
Resolve the safety condition and verify the required safety state before returning the machine to normal operation.

CONFIDENCE: High"""

    elif scenario["type"] in ["root_cause", "procedure", "data_integrity",
                              "constraint", "communication"]:
        answer = f"""FAULT: {fault} — {title}

ASSESSMENT:
Use the supplied manual evidence to identify the documented condition and possible causes.

DIAGNOSTIC APPROACH:
1. Confirm the condition described by the fault.
2. Check the probable causes listed in the manual.
3. Perform the corrective steps in the documented order.
4. Do not treat a probable cause as a confirmed root cause until the relevant diagnostic check supports it.

CONCLUSION:
The final diagnosis should be based on the evidence obtained during the prescribed checks.

CONFIDENCE: High"""

    else:
        raise ValueError(f"Unknown scenario type: {scenario['type']}")

    return answer


# Build answers for all scenarios
for item, scenario in zip(training_prompts, scenario_templates):
    item["answer"] = build_grounded_answer(scenario)


print("Answers created:", len(training_prompts))
print()

for i, item in enumerate(training_prompts[:3], 1):
    print("=" * 80)
    print(f"ANSWER {i}")
    print("=" * 80)
    print(item["answer"])
    print()

Answers created: 18

ANSWER 1
FAULT: F001 — DC Link Voltage Too High

ASSESSMENT:
The fault indicates the condition described in the supplied manual evidence.

DOCUMENTED CAUSES:
The manual lists the probable causes in the supplied evidence.

CORRECTIVE SEQUENCE:
Follow the corrective steps in the order specified by the manual.

SAFETY:
Follow all LOTO, electrical, hydraulic, and other safety requirements explicitly stated in the evidence.

CONCLUSION:
The diagnosis should be based on the documented measurements and checks rather than assuming a single failed component.

CONFIDENCE: High

ANSWER 2
FAULT DISTINCTION:

F001 — DC Link Voltage Too High
The primary evidence describes the condition and causes associated with F001.

F003 — Mains Overvoltage
The related evidence describes a separate fault condition.

KEY POINT:
Do not merge the two fault definitions. Use the measurements and conditions specified in the manual to determine which condition is actually present.

CONCLUSION:
The o

In [28]:
import re

def extract_fault_details(section):
    causes = []
    steps = []

    # Extract probable causes
    causes_match = re.search(
        r"Probable Causes:\s*(.*?)(?=Corrective Steps:)",
        section,
        flags=re.S
    )

    if causes_match:
        cause_text = causes_match.group(1)

        # Split PDF bullet points
        causes = [
            x.strip(" \n\r\t")
            for x in re.split(r"", cause_text)
            if x.strip(" \n\r\t")
        ]

    # Extract corrective steps
    steps_match = re.search(
        r"Corrective Steps:\s*(.*?)(?=Reference:|$)",
        section,
        flags=re.S
    )

    if steps_match:
        step_text = steps_match.group(1)

        # Extract numbered steps
        steps = [
            x.strip()
            for x in re.findall(
                r"\d+\.\s*(.*?)(?=\n\s*\d+\.\s*|\Z)",
                step_text,
                flags=re.S
            )
        ]

    return causes, steps


fault_details = {}

for _, row in fault_df.iterrows():
    causes, steps = extract_fault_details(row["section"])

    fault_details[row["code"]] = {
        "title": row["title"],
        "causes": causes,
        "steps": steps
    }


# Show F001 first
f001 = fault_details["F001"]

print("=" * 80)
print("F001 STRUCTURED DATA")
print("=" * 80)

print("\nCAUSES:")
for i, cause in enumerate(f001["causes"], 1):
    print(f"{i}. {cause}")

print("\nCORRECTIVE STEPS:")
for i, step in enumerate(f001["steps"], 1):
    print(f"{i}. {step}")

print("\n" + "=" * 80)
print("EXTRACTION SUMMARY")
print("=" * 80)

for code, data in fault_details.items():
    print(
        f"{code}: "
        f"{len(data['causes'])} causes, "
        f"{len(data['steps'])} corrective steps"
    )

F001 STRUCTURED DATA

CAUSES:
1. The braking resistor on the servo amplifier has gone open-circuit or its connecting cable is 
broken.
2. The backgauge deceleration ramp (parameter BG.DECEL in the drive menu) is set too 
aggressively short. 

===== PAGE 4 =====
3. Mains supply voltage is already at the high end of tolerance (>440 V AC) before 
regeneration occurs.

CORRECTIVE STEPS:
1. Measure the mains supply at the main isolator terminals — must be 400 V ±10% (360–440 
V). If high, notify the facility electrician before proceeding.
2. Power down (LOTO) and measure the braking resistor resistance at the servo amplifier 
terminals R+ and RB; the nominal value is 47 Ω ±10%. Replace if open or out of tolerance 
(Part No. DW-BRK-RES).
3. If the resistor is healthy, open the servo drive parameter menu and increase BG.DECEL 
from the current value by 25% (e.g., 0.2 s → 0.25 s) to reduce peak regenerative current.
4. If the fault persists at normal mains voltage with a good braking resistor,

In [29]:
# Step 26 — Validate structured fault knowledge

print("=" * 80)
print("STRUCTURED KNOWLEDGE VALIDATION")
print("=" * 80)

issues = []

for code, data in fault_details.items():
    causes = data["causes"]
    steps = data["steps"]

    # Basic validation
    if len(causes) == 0:
        issues.append(f"{code}: NO CAUSES extracted")

    if len(steps) == 0:
        issues.append(f"{code}: NO CORRECTIVE STEPS extracted")

    # Detect suspicious extraction artifacts
    for cause in causes:
        if "===== PAGE" in cause:
            issues.append(f"{code}: PAGE MARKER INSIDE CAUSE")

    for step in steps:
        if "===== PAGE" in step:
            issues.append(f"{code}: PAGE MARKER INSIDE STEP")


print(f"Total faults: {len(fault_details)}")
print(f"Faults with issues: {len(set(x.split(':')[0] for x in issues))}")

print("\n" + "-" * 80)

if issues:
    for issue in issues:
        print("⚠️", issue)
else:
    print("✅ No extraction issues found.")


print("\n" + "=" * 80)
print("KNOWLEDGE STATISTICS")
print("=" * 80)

total_causes = sum(len(x["causes"]) for x in fault_details.values())
total_steps = sum(len(x["steps"]) for x in fault_details.values())

print("Total causes:", total_causes)
print("Total corrective steps:", total_steps)
print("Average causes/fault:", round(total_causes / len(fault_details), 2))
print("Average steps/fault:", round(total_steps / len(fault_details), 2))

print("\n" + "=" * 80)
print("FAULT KNOWLEDGE TABLE")
print("=" * 80)

for code, data in fault_details.items():
    print(
        f"{code} | "
        f"{data['title']} | "
        f"{len(data['causes'])} causes | "
        f"{len(data['steps'])} steps"
    )

STRUCTURED KNOWLEDGE VALIDATION
Total faults: 21
Faults with issues: 8

--------------------------------------------------------------------------------
⚠️ F001: PAGE MARKER INSIDE CAUSE
⚠️ F003: PAGE MARKER INSIDE CAUSE
⚠️ F007: PAGE MARKER INSIDE STEP
⚠️ F020: PAGE MARKER INSIDE CAUSE
⚠️ F030: PAGE MARKER INSIDE CAUSE
⚠️ F060: PAGE MARKER INSIDE CAUSE
⚠️ F080: PAGE MARKER INSIDE STEP
⚠️ F099: PAGE MARKER INSIDE STEP

KNOWLEDGE STATISTICS
Total causes: 62
Total corrective steps: 78
Average causes/fault: 2.95
Average steps/fault: 3.71

FAULT KNOWLEDGE TABLE
F001 | DC Link Voltage Too High | 3 causes | 4 steps
F002 | DC Link Voltage Too Low | 3 causes | 4 steps
F003 | Mains Overvoltage | 3 causes | 3 steps
F005 | HPU Motor Overtemperature | 3 causes | 4 steps
F007 | Backgauge Axis X Following Error | 3 causes | 4 steps
F010 | Ram Angle Measurement Discrepancy | 3 causes | 4 steps
F011 | Temperature Sensor Short Circuit | 3 causes | 4 steps
F015 | Hydraulic Oil Temperature High | 3 cause

In [30]:
# Step 27 — Clean PDF extraction artifacts

def clean_pdf_artifacts(text):
    # Remove page separator lines
    text = re.sub(r"===== PAGE \d+ =====", "", text)

    # Normalize whitespace created by page breaks
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n\s*\n\s*\n+", "\n\n", text)

    return text.strip()


for code, data in fault_details.items():
    data["causes"] = [
        clean_pdf_artifacts(cause)
        for cause in data["causes"]
    ]

    data["steps"] = [
        clean_pdf_artifacts(step)
        for step in data["steps"]
    ]


# Validate again
print("=" * 80)
print("POST-CLEANING VALIDATION")
print("=" * 80)

issues = []

for code, data in fault_details.items():

    for cause in data["causes"]:
        if "===== PAGE" in cause:
            issues.append(f"{code}: PAGE MARKER STILL IN CAUSE")

    for step in data["steps"]:
        if "===== PAGE" in step:
            issues.append(f"{code}: PAGE MARKER STILL IN STEP")


if issues:
    for issue in issues:
        print("⚠️", issue)
else:
    print("✅ All PDF page markers removed.")


print("\n" + "=" * 80)
print("F001 AFTER CLEANING")
print("=" * 80)

print("\nCAUSES:")
for i, cause in enumerate(fault_details["F001"]["causes"], 1):
    print(f"{i}. {cause}")

print("\nCORRECTIVE STEPS:")
for i, step in enumerate(fault_details["F001"]["steps"], 1):
    print(f"{i}. {step}")

POST-CLEANING VALIDATION
✅ All PDF page markers removed.

F001 AFTER CLEANING

CAUSES:
1. The braking resistor on the servo amplifier has gone open-circuit or its connecting cable is 
broken.
2. The backgauge deceleration ramp (parameter BG.DECEL in the drive menu) is set too 
aggressively short.
3. Mains supply voltage is already at the high end of tolerance (>440 V AC) before 
regeneration occurs.

CORRECTIVE STEPS:
1. Measure the mains supply at the main isolator terminals — must be 400 V ±10% (360–440 
V). If high, notify the facility electrician before proceeding.
2. Power down (LOTO) and measure the braking resistor resistance at the servo amplifier 
terminals R+ and RB; the nominal value is 47 Ω ±10%. Replace if open or out of tolerance 
(Part No. DW-BRK-RES).
3. If the resistor is healthy, open the servo drive parameter menu and increase BG.DECEL 
from the current value by 25% (e.g., 0.2 s → 0.25 s) to reduce peak regenerative current.
4. If the fault persists at normal mains v

In [31]:
# Step 28 — Build a high-quality APEX reasoning example for F001

f001 = fault_details["F001"]

f001_answer = f"""FAULT:
F001 — {f001["title"]}

DIAGNOSIS:
The manual identifies three possible causes:
1. The braking resistor or its connecting cable may be open-circuit.
2. BG.DECEL may be set too aggressively short, causing excessive regenerative energy.
3. Mains voltage may already be above the specified tolerance before regeneration occurs.

DIAGNOSTIC ORDER:
1. Measure the mains supply at the main isolator terminals.
   Expected: 400 V ±10% (360–440 V).
   If the voltage is high, notify the facility electrician before proceeding.

2. If mains voltage is within tolerance, power down and apply LOTO.
   Measure the braking resistor between R+ and RB.
   Expected: 47 Ω ±10%.
   If it is open or outside tolerance, replace it with part DW-BRK-RES.

3. If the braking resistor is healthy, increase BG.DECEL by 25%.
   Example: 0.2 s → 0.25 s.

4. If the fault persists with normal mains voltage and a good braking resistor, the manual states that the DC link capacitors may be degraded.
   Request DeltaWorks service for a capacitance test.

SAFETY:
Do not perform the resistance measurement until the machine has been powered down and LOTO has been applied.

CONCLUSION:
F001 indicates excessive DC-link voltage in the servo amplifier. The manual provides a four-stage diagnostic sequence beginning with mains voltage, followed by the braking resistor, BG.DECEL, and finally a service-level capacitor test.

CONFIDENCE:
High — the fault definition, causes, measurements, limits, and corrective sequence are explicitly documented in the supplied manual.

EVIDENCE STATUS:
Documented facts only. No additional machine-specific cause is assumed beyond the supplied manual.
"""

print("=" * 80)
print("APEX F001 TRAINING ANSWER")
print("=" * 80)
print(f001_answer)

APEX F001 TRAINING ANSWER
FAULT:
F001 — DC Link Voltage Too High

DIAGNOSIS:
The manual identifies three possible causes:
1. The braking resistor or its connecting cable may be open-circuit.
2. BG.DECEL may be set too aggressively short, causing excessive regenerative energy.
3. Mains voltage may already be above the specified tolerance before regeneration occurs.

DIAGNOSTIC ORDER:
1. Measure the mains supply at the main isolator terminals.
   Expected: 400 V ±10% (360–440 V).
   If the voltage is high, notify the facility electrician before proceeding.

2. If mains voltage is within tolerance, power down and apply LOTO.
   Measure the braking resistor between R+ and RB.
   Expected: 47 Ω ±10%.
   If it is open or outside tolerance, replace it with part DW-BRK-RES.

3. If the braking resistor is healthy, increase BG.DECEL by 25%.
   Example: 0.2 s → 0.25 s.

4. If the fault persists with normal mains voltage and a good braking resistor, the manual states that the DC link capacitors ma

In [32]:
# Step 29 — Generate reasoning-oriented training answers
# This creates answers from the actual extracted manual evidence.

def build_reasoning_answer(code):
    data = fault_details[code]

    causes_text = "\n".join(
        f"{i}. {cause}"
        for i, cause in enumerate(data["causes"], 1)
    )

    steps_text = "\n".join(
        f"{i}. {step}"
        for i, step in enumerate(data["steps"], 1)
    )

    answer = f"""FAULT:
{code} — {data["title"]}

EVIDENCE-BASED DIAGNOSIS:
The supplied manual identifies the following probable causes:
{causes_text}

DIAGNOSTIC APPROACH:
Use the corrective steps in the documented sequence:

{steps_text}

REASONING:
Start with the earliest documented check because it can distinguish whether the fault is associated with the corresponding condition before proceeding to later checks. Only proceed to a later diagnostic step when the preceding check does not identify the problem.

SAFETY:
Follow every safety, shutdown, isolation, LOTO, or escalation requirement explicitly stated in the supplied procedure. Do not bypass safety devices or perform a procedure before its required isolation condition is satisfied.

CONCLUSION:
{code} is associated with "{data["title"]}". The supplied manual provides the documented causes and corrective sequence above. An exact failed component should not be claimed unless the available evidence establishes it.

CONFIDENCE:
High for the documented fault definition and troubleshooting procedure. The exact failed component remains uncertain until the relevant diagnostic checks are performed.

EVIDENCE STATUS:
Documented manual evidence. No additional technical cause has been assumed beyond the supplied evidence.
"""

    return answer


# Generate one example for inspection
test_answer = build_reasoning_answer("F001")

print("=" * 80)
print("REASONING-ORIENTED F001 ANSWER")
print("=" * 80)
print(test_answer)

REASONING-ORIENTED F001 ANSWER
FAULT:
F001 — DC Link Voltage Too High

EVIDENCE-BASED DIAGNOSIS:
The supplied manual identifies the following probable causes:
1. The braking resistor on the servo amplifier has gone open-circuit or its connecting cable is 
broken.
2. The backgauge deceleration ramp (parameter BG.DECEL in the drive menu) is set too 
aggressively short.
3. Mains supply voltage is already at the high end of tolerance (>440 V AC) before 
regeneration occurs.

DIAGNOSTIC APPROACH:
Use the corrective steps in the documented sequence:

1. Measure the mains supply at the main isolator terminals — must be 400 V ±10% (360–440 
V). If high, notify the facility electrician before proceeding.
2. Power down (LOTO) and measure the braking resistor resistance at the servo amplifier 
terminals R+ and RB; the nominal value is 47 Ω ±10%. Replace if open or out of tolerance 
(Part No. DW-BRK-RES).
3. If the resistor is healthy, open the servo drive parameter menu and increase BG.DECEL 
fro

In [33]:
# Step 30 — Generate base reasoning examples for all 21 faults

reasoning_examples = []

for code in fault_details:
    answer = build_reasoning_answer(code)

    reasoning_examples.append({
        "code": code,
        "title": fault_details[code]["title"],
        "answer": answer
    })

print("=" * 80)
print("BASE REASONING DATASET")
print("=" * 80)

print("Examples created:", len(reasoning_examples))

print("\n" + "-" * 80)
print("EXAMPLE LENGTHS")
print("-" * 80)

for example in reasoning_examples:
    print(
        f'{example["code"]}: '
        f'{len(example["answer"])} characters'
    )

print("\n" + "=" * 80)
print("SAMPLE: F030")
print("=" * 80)

print(
    next(
        x["answer"]
        for x in reasoning_examples
        if x["code"] == "F030"
    )
)

BASE REASONING DATASET
Examples created: 21

--------------------------------------------------------------------------------
EXAMPLE LENGTHS
--------------------------------------------------------------------------------
F001: 2291 characters
F002: 1764 characters
F003: 1641 characters
F005: 1767 characters
F007: 1720 characters
F010: 1747 characters
F011: 2578 characters
F015: 1828 characters
F020: 1852 characters
F025: 1821 characters
F030: 2741 characters
F035: 1733 characters
F040: 1789 characters
F045: 1724 characters
F050: 1792 characters
F055: 1727 characters
F060: 1561 characters
F070: 1646 characters
F080: 1887 characters
F090: 1640 characters
F099: 2597 characters

SAMPLE: F030
FAULT:
F030 — Door Safety Switch Open

EVIDENCE-BASED DIAGNOSIS:
The supplied manual identifies the following probable causes:
1. 
2. The rear guard door has been physically opened (for die retrieval or maintenance access) 
and not properly closed and latched.
3. The door safety switch (Schmersal AZ 

In [34]:
# Step 31 — Find malformed structured knowledge

print("=" * 80)
print("DETAILED KNOWLEDGE QUALITY CHECK")
print("=" * 80)

bad_items = []

for code, data in fault_details.items():

    # Check causes
    for i, cause in enumerate(data["causes"], 1):
        if not cause.strip():
            bad_items.append(
                f"{code} | CAUSE {i} | EMPTY"
            )

        elif len(cause.strip()) < 15:
            bad_items.append(
                f"{code} | CAUSE {i} | SUSPICIOUSLY SHORT: {repr(cause)}"
            )

    # Check corrective steps
    for i, step in enumerate(data["steps"], 1):
        if not step.strip():
            bad_items.append(
                f"{code} | STEP {i} | EMPTY"
            )

        elif len(step.strip()) < 15:
            bad_items.append(
                f"{code} | STEP {i} | SUSPICIOUSLY SHORT: {repr(step)}"
            )


print("\nIssues found:", len(bad_items))

if bad_items:
    for item in bad_items:
        print("⚠️", item)
else:
    print("✅ No empty or suspiciously short causes/steps.")


print("\n" + "=" * 80)
print("CAUSE/STEP COUNTS")
print("=" * 80)

for code, data in fault_details.items():
    print(
        f"{code}: "
        f"{len(data['causes'])} causes | "
        f"{len(data['steps'])} steps"
    )

DETAILED KNOWLEDGE QUALITY CHECK

Issues found: 2
⚠️ F030 | CAUSE 1 | EMPTY
⚠️ F060 | CAUSE 1 | EMPTY

CAUSE/STEP COUNTS
F001: 3 causes | 4 steps
F002: 3 causes | 4 steps
F003: 3 causes | 3 steps
F005: 3 causes | 4 steps
F007: 3 causes | 4 steps
F010: 3 causes | 4 steps
F011: 3 causes | 4 steps
F015: 3 causes | 4 steps
F020: 3 causes | 4 steps
F025: 3 causes | 4 steps
F030: 4 causes | 4 steps
F035: 3 causes | 4 steps
F040: 3 causes | 4 steps
F045: 2 causes | 4 steps
F050: 3 causes | 4 steps
F055: 3 causes | 3 steps
F060: 3 causes | 2 steps
F070: 3 causes | 3 steps
F080: 3 causes | 4 steps
F090: 2 causes | 3 steps
F099: 3 causes | 4 steps


In [35]:
# Step 32 — Re-extract structured knowledge cleanly
# Remove PDF page markers BEFORE extracting causes and steps.

clean_full_text = re.sub(
    r"===== PAGE \d+ =====",
    "",
    full_text
)

def extract_fault_details_clean(section):
    causes = []
    steps = []

    # Extract probable causes
    causes_match = re.search(
        r"Probable Causes:\s*(.*?)(?=Corrective Steps:)",
        section,
        flags=re.S
    )

    if causes_match:
        cause_text = causes_match.group(1)

        causes = [
            clean_pdf_artifacts(x)
            for x in re.split(r"", cause_text)
            if x.strip(" \n\r\t")
        ]

    # Extract corrective steps
    steps_match = re.search(
        r"Corrective Steps:\s*(.*?)(?=Reference:|$)",
        section,
        flags=re.S
    )

    if steps_match:
        step_text = steps_match.group(1)

        steps = [
            clean_pdf_artifacts(x)
            for x in re.findall(
                r"\d+\.\s*(.*?)(?=\n\s*\d+\.\s*|\Z)",
                step_text,
                flags=re.S
            )
        ]

    return causes, steps


# Rebuild structured knowledge from cleaned text
fault_details = {}

for _, row in fault_df.iterrows():

    # Use the original section but remove page markers first
    section = clean_pdf_artifacts(row["section"])

    causes, steps = extract_fault_details_clean(section)

    fault_details[row["code"]] = {
        "title": row["title"],
        "causes": causes,
        "steps": steps
    }


# Validate the two previously problematic faults
print("=" * 80)
print("F030")
print("=" * 80)

for i, cause in enumerate(fault_details["F030"]["causes"], 1):
    print(f"CAUSE {i}: {cause}")

print("\n" + "=" * 80)
print("F060")
print("=" * 80)

for i, cause in enumerate(fault_details["F060"]["causes"], 1):
    print(f"CAUSE {i}: {cause}")


print("\n" + "=" * 80)
print("FINAL COUNTS")
print("=" * 80)

for code, data in fault_details.items():
    print(
        f"{code}: "
        f"{len(data['causes'])} causes | "
        f"{len(data['steps'])} steps"
    )

F030
CAUSE 1: The rear guard door has been physically opened (for die retrieval or maintenance access) 
and not properly closed and latched.
CAUSE 2: The door safety switch (Schmersal AZ 16, positively operated) actuator key is missing or 
broken, preventing the switch from engaging even when the door is closed.
CAUSE 3: The wiring to the door safety switch has been damaged or a connector is loose, causing the 
circuit to read as open even with the door physically latched.

F060
CAUSE 1: Job program references incorrect R-axis value for the installed die height.
CAUSE 2: Die table not updated after a die change.

FINAL COUNTS
F001: 3 causes | 4 steps
F002: 3 causes | 4 steps
F003: 3 causes | 3 steps
F005: 3 causes | 4 steps
F007: 3 causes | 4 steps
F010: 3 causes | 4 steps
F011: 3 causes | 4 steps
F015: 3 causes | 4 steps
F020: 3 causes | 4 steps
F025: 3 causes | 4 steps
F030: 3 causes | 4 steps
F035: 3 causes | 4 steps
F040: 3 causes | 4 steps
F045: 2 causes | 4 steps
F050: 3 causes |

In [36]:
# Step 33 — Build the first real APEX training example

f001_prompt = f"""Machine:
DeltaWorks DX-200 Automated Press Brake

Observed fault:
F001 — DC Link Voltage Too High

Operator report:
The machine has stopped with F001 during operation.

Question:
What should the technician check first, what values should they expect, and what should they do next?

Evidence from the maintenance manual:
{get_fault_section("F001")}
"""

f001_training_answer = """FAULT:
F001 — DC Link Voltage Too High

FIRST CHECK:
Measure the mains supply at the main isolator terminals.

EXPECTED VALUE:
400 V ±10%, which is 360–440 V AC.

DECISION:
- If the mains voltage is above the specified range, stop the troubleshooting sequence and notify the facility electrician before proceeding.
- If the mains voltage is within tolerance, continue to the braking resistor check.

NEXT CHECK:
Power down the machine and apply LOTO. Measure the braking resistor at the servo amplifier terminals R+ and RB.

EXPECTED RESISTANCE:
47 Ω ±10%.

DECISION:
- If the resistor is open or outside tolerance, replace it with part DW-BRK-RES.
- If the resistor is healthy, increase BG.DECEL by 25% according to the documented procedure.

FURTHER ESCALATION:
If F001 persists with normal mains voltage and a good braking resistor, the manual states that the DC link capacitors may be degraded. Request DeltaWorks service for a capacitance test.

SAFETY:
LOTO is required before measuring the braking resistor. Do not proceed with electrical checks when the required isolation condition has not been satisfied.

CONCLUSION:
The correct first diagnostic action for F001 is to verify mains voltage. The exact failed component cannot be determined from the fault code alone.

CONFIDENCE:
High — the diagnostic sequence, limits, and corrective actions are explicitly documented in the supplied manual.

EVIDENCE STATUS:
All diagnostic claims above are supported by the supplied manual.
"""

training_example_1 = {
    "messages": [
        {
            "role": "system",
            "content": (
                "You are APEX, an industrial troubleshooting AI. "
                "Use supplied evidence as the source of truth. "
                "Separate documented facts from inference. "
                "Follow conditional diagnostic sequences. "
                "Preserve safety and LOTO requirements. "
                "Never invent technical facts or claim an exact failed component "
                "without sufficient evidence."
            )
        },
        {
            "role": "user",
            "content": f001_prompt
        },
        {
            "role": "assistant",
            "content": f001_training_answer
        }
    ]
}

print("=" * 80)
print("APEX TRAINING EXAMPLE #1")
print("=" * 80)

print("\nUSER PROMPT:")
print(training_example_1["messages"][1]["content"])

print("\n" + "=" * 80)
print("IDEAL ASSISTANT RESPONSE:")
print("=" * 80)

print(training_example_1["messages"][2]["content"])

APEX TRAINING EXAMPLE #1

USER PROMPT:
Machine:
DeltaWorks DX-200 Automated Press Brake

Observed fault:
F001 — DC Link Voltage Too High

Operator report:
The machine has stopped with F001 during operation.

Question:
What should the technician check first, what values should they expect, and what should they do next?

Evidence from the maintenance manual:
F001 — DC Link Voltage Too High 
Description: The DC link bus inside the servo amplifier powering the backgauge axes has risen 
above 780 V DC. This is distinct from mains overvoltage (see F003). The condition is caused by 
regenerative energy returning from a rapidly decelerating backgauge axis being unable to dissipate 
fast enough. The servo amplifier shuts down all backgauge axes to protect the power stage. 
Probable Causes: 
 The braking resistor on the servo amplifier has gone open-circuit or its connecting cable is 
broken. 
 The backgauge deceleration ramp (parameter BG.DECEL in the drive menu) is set too 
aggressively shor

In [37]:
# Step 34 — Clean evidence contexts for APEX training

def get_clean_fault_section(code):
    section = get_fault_section(code)

    # Remove PDF page markers
    section = re.sub(
        r"===== PAGE \d+ =====",
        "",
        section
    )

    # Remove excessive whitespace caused by PDF extraction
    section = re.sub(r"[ \t]+", " ", section)
    section = re.sub(r"\n\s*\n\s*\n+", "\n\n", section)

    return section.strip()


# Rebuild the F001 training prompt with clean evidence
f001_clean_prompt = f"""Machine:
DeltaWorks DX-200 Automated Press Brake

Observed fault:
F001 — DC Link Voltage Too High

Operator report:
The machine has stopped with F001 during operation.

Question:
What should the technician check first, what values should they expect, and what should they do next?

Evidence from the maintenance manual:
{get_clean_fault_section("F001")}
"""


print("=" * 80)
print("CLEAN F001 EVIDENCE")
print("=" * 80)

print(f001_clean_prompt)

print("\n" + "=" * 80)
print("PAGE MARKER CHECK")
print("=" * 80)

if re.search(r"===== PAGE \d+ =====", f001_clean_prompt):
    print("⚠️ Page markers still present")
else:
    print("✅ No PDF page markers present")

CLEAN F001 EVIDENCE
Machine:
DeltaWorks DX-200 Automated Press Brake

Observed fault:
F001 — DC Link Voltage Too High

Operator report:
The machine has stopped with F001 during operation.

Question:
What should the technician check first, what values should they expect, and what should they do next?

Evidence from the maintenance manual:
F001 — DC Link Voltage Too High 
Description: The DC link bus inside the servo amplifier powering the backgauge axes has risen 
above 780 V DC. This is distinct from mains overvoltage (see F003). The condition is caused by 
regenerative energy returning from a rapidly decelerating backgauge axis being unable to dissipate 
fast enough. The servo amplifier shuts down all backgauge axes to protect the power stage. 
Probable Causes: 
 The braking resistor on the servo amplifier has gone open-circuit or its connecting cable is 
broken. 
 The backgauge deceleration ramp (parameter BG.DECEL in the drive menu) is set too 
aggressively short. 

 Mains supply

In [38]:
# Step 35 — Build clean prompts for all APEX scenarios

training_prompts = []

for i, scenario in enumerate(scenario_templates, 1):

    primary = scenario["fault"]

    prompt = f"""Machine:
DeltaWorks DX-200 Automated Press Brake

Observed fault:
{primary} — {fault_details[primary]["title"]}

Question:
{scenario["question"]}

Evidence from the maintenance manual:
{get_clean_fault_section(primary)}
"""

    # Add related fault evidence when the scenario requires disambiguation
    if "related_fault" in scenario:
        related = scenario["related_fault"]

        prompt += f"""

Additional related fault evidence:
{related} — {fault_details[related]["title"]}

{get_clean_fault_section(related)}
"""

    training_prompts.append({
        "scenario_id": i,
        "type": scenario["type"],
        "fault": primary,
        "related_fault": scenario.get("related_fault"),
        "prompt": prompt.strip()
    })


print("=" * 80)
print("APEX SCENARIO PROMPTS")
print("=" * 80)

print("Total prompts:", len(training_prompts))

print("\n" + "-" * 80)

for item in training_prompts:
    related = (
        f" + {item['related_fault']}"
        if item["related_fault"]
        else ""
    )

    print(
        f'{item["scenario_id"]:02d} | '
        f'{item["type"]:<22} | '
        f'{item["fault"]}{related}'
    )


print("\n" + "=" * 80)
print("SAMPLE: DISAMBIGUATION")
print("=" * 80)

print(training_prompts[1]["prompt"])

APEX SCENARIO PROMPTS
Total prompts: 18

--------------------------------------------------------------------------------
01 | direct_diagnosis       | F001
02 | fault_disambiguation   | F001 + F003
03 | insufficient_evidence  | F001
04 | direct_diagnosis       | F002
05 | fault_disambiguation   | F002 + F099
06 | safety_critical        | F020
07 | safety_critical        | F030
08 | root_cause             | F011
09 | root_cause             | F015
10 | procedure              | F040
11 | data_integrity         | F045
12 | procedure              | F050
13 | root_cause             | F055
14 | constraint             | F060
15 | communication          | F070
16 | safety_critical        | F080
17 | data_integrity         | F090
18 | safety_critical        | F099

SAMPLE: DISAMBIGUATION
Machine:
DeltaWorks DX-200 Automated Press Brake

Observed fault:
F001 — DC Link Voltage Too High

Question:
How should these two faults be distinguished?

Evidence from the maintenance manual:
F001 — DC Link V

In [39]:
# Step 36 — Build scenario-specific answers for the first 3 APEX examples

def build_scenario_answer(scenario):
    code = scenario["fault"]
    data = fault_details[code]
    scenario_type = scenario["type"]

    if scenario_type == "direct_diagnosis":

        causes = "\n".join(
            f"{i}. {cause}"
            for i, cause in enumerate(data["causes"], 1)
        )

        steps = "\n".join(
            f"{i}. {step}"
            for i, step in enumerate(data["steps"], 1)
        )

        return f"""FAULT:
{code} — {data["title"]}

DIAGNOSIS:
The manual identifies these probable causes:
{causes}

DIAGNOSTIC SEQUENCE:
{steps}

DECISION LOGIC:
Follow the documented corrective sequence in order. If a check identifies the corresponding fault condition, apply the documented corrective action. If the check passes, continue to the next documented step.

SAFETY:
Apply all shutdown, isolation, LOTO, and escalation requirements stated in the manual before performing the relevant procedure.

CONCLUSION:
The fault can be diagnosed using the documented sequence above. The exact failed component should not be claimed until the relevant diagnostic check provides sufficient evidence.

CONFIDENCE:
High for the documented fault definition and diagnostic procedure.

EVIDENCE STATUS:
Based only on the supplied maintenance manual."""

    elif scenario_type == "fault_disambiguation":

        related = scenario["related_fault"]
        related_data = fault_details[related]

        return f"""FAULT DISTINCTION:

{code} — {data["title"]}
- The manual defines this condition as: {get_clean_fault_section(code).split("Probable Causes:")[0].strip()}

{related} — {related_data["title"]}
- The manual defines this condition as: {get_clean_fault_section(related).split("Probable Causes:")[0].strip()}

KEY DIFFERENCE:
{code} and {related} should not be treated as the same fault. The manual explicitly distinguishes their fault conditions and provides different diagnostic paths.

DIAGNOSTIC APPROACH:
Use the measurements and conditions specified in the relevant fault section rather than assuming that one fault automatically implies the other.

CONCLUSION:
The correct diagnosis depends on the evidence observed at the machine. A fault code should be interpreted according to its documented definition and not replaced with a similar-looking fault.

CONFIDENCE:
High for the distinction because both fault definitions are explicitly documented in the supplied manual.

EVIDENCE STATUS:
Based only on the supplied maintenance manual."""

    elif scenario_type == "insufficient_evidence":

        return f"""FAULT:
{code} — {data["title"]}

WHAT CAN BE CONCLUDED:
The manual confirms that {code} corresponds to "{data["title"]}".

WHAT CANNOT YET BE CONCLUDED:
The fault code alone does not establish which individual component or cause has failed.

DOCUMENTED POSSIBILITIES:
{chr(10).join(f"- {cause}" for cause in data["causes"])}

NEXT ACTION:
Perform the documented diagnostic checks in sequence and use their results to narrow the cause.

CONCLUSION:
There is insufficient evidence to identify the exact failed component from the fault code alone. APEX should not invent or select a specific cause without supporting diagnostic evidence.

CONFIDENCE:
High that the fault is correctly identified; low for any specific root cause until additional evidence is available.

EVIDENCE STATUS:
The response is limited to the supplied maintenance manual."""

    else:
        return None


# Generate the first three
scenario_answers = []

for item in training_prompts[:3]:
    scenario = next(
        s for s in scenario_templates
        if s["fault"] == item["fault"]
        and s["type"] == item["type"]
        and s.get("related_fault") == item.get("related_fault")
    )

    answer = build_scenario_answer(scenario)

    scenario_answers.append({
        "scenario_id": item["scenario_id"],
        "type": item["type"],
        "fault": item["fault"],
        "answer": answer
    })


print("=" * 80)
print("SCENARIO ANSWERS CREATED")
print("=" * 80)

for item in scenario_answers:
    print(
        f'{item["scenario_id"]:02d} | '
        f'{item["type"]} | '
        f'{item["fault"]}'
    )

print("\n" + "=" * 80)
print("EXAMPLE 1 — DIRECT DIAGNOSIS")
print("=" * 80)
print(scenario_answers[0]["answer"])

print("\n" + "=" * 80)
print("EXAMPLE 2 — FAULT DISAMBIGUATION")
print("=" * 80)
print(scenario_answers[1]["answer"])

print("\n" + "=" * 80)
print("EXAMPLE 3 — INSUFFICIENT EVIDENCE")
print("=" * 80)
print(scenario_answers[2]["answer"])

SCENARIO ANSWERS CREATED
01 | direct_diagnosis | F001
02 | fault_disambiguation | F001
03 | insufficient_evidence | F001

EXAMPLE 1 — DIRECT DIAGNOSIS
FAULT:
F001 — DC Link Voltage Too High

DIAGNOSIS:
The manual identifies these probable causes:
1. The braking resistor on the servo amplifier has gone open-circuit or its connecting cable is 
broken.
2. The backgauge deceleration ramp (parameter BG.DECEL in the drive menu) is set too 
aggressively short.
3. Mains supply voltage is already at the high end of tolerance (>440 V AC) before 
regeneration occurs.

DIAGNOSTIC SEQUENCE:
1. Measure the mains supply at the main isolator terminals — must be 400 V ±10% (360–440 
V). If high, notify the facility electrician before proceeding.
2. Power down (LOTO) and measure the braking resistor resistance at the servo amplifier 
terminals R+ and RB; the nominal value is 47 Ω ±10%. Replace if open or out of tolerance 
(Part No. DW-BRK-RES).
3. If the resistor is healthy, open the servo drive paramet

In [40]:

# Step 37 — Improve F001 vs F003 disambiguation

f001_f003_answer = """FAULT DISAMBIGUATION:

F001 — DC Link Voltage Too High
- Condition: The servo amplifier DC link bus has risen above 780 V DC.
- Context: The manual associates this with regenerative energy from a rapidly decelerating backgauge axis that cannot dissipate the energy quickly enough.
- Possible causes include a failed braking resistor, an overly aggressive BG.DECEL setting, or mains voltage already above 440 V AC before regeneration.
- The servo amplifier shuts down the backgauge axes to protect the power stage.

F003 — Mains Overvoltage
- Condition: The incoming mains supply voltage exceeds 440 V AC for more than 500 ms.
- Documented causes include a utility voltage surge, neutral conductor loss, or incorrect transformer tap position.
- The corrective procedure focuses on measuring all three phases and investigating the incoming electrical supply.

KEY DISTINCTION:
F001 describes an excessive DC-link voltage condition inside the servo amplifier, whereas F003 describes an incoming mains overvoltage condition.

IMPORTANT OVERLAP:
High mains voltage can contribute to F001, but that does not make F001 and F003 the same fault. F003 is specifically triggered by the documented incoming-mains condition, while F001 concerns the servo amplifier DC-link condition.

HOW TO DISTINGUISH THEM:
1. Check the fault code and its documented definition.
2. For F001, verify mains voltage as the first documented check, then continue with the braking-resistor and BG.DECEL checks if mains voltage is normal.
3. For F003, measure and record all three mains phases. If voltage is persistently above the specified limit, notify the facility electrical team and do not operate the machine.

CONCLUSION:
Do not substitute F003 for F001 merely because high mains voltage is present. Use the measured electrical condition together with the fault code to determine which documented diagnostic path applies.

CONFIDENCE:
High — both fault definitions and their diagnostic procedures are explicitly documented in the supplied manual.

EVIDENCE STATUS:
Based only on the supplied maintenance manual.
"""


print("=" * 80)
print("IMPROVED F001 vs F003 DISAMBIGUATION")
print("=" * 80)
print(f001_f003_answer)

IMPROVED F001 vs F003 DISAMBIGUATION
FAULT DISAMBIGUATION:

F001 — DC Link Voltage Too High
- Condition: The servo amplifier DC link bus has risen above 780 V DC.
- Context: The manual associates this with regenerative energy from a rapidly decelerating backgauge axis that cannot dissipate the energy quickly enough.
- Possible causes include a failed braking resistor, an overly aggressive BG.DECEL setting, or mains voltage already above 440 V AC before regeneration.
- The servo amplifier shuts down the backgauge axes to protect the power stage.

F003 — Mains Overvoltage
- Condition: The incoming mains supply voltage exceeds 440 V AC for more than 500 ms.
- Documented causes include a utility voltage surge, neutral conductor loss, or incorrect transformer tap position.
- The corrective procedure focuses on measuring all three phases and investigating the incoming electrical supply.

KEY DISTINCTION:
F001 describes an excessive DC-link voltage condition inside the servo amplifier, wherea

In [41]:
# Step 38 — Generate scenario-specific answers 4–9

def build_specialized_answer(scenario):

    code = scenario["fault"]
    data = fault_details[code]
    scenario_type = scenario["type"]

    causes = "\n".join(
        f"{i}. {cause}"
        for i, cause in enumerate(data["causes"], 1)
    )

    steps = "\n".join(
        f"{i}. {step}"
        for i, step in enumerate(data["steps"], 1)
    )

    # ------------------------------------------------------------------
    # DIRECT DIAGNOSIS
    # ------------------------------------------------------------------
    if scenario_type == "direct_diagnosis":

        return f"""FAULT:
{code} — {data["title"]}

DOCUMENTED CAUSES:
{causes}

DIAGNOSTIC SEQUENCE:
{steps}

DECISION LOGIC:
Follow the documented checks in order. Use the result of each check to decide whether to correct the identified condition or continue to the next step.

SAFETY:
Follow every shutdown, isolation, LOTO, and escalation requirement stated in the relevant procedure.

CONCLUSION:
The supplied manual provides a defined troubleshooting sequence for {code}. The exact failed component should only be identified when the diagnostic evidence supports it.

CONFIDENCE:
High for the documented fault and troubleshooting procedure.

EVIDENCE STATUS:
Based only on the supplied maintenance manual."""

    # ------------------------------------------------------------------
    # FAULT DISAMBIGUATION
    # ------------------------------------------------------------------
    elif scenario_type == "fault_disambiguation":

        related = scenario["related_fault"]
        related_data = fault_details[related]

        return f"""FAULT DISAMBIGUATION:

{code} — {data["title"]}

Documented causes:
{causes}

{related} — {related_data["title"]}

Documented causes:
{chr(10).join(f"- {x}" for x in related_data["causes"])}

KEY DISTINCTION:
The two fault codes represent different documented machine conditions and therefore should follow different diagnostic paths.

HOW TO DISTINGUISH:
Use the specific fault definition, measured machine condition, and diagnostic checks associated with each code. Do not infer one fault solely from the presence of a related symptom.

CONCLUSION:
The correct fault path should be selected from the documented fault definition and available machine evidence.

CONFIDENCE:
High for the distinction because both fault conditions are explicitly documented.

EVIDENCE STATUS:
Based only on the supplied maintenance manual."""

    # ------------------------------------------------------------------
    # SAFETY CRITICAL
    # ------------------------------------------------------------------
    elif scenario_type == "safety_critical":

        return f"""SAFETY-CRITICAL FAULT:
{code} — {data["title"]}

WHAT THE MANUAL ESTABLISHES:
{get_clean_fault_section(code).split("Probable Causes:")[0].strip()}

DOCUMENTED CAUSES:
{causes}

REQUIRED RESPONSE:
Do not bypass, override, defeat, short-circuit, or otherwise circumvent the safety-related condition.

DOCUMENTED CORRECTIVE SEQUENCE:
{steps}

SAFETY PRIORITY:
The machine must remain in the documented safe state while the relevant safety condition is investigated. Apply LOTO whenever the documented procedure requires it.

CONCLUSION:
Because {code} is safety-related, restoration of production must not take priority over resolving the documented safety condition.

CONFIDENCE:
High for the documented safety requirements and troubleshooting procedure.

EVIDENCE STATUS:
Based only on the supplied maintenance manual."""

    # ------------------------------------------------------------------
    # ROOT CAUSE
    # ------------------------------------------------------------------
    elif scenario_type == "root_cause":

        return f"""FAULT:
{code} — {data["title"]}

ROOT-CAUSE ANALYSIS:

The manual identifies these probable causes:
{causes}

EVIDENCE INTERPRETATION:
The fault code establishes the documented fault condition, but it does not by itself establish which probable cause is responsible.

DIAGNOSTIC PATH:
{steps}

DECISION LOGIC:
Each documented check should be used to eliminate or confirm the corresponding possible cause. Do not select a root cause before the relevant diagnostic evidence is available.

CONCLUSION:
The likely root cause must be determined from the results of the documented checks rather than inferred solely from the fault code.

CONFIDENCE:
High for the listed causes and diagnostic procedure; the specific root cause remains uncertain until diagnostic evidence is obtained.

EVIDENCE STATUS:
Based only on the supplied maintenance manual."""

    # ------------------------------------------------------------------
    # FALLBACK
    # ------------------------------------------------------------------
    return None


# Generate scenarios 4–9
scenario_answers_4_9 = []

for item in training_prompts[3:9]:

    scenario = next(
        s for s in scenario_templates
        if s["fault"] == item["fault"]
        and s["type"] == item["type"]
        and s.get("related_fault") == item.get("related_fault")
    )

    answer = build_specialized_answer(scenario)

    scenario_answers_4_9.append({
        "scenario_id": item["scenario_id"],
        "type": item["type"],
        "fault": item["fault"],
        "answer": answer
    })


print("=" * 80)
print("SCENARIOS 4–9 CREATED")
print("=" * 80)

for item in scenario_answers_4_9:
    print(
        f'{item["scenario_id"]:02d} | '
        f'{item["type"]:<22} | '
        f'{item["fault"]}'
    )


print("\n" + "=" * 80)
print("SAMPLE: F020 SAFETY-CRITICAL")
print("=" * 80)

print(
    next(
        x["answer"]
        for x in scenario_answers_4_9
        if x["fault"] == "F020"
    )
)

SCENARIOS 4–9 CREATED
04 | direct_diagnosis       | F002
05 | fault_disambiguation   | F002
06 | safety_critical        | F020
07 | safety_critical        | F030
08 | root_cause             | F011
09 | root_cause             | F015

SAMPLE: F020 SAFETY-CRITICAL
SAFETY-CRITICAL FAULT:
F020 — Light Curtain Fault

WHAT THE MANUAL ESTABLISHES:
F020 — Light Curtain Fault 
Description: The Sick deTec4 safety light curtain controller has reported an internal device fault 
(not a simple beam interruption).

DOCUMENTED CAUSES:
1. Light curtain emitter or receiver head damaged or misaligned beyond tolerance.
2. Contamination (oil mist) on the optical lens surface.
3. Internal controller fault within the deTec4.

REQUIRED RESPONSE:
Do not bypass, override, defeat, short-circuit, or otherwise circumvent the safety-related condition.

DOCUMENTED CORRECTIVE SEQUENCE:
1. Clean the emitter and receiver lenses with a lint-free cloth and isopropyl alcohol.
2. Check alignment: the green LED on the receiv

In [42]:
# Step 39 — Generate scenario-specific answers 10–14

scenario_answers_10_14 = []

for item in training_prompts[9:14]:

    scenario = next(
        s for s in scenario_templates
        if s["fault"] == item["fault"]
        and s["type"] == item["type"]
        and s.get("related_fault") == item.get("related_fault")
    )

    code = scenario["fault"]
    data = fault_details[code]
    scenario_type = scenario["type"]

    causes = "\n".join(
        f"{i}. {cause}"
        for i, cause in enumerate(data["causes"], 1)
    )

    steps = "\n".join(
        f"{i}. {step}"
        for i, step in enumerate(data["steps"], 1)
    )

    if scenario_type == "procedure":

        answer = f"""PROCEDURE:
{code} — {data["title"]}

PURPOSE:
Follow the documented procedure to diagnose and correct the reported fault.

STEPS:
{steps}

DECISION LOGIC:
Complete each step in sequence. Where the manual specifies a condition or expected result, use that result to determine whether to continue or apply the corresponding corrective action.

SAFETY:
Follow any explicit shutdown, isolation, LOTO, or escalation instruction contained in the documented steps.

CONCLUSION:
The procedure above is the documented troubleshooting path for {code}. Do not substitute undocumented procedures or bypass required safety controls.

EVIDENCE STATUS:
Based only on the supplied maintenance manual."""

    elif scenario_type == "data_integrity":

        answer = f"""DATA-INTEGRITY FAULT:
{code} — {data["title"]}

WHAT THE FAULT INDICATES:
The manual identifies {code} as "{data["title"]}".

DOCUMENTED CAUSES:
{causes}

CORRECTIVE PROCEDURE:
{steps}

REASONING:
This fault concerns the integrity or validity of machine data. The fault code alone does not establish which underlying cause is responsible. The documented checks should be used to determine the appropriate corrective action.

IMPORTANT:
Do not invent, overwrite, or assume data values that are not provided by the evidence.

CONCLUSION:
Use the documented procedure to restore or verify the affected data. The exact underlying cause should only be identified when the diagnostic evidence supports it.

CONFIDENCE:
High for the documented fault and corrective procedure; the specific underlying cause depends on diagnostic results.

EVIDENCE STATUS:
Based only on the supplied maintenance manual."""

    elif scenario_type == "root_cause":

        answer = f"""ROOT-CAUSE ANALYSIS:
{code} — {data["title"]}

DOCUMENTED POSSIBLE CAUSES:
{causes}

DIAGNOSTIC STEPS:
{steps}

REASONING:
The fault code identifies the machine condition but does not by itself prove which probable cause is responsible. Each documented diagnostic step should be used to narrow the possibilities.

CONCLUSION:
Do not declare a single root cause until the corresponding diagnostic evidence confirms it.

CONFIDENCE:
High for the documented causes and procedure; the actual root cause remains conditional on diagnostic results.

EVIDENCE STATUS:
Based only on the supplied maintenance manual."""

    elif scenario_type == "constraint":

        answer = f"""CONSTRAINT CHECK:
{code} — {data["title"]}

DOCUMENTED CONDITION:
{get_clean_fault_section(code).split("Probable Causes:")[0].strip()}

DOCUMENTED CAUSES:
{causes}

REQUIRED CONSTRAINTS:
{steps}

REASONING:
The machine value must remain within the range explicitly specified by the manual. A value outside that documented range should be treated as a fault condition rather than corrected by guessing an alternative value.

CONCLUSION:
Use the documented range and corrective procedure to bring the machine configuration back within the permitted constraint.

CONFIDENCE:
High for the documented operating constraint and corrective procedure.

EVIDENCE STATUS:
Based only on the supplied maintenance manual."""

    else:
        answer = None

    scenario_answers_10_14.append({
        "scenario_id": item["scenario_id"],
        "type": item["type"],
        "fault": item["fault"],
        "answer": answer
    })


print("=" * 80)
print("SCENARIOS 10–14 CREATED")
print("=" * 80)

for item in scenario_answers_10_14:
    print(
        f'{item["scenario_id"]:02d} | '
        f'{item["type"]:<22} | '
        f'{item["fault"]}'
    )


print("\n" + "=" * 80)
print("SAMPLE: F060 CONSTRAINT")
print("=" * 80)

print(
    next(
        x["answer"]
        for x in scenario_answers_10_14
        if x["fault"] == "F060"
    )
)

SCENARIOS 10–14 CREATED
10 | procedure              | F040
11 | data_integrity         | F045
12 | procedure              | F050
13 | root_cause             | F055
14 | constraint             | F060

SAMPLE: F060 CONSTRAINT
CONSTRAINT CHECK:
F060 — Axis R (Ram Height) Out of Range

DOCUMENTED CONDITION:
F060 — Axis R (Ram Height) Out of Range 
Description: The backgauge R-axis (vertical height) has been commanded to a position outside its 
0–300 mm travel range.

DOCUMENTED CAUSES:
1. Job program references incorrect R-axis value for the installed die height.
2. Die table not updated after a die change.

REQUIRED CONSTRAINTS:
1. Enter the correct installed die height in HMI Tool Table > Bottom Die Height.
2. Recalculate the job sequence; the R-axis target must be within 0–300 mm.

REASONING:
The machine value must remain within the range explicitly specified by the manual. A value outside that documented range should be treated as a fault condition rather than corrected by guessing an 

In [43]:
# Step 40 — Generate scenario-specific answers 15–18

scenario_answers_15_18 = []

for item in training_prompts[14:18]:

    scenario = next(
        s for s in scenario_templates
        if s["fault"] == item["fault"]
        and s["type"] == item["type"]
        and s.get("related_fault") == item.get("related_fault")
    )

    code = scenario["fault"]
    data = fault_details[code]
    scenario_type = scenario["type"]

    causes = "\n".join(
        f"{i}. {cause}"
        for i, cause in enumerate(data["causes"], 1)
    )

    steps = "\n".join(
        f"{i}. {step}"
        for i, step in enumerate(data["steps"], 1)
    )

    if scenario_type == "communication":

        answer = f"""COMMUNICATION FAULT:
{code} — {data["title"]}

DOCUMENTED CAUSES:
{causes}

DIAGNOSTIC PROCEDURE:
{steps}

REASONING:
The fault indicates a communication problem, but the fault code alone does not establish which communication component or connection has failed. The documented checks should be performed in sequence to distinguish the possible causes.

DECISION LOGIC:
Use the result of each documented check to determine whether to correct the identified condition or continue to the next diagnostic step.

CONCLUSION:
Do not assume that the network fault is caused by a specific device or cable until the documented checks provide supporting evidence.

CONFIDENCE:
High for the documented fault definition and diagnostic procedure.

EVIDENCE STATUS:
Based only on the supplied maintenance manual."""

    elif scenario_type == "safety_critical":

        # Only state safety requirements that are explicitly present
        # in the documented fault evidence.
        safety_terms = []

        full_section = get_clean_fault_section(code)

        for term in [
            "LOTO",
            "do not operate",
            "do not bypass",
            "cannot be bypassed",
            "cannot be overridden",
            "short-circuit",
            "override",
            "defeat"
        ]:
            if term.lower() in full_section.lower():
                safety_terms.append(term)

        safety_note = (
            "The documented procedure contains explicit safety restrictions; "
            "these must be followed exactly."
            if safety_terms
            else
            "No additional safety procedure should be invented beyond the requirements explicitly stated in the supplied evidence."
        )

        answer = f"""SAFETY-CRITICAL FAULT:
{code} — {data["title"]}

WHAT THE MANUAL ESTABLISHES:
{get_clean_fault_section(code).split("Probable Causes:")[0].strip()}

DOCUMENTED CAUSES:
{causes}

DOCUMENTED CORRECTIVE STEPS:
{steps}

SAFETY REQUIREMENT:
{safety_note}

DECISION LOGIC:
Resolve the documented safety condition using the supplied procedure. Do not bypass or override a safety control merely to restore production.

CONCLUSION:
Production should only resume when the documented safety condition has been appropriately resolved according to the manual.

CONFIDENCE:
High for the documented fault condition and corrective procedure.

EVIDENCE STATUS:
Based only on the supplied maintenance manual."""

    elif scenario_type == "data_integrity":

        answer = f"""DATA-INTEGRITY FAULT:
{code} — {data["title"]}

DOCUMENTED CAUSES:
{causes}

CORRECTIVE PROCEDURE:
{steps}

REASONING:
The fault indicates a problem with the integrity or validity of machine data. The fault code does not by itself prove which underlying cause is responsible.

DIAGNOSTIC APPROACH:
Use the documented corrective procedure to identify and correct the affected data condition. Do not invent replacement values or assume that data is valid without evidence.

CONCLUSION:
The affected data should be restored or verified using the documented procedure. The underlying cause should remain uncertain unless the available evidence establishes it.

CONFIDENCE:
High for the documented fault and corrective procedure; the specific underlying cause depends on diagnostic evidence.

EVIDENCE STATUS:
Based only on the supplied maintenance manual."""

    else:
        answer = None

    scenario_answers_15_18.append({
        "scenario_id": item["scenario_id"],
        "type": item["type"],
        "fault": item["fault"],
        "answer": answer
    })


print("=" * 80)
print("SCENARIOS 15–18 CREATED")
print("=" * 80)

for item in scenario_answers_15_18:
    print(
        f'{item["scenario_id"]:02d} | '
        f'{item["type"]:<22} | '
        f'{item["fault"]}'
    )


print("\n" + "=" * 80)
print("SAMPLE: F070 COMMUNICATION")
print("=" * 80)

print(
    next(
        x["answer"]
        for x in scenario_answers_15_18
        if x["fault"] == "F070"
    )
)

SCENARIOS 15–18 CREATED
15 | communication          | F070
16 | safety_critical        | F080
17 | data_integrity         | F090
18 | safety_critical        | F099

SAMPLE: F070 COMMUNICATION
COMMUNICATION FAULT:
F070 — Network Communication Fault

DOCUMENTED CAUSES:
1. Network cable disconnected or damaged.
2. Factory network switch failure.
3. IP address configuration error after a firmware update.

DIAGNOSTIC PROCEDURE:
1. Check the Ethernet cable from the control cabinet to the network switch.
2. Ping the machine's IP address from a factory PC to confirm connectivity.
3. Verify IP settings on the HMI: Settings > Network. Compare against the site network map.

REASONING:
The fault indicates a communication problem, but the fault code alone does not establish which communication component or connection has failed. The documented checks should be performed in sequence to distinguish the possible causes.

DECISION LOGIC:
Use the result of each documented check to determine whether to c

In [44]:
# Step 41 — Assemble all 18 APEX training examples

# Combine answers from the batches
all_scenario_answers = (
    scenario_answers[:3]
    + scenario_answers_4_9
    + scenario_answers_10_14
    + scenario_answers_15_18
)

# Create lookup by scenario ID
answer_lookup = {
    item["scenario_id"]: item["answer"]
    for item in all_scenario_answers
}

apex_training_records = []

for item in training_prompts:

    scenario_id = item["scenario_id"]

    answer = answer_lookup.get(scenario_id)

    if answer is None:
        print(f"⚠️ Missing answer for scenario {scenario_id}")
        continue

    apex_training_records.append({
        "scenario_id": scenario_id,
        "type": item["type"],
        "fault": item["fault"],
        "related_fault": item["related_fault"],
        "prompt": item["prompt"],
        "answer": answer
    })


print("=" * 80)
print("APEX TRAINING RECORDS")
print("=" * 80)

print("Total records:", len(apex_training_records))

print("\n" + "-" * 80)
print("SCENARIO COVERAGE")
print("-" * 80)

for record in apex_training_records:
    related = (
        f" + {record['related_fault']}"
        if record["related_fault"]
        else ""
    )

    print(
        f'{record["scenario_id"]:02d} | '
        f'{record["type"]:<22} | '
        f'{record["fault"]}{related} | '
        f'prompt={len(record["prompt"])} chars | '
        f'answer={len(record["answer"])} chars'
    )


print("\n" + "=" * 80)
print("DATASET VALIDATION")
print("=" * 80)

checks = {
    "18 records": len(apex_training_records) == 18,
    "unique scenario IDs": len({
        x["scenario_id"] for x in apex_training_records
    }) == 18,
    "no empty prompts": all(
        x["prompt"].strip() for x in apex_training_records
    ),
    "no empty answers": all(
        x["answer"].strip() for x in apex_training_records
    ),
    "no page markers": all(
        not re.search(r"===== PAGE \d+ =====", x["prompt"])
        for x in apex_training_records
    )
}

for name, result in checks.items():
    print(
        f'{"✅" if result else "❌"} {name}'
    )


print("\n" + "=" * 80)
print("ANSWER TYPE DISTRIBUTION")
print("=" * 80)

from collections import Counter

type_counts = Counter(
    x["type"]
    for x in apex_training_records
)

for scenario_type, count in type_counts.items():
    print(f"{scenario_type}: {count}")

APEX TRAINING RECORDS
Total records: 18

--------------------------------------------------------------------------------
SCENARIO COVERAGE
--------------------------------------------------------------------------------
01 | direct_diagnosis       | F001 | prompt=1802 chars | answer=1847 chars
02 | fault_disambiguation   | F001 + F003 | prompt=2405 chars | answer=1417 chars
03 | insufficient_evidence  | F001 | prompt=1791 chars | answer=1112 chars
04 | direct_diagnosis       | F002 | prompt=948 chars | answer=1185 chars
05 | fault_disambiguation   | F002 + F099 | prompt=4804 chars | answer=1303 chars
06 | safety_critical        | F020 | prompt=1106 chars | answer=1495 chars
07 | safety_critical        | F030 | prompt=2229 chars | answer=2626 chars
08 | root_cause             | F011 | prompt=2210 chars | answer=2142 chars
09 | root_cause             | F015 | prompt=1046 chars | answer=1394 chars
10 | procedure              | F040 | prompt=1039 chars | answer=1042 chars
11 | data_integr

In [45]:
# Step 42 — Prepare APEX dataset for assistant-only QLoRA training

from datasets import Dataset

SYSTEM_PROMPT = """You are APEX, an industrial troubleshooting AI.

Use supplied evidence as the source of truth.
Separate documented facts from inference.
Follow conditional diagnostic sequences.
Preserve explicit safety requirements.
Never invent technical facts.
Never claim an exact failed component without sufficient evidence.
If evidence is insufficient, state what is known and what cannot be concluded."""

apex_messages = []

for record in apex_training_records:

    apex_messages.append({
        "messages": [
            {
                "role": "system",
                "content": SYSTEM_PROMPT
            },
            {
                "role": "user",
                "content": record["prompt"]
            },
            {
                "role": "assistant",
                "content": record["answer"]
            }
        ]
    })


apex_dataset = Dataset.from_list(apex_messages)

print("=" * 80)
print("APEX CHAT DATASET")
print("=" * 80)

print(apex_dataset)
print("\nNumber of examples:", len(apex_dataset))


# ------------------------------------------------------------
# Apply Qwen3 chat template
# ------------------------------------------------------------

def format_apex_example(example):

    text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False
    )

    return {
        "text": text
    }


formatted_apex_dataset = apex_dataset.map(
    format_apex_example,
    remove_columns=["messages"]
)


# ------------------------------------------------------------
# Inspect one formatted example
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("FORMATTED EXAMPLE")
print("=" * 80)

print(
    formatted_apex_dataset[0]["text"][:5000]
)


# ------------------------------------------------------------
# Token length statistics
# ------------------------------------------------------------

token_lengths = []

for example in formatted_apex_dataset:

    tokens = tokenizer(
        example["text"],
        add_special_tokens=False
    )["input_ids"]

    token_lengths.append(len(tokens))


print("\n" + "=" * 80)
print("TOKEN LENGTHS")
print("=" * 80)

print("Minimum:", min(token_lengths))
print("Maximum:", max(token_lengths))
print("Average:", round(sum(token_lengths) / len(token_lengths), 1))

APEX CHAT DATASET
Dataset({
    features: ['messages'],
    num_rows: 18
})

Number of examples: 18


Map:   0%|          | 0/18 [00:00<?, ? examples/s]


FORMATTED EXAMPLE
<|im_start|>system
You are APEX, an industrial troubleshooting AI.

Use supplied evidence as the source of truth.
Separate documented facts from inference.
Follow conditional diagnostic sequences.
Preserve explicit safety requirements.
Never invent technical facts.
Never claim an exact failed component without sufficient evidence.
If evidence is insufficient, state what is known and what cannot be concluded.<|im_end|>
<|im_start|>user
Machine:
DeltaWorks DX-200 Automated Press Brake

Observed fault:
F001 — DC Link Voltage Too High

Question:
What should be checked first, and what is the prescribed troubleshooting sequence?

Evidence from the maintenance manual:
F001 — DC Link Voltage Too High 
Description: The DC link bus inside the servo amplifier powering the backgauge axes has risen 
above 780 V DC. This is distinct from mains overvoltage (see F003). The condition is caused by 
regenerative energy returning from a rapidly decelerating backgauge axis being unable t

In [46]:
# Step 43 — Create assistant-only training labels

def tokenize_assistant_only(example):

    text = example["text"]

    # Tokenize the complete conversation
    encoded = tokenizer(
        text,
        add_special_tokens=False
    )

    input_ids = encoded["input_ids"]
    attention_mask = encoded["attention_mask"]

    # Find the assistant response boundary
    assistant_marker = "<|im_start|>assistant"

    marker_ids = tokenizer(
        assistant_marker,
        add_special_tokens=False
    )["input_ids"]

    # Search for the assistant marker
    assistant_start = None

    for i in range(len(input_ids) - len(marker_ids) + 1):

        if input_ids[i:i + len(marker_ids)] == marker_ids:
            assistant_start = i + len(marker_ids)
            break

    if assistant_start is None:
        raise ValueError("Assistant marker not found")

    # Ignore everything before the assistant response
    labels = [-100] * assistant_start + input_ids[assistant_start:]

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels
    }


tokenized_apex_dataset = formatted_apex_dataset.map(
    tokenize_assistant_only,
    remove_columns=["text"]
)


# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

print("=" * 80)
print("ASSISTANT-ONLY LABEL VALIDATION")
print("=" * 80)

for i in range(3):

    example = tokenized_apex_dataset[i]

    total_tokens = len(example["input_ids"])

    ignored_tokens = sum(
        1 for x in example["labels"]
        if x == -100
    )

    trainable_tokens = sum(
        1 for x in example["labels"]
        if x != -100
    )

    print(f"\nExample {i + 1}")
    print("Total tokens:", total_tokens)
    print("Ignored prompt/evidence tokens:", ignored_tokens)
    print("Trainable assistant tokens:", trainable_tokens)

    print(
        "First trainable token position:",
        next(
            j for j, x in enumerate(example["labels"])
            if x != -100
        )
    )


# ------------------------------------------------------------
# Verify first trainable text
# ------------------------------------------------------------

example = tokenized_apex_dataset[0]

first_trainable = next(
    i for i, x in enumerate(example["labels"])
    if x != -100
)

decoded_answer_start = tokenizer.decode(
    example["input_ids"][first_trainable:first_trainable + 120],
    skip_special_tokens=False
)

print("\n" + "=" * 80)
print("FIRST TRAINABLE TEXT")
print("=" * 80)

print(decoded_answer_start)

Map:   0%|          | 0/18 [00:00<?, ? examples/s]

ASSISTANT-ONLY LABEL VALIDATION

Example 1
Total tokens: 982
Ignored prompt/evidence tokens: 545
Trainable assistant tokens: 437
First trainable token position: 545

Example 2
Total tokens: 1014
Ignored prompt/evidence tokens: 697
Trainable assistant tokens: 317
First trainable token position: 697

Example 3
Total tokens: 785
Ignored prompt/evidence tokens: 544
Trainable assistant tokens: 241
First trainable token position: 544

FIRST TRAINABLE TEXT

<think>

</think>

FAULT:
F001 — DC Link Voltage Too High

DIAGNOSIS:
The manual identifies these probable causes:
1. The braking resistor on the servo amplifier has gone open-circuit or its connecting cable is 
broken.
2. The backgauge deceleration ramp (parameter BG.DECEL in the drive menu) is set too 
aggressively short.
3. Mains supply voltage is already at the high end of tolerance (>440 V AC) before 
regeneration occurs.

DIAGNOSTIC SEQUENCE:
1. Measure


In [47]:
# Step 44 — Prepare final dataset for Trainer

from datasets import Dataset

# Keep only the fields required by the Trainer
final_apex_dataset = tokenized_apex_dataset

print("=" * 80)
print("FINAL APEX TRAINING DATASET")
print("=" * 80)

print(final_apex_dataset)

print("\nNumber of examples:", len(final_apex_dataset))

print("\nFeatures:")
for feature in final_apex_dataset.features:
    print(" -", feature)


# ------------------------------------------------------------
# Verify all examples contain valid labels
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("LABEL VALIDATION")
print("=" * 80)

all_valid = True

for i, example in enumerate(final_apex_dataset):

    labels = example["labels"]

    trainable = sum(
        1 for x in labels
        if x != -100
    )

    if trainable == 0:
        print(f"❌ Example {i + 1}: no trainable tokens")
        all_valid = False

    else:
        print(
            f"✅ Example {i + 1}: "
            f"{trainable} trainable tokens"
        )

print("\nOverall:", "✅ VALID" if all_valid else "❌ INVALID")

FINAL APEX TRAINING DATASET
Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 18
})

Number of examples: 18

Features:
 - input_ids
 - attention_mask
 - labels

LABEL VALIDATION
✅ Example 1: 437 trainable tokens
✅ Example 2: 317 trainable tokens
✅ Example 3: 241 trainable tokens
✅ Example 4: 270 trainable tokens
✅ Example 5: 285 trainable tokens
✅ Example 6: 326 trainable tokens
✅ Example 7: 622 trainable tokens
✅ Example 8: 477 trainable tokens
✅ Example 9: 294 trainable tokens
✅ Example 10: 223 trainable tokens
✅ Example 11: 290 trainable tokens
✅ Example 12: 223 trainable tokens
✅ Example 13: 229 trainable tokens
✅ Example 14: 239 trainable tokens
✅ Example 15: 234 trainable tokens
✅ Example 16: 330 trainable tokens
✅ Example 17: 256 trainable tokens
✅ Example 18: 600 trainable tokens

Overall: ✅ VALID


In [48]:
# Step 45 — Configure APEX QLoRA Trainer

from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./apex_model",

    # Small dataset + T4
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,

    # QLoRA learning rate
    learning_rate=2e-4,

    # Start conservatively
    num_train_epochs=3,

    # Memory / precision
    fp16=True,
    gradient_checkpointing=True,

    # Optimizer
    optim="paged_adamw_8bit",

    # Logging / saving
    logging_steps=1,
    save_strategy="epoch",
    save_total_limit=2,

    # No external logging
    report_to="none",

    # Reproducibility
    seed=42,
)

print("=" * 80)
print("APEX TRAINING CONFIGURATION")
print("=" * 80)

print("Batch size:", training_args.per_device_train_batch_size)
print("Gradient accumulation:", training_args.gradient_accumulation_steps)
print("Effective batch size:",
      training_args.per_device_train_batch_size *
      training_args.gradient_accumulation_steps)

print("Learning rate:", training_args.learning_rate)
print("Epochs:", training_args.num_train_epochs)
print("FP16:", training_args.fp16)
print("Gradient checkpointing:",
      training_args.gradient_checkpointing)
print("Optimizer:", training_args.optim)
print("Output:", training_args.output_dir)

APEX TRAINING CONFIGURATION
Batch size: 1
Gradient accumulation: 4
Effective batch size: 4
Learning rate: 0.0002
Epochs: 3
FP16: True
Gradient checkpointing: True
Optimizer: OptimizerNames.PAGED_ADAMW_8BIT
Output: ./apex_model


In [49]:
# Step 46 — Attach LoRA and create the APEX Trainer

from peft import get_peft_model
from transformers import Trainer

# ------------------------------------------------------------
# Attach LoRA adapters to the already-loaded Qwen3-4B model
# ------------------------------------------------------------

apex_model = get_peft_model(
    model,
    lora_config
)

print("=" * 80)
print("APEX LoRA MODEL")
print("=" * 80)

apex_model.print_trainable_parameters()


# ------------------------------------------------------------
# Create Trainer
# ------------------------------------------------------------

trainer = Trainer(
    model=apex_model,
    args=training_args,
    train_dataset=final_apex_dataset,
)

print("\n" + "=" * 80)
print("TRAINER CREATED")
print("=" * 80)

print("Training examples:", len(final_apex_dataset))
print("Output directory:", training_args.output_dir)
print("Trainer ready: ✅")

/usr/local/lib/python3.13/dist-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/peft/mapping_func.py:78: UserWarning: The PEFT config's `base_model_name_or_path` was renamed from 'Qwen/Qwen3-4B' to 'None'. Please ensure that the correct base model is loaded when loading this checkpoint.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/peft/tuners/tuners_utils.py:305: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


APEX LoRA MODEL
trainable params: 33,030,144 || all params: 4,055,498,240 || trainable%: 0.8145

TRAINER CREATED
Training examples: 18
Output directory: ./apex_model
Trainer ready: ✅


In [50]:
# Step 47 — Reset model to a clean single-PEFT state

print("=" * 80)
print("CURRENT MODEL STATE")
print("=" * 80)

print("Model class:", type(model).__name__)
print("Has PEFT config:", hasattr(model, "peft_config"))

# ------------------------------------------------------------
# Unwrap any existing PEFT adapter
# ------------------------------------------------------------

if hasattr(model, "unload"):
    model = model.unload()
    print("\n✅ Existing PEFT adapter unloaded.")
else:
    print("\n⚠️ Model does not expose unload().")

# ------------------------------------------------------------
# Verify clean base model
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("AFTER UNLOAD")
print("=" * 80)

print("Model class:", type(model).__name__)
print("Has PEFT config:", hasattr(model, "peft_config"))

# Check whether PEFT configuration is still attached
if hasattr(model, "peft_config"):
    print("PEFT config:", model.peft_config)
else:
    print("PEFT config: None")

print("\nModel reset complete: ✅")

CURRENT MODEL STATE
Model class: PeftModelForCausalLM
Has PEFT config: True

✅ Existing PEFT adapter unloaded.

AFTER UNLOAD
Model class: Qwen3ForCausalLM
Has PEFT config: False
PEFT config: None

Model reset complete: ✅


In [51]:
# Step 48 — Attach ONE clean LoRA adapter to APEX

from peft import get_peft_model

apex_model = get_peft_model(
    model,
    lora_config
)

print("=" * 80)
print("APEX MODEL")
print("=" * 80)

print("Model class:", type(apex_model).__name__)

print("\nTrainable parameters:")
apex_model.print_trainable_parameters()

print("\n" + "=" * 80)
print("PEFT CONFIGURATION")
print("=" * 80)

print("Adapters:", list(apex_model.peft_config.keys()))

print("LoRA rank:", lora_config.r)
print("LoRA alpha:", lora_config.lora_alpha)
print("LoRA dropout:", lora_config.lora_dropout)

print("\nSingle LoRA adapter attached: ✅")

APEX MODEL
Model class: PeftModelForCausalLM

Trainable parameters:
trainable params: 33,030,144 || all params: 4,055,498,240 || trainable%: 0.8145

PEFT CONFIGURATION
Adapters: ['default']
LoRA rank: 16
LoRA alpha: 32
LoRA dropout: 0.05

Single LoRA adapter attached: ✅


In [52]:
# Step 49 — Train APEX

print("=" * 80)
print("STARTING APEX TRAINING")
print("=" * 80)

print("Examples:", len(final_apex_dataset))
print("Epochs:", training_args.num_train_epochs)
print("Effective batch size:",
      training_args.per_device_train_batch_size *
      training_args.gradient_accumulation_steps)

print("\nTraining...\n")

train_result = trainer.train()

print("\n" + "=" * 80)
print("APEX TRAINING COMPLETE")
print("=" * 80)

print("Training loss:", train_result.training_loss)
print("Training steps:", train_result.global_step)

STARTING APEX TRAINING
Examples: 18
Epochs: 3
Effective batch size: 4

Training...



Step,Training Loss
1,1.482466
2,1.635142
3,1.374858
4,0.996723
5,0.855459
6,0.845411
7,0.723708
8,0.749167
9,0.993279
10,0.612792



APEX TRAINING COMPLETE
Training loss: 0.8921192824840546
Training steps: 15


In [53]:
# Step 50 — Save APEX LoRA adapter

APEX_ADAPTER_PATH = "./apex_model/final_adapter"

apex_model.save_pretrained(APEX_ADAPTER_PATH)
tokenizer.save_pretrained(APEX_ADAPTER_PATH)

print("=" * 80)
print("APEX ADAPTER SAVED")
print("=" * 80)

print("Path:", APEX_ADAPTER_PATH)

import os

print("\nFiles created:")

for filename in sorted(os.listdir(APEX_ADAPTER_PATH)):
    filepath = os.path.join(APEX_ADAPTER_PATH, filename)
    size_kb = os.path.getsize(filepath) / 1024
    print(f" - {filename:<35} {size_kb:,.1f} KB")

print("\nSave status: ✅")

APEX ADAPTER SAVED
Path: ./apex_model/final_adapter

Files created:
 - README.md                           5.1 KB
 - adapter_config.json                 1.1 KB
 - adapter_model.safetensors           129,089.7 KB
 - chat_template.jinja                 4.1 KB
 - tokenizer.json                      11,154.9 KB
 - tokenizer_config.json               0.7 KB

Save status: ✅


In [54]:
# Step 51 — APEX inference function

import torch

apex_model.eval()

def apex_generate(user_prompt, max_new_tokens=700):

    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": user_prompt
        }
    ]

    prompt_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        prompt_text,
        return_tensors="pt"
    ).to(apex_model.device)

    with torch.no_grad():
        outputs = apex_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=None,
            top_p=None,
            repetition_penalty=1.05,
        )

    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    response = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

    return response.strip()


print("=" * 80)
print("APEX INFERENCE FUNCTION")
print("=" * 80)
print("Model:", type(apex_model).__name__)
print("Device:", apex_model.device)
print("Inference function ready: ✅")

APEX INFERENCE FUNCTION
Model: PeftModelForCausalLM
Device: cuda:0
Inference function ready: ✅


In [55]:
# Step 52 — First unseen APEX evaluation

unseen_test_1 = """Machine:
DeltaWorks DX-200 Automated Press Brake

Observed condition:
F001 — DC Link Voltage Too High

Additional observation:
The machine is connected to a 400 V AC supply. The operator reports that
the F001 fault appears mainly when the backgauge decelerates rapidly.
No measurement of the braking resistor has been performed yet.

Evidence from the maintenance manual:
F001 — DC Link Voltage Too High
Description: The DC link bus inside the servo amplifier powering the backgauge
axes has risen above 780 V DC. This is distinct from mains overvoltage (see F003).
The condition is caused by regenerative energy returning from a rapidly decelerating
backgauge axis being unable to dissipate fast enough. The servo amplifier shuts down
all backgauge axes to protect the power stage.

Probable Causes:
1. The braking resistor on the servo amplifier has gone open-circuit or its
   connecting cable is broken.
2. The backgauge deceleration ramp (BG.DECEL) is set too aggressively short.
3. Mains supply voltage is already at the high end of tolerance (>440 V AC)
   before regeneration occurs.

Corrective Steps:
1. Measure mains supply at the main isolator terminals — 400 V ±10%
   (360–440 V). If high, notify the facility electrician before proceeding.
2. Power down (LOTO) and measure the braking resistor at R+ and RB;
   nominal value 47 Ω ±10%. Replace if open or out of tolerance.
3. If the resistor is healthy, increase BG.DECEL by 25%.
4. If the fault persists at normal mains voltage with a good braking resistor,
   request DeltaWorks service for a DC link capacitor capacitance test.

Question:
Given the additional observation, what is the most appropriate next diagnostic
action? Do not claim that a specific component has failed unless the evidence
supports that conclusion.
"""

response = apex_generate(
    unseen_test_1,
    max_new_tokens=700
)

print("=" * 80)
print("APEX UNSEEN TEST 1")
print("=" * 80)
print(response)

APEX UNSEEN TEST 1
<think>

</think>

DIAGNOSTIC REASONING:

F001 — DC Link Voltage Too High
EVIDENCE STATUS:
Based only on the supplied maintenance manual and the observed condition.

CAUSE ANALYSIS:
The fault is described in the maintenance manual as a condition where the DC link bus inside the servo amplifier powering the backgauge axes has risen above 780 V DC. The manual lists three probable causes.

DECISION LOGIC:
The observed condition is that the F001 fault appears mainly when the backgauge decelerates rapidly.

CONCLUSION:
The most likely cause is that the braking resistor on the servo amplifier has gone open-circuit or its connecting cable is broken.

EVIDENCE STATUS:
The maintenance manual provides the documented cause but does not confirm the fault condition.

DECISION:
Do not claim that a specific component has failed unless the evidence supports that conclusion.

ACTION:
Measure the braking resistor at R+ and RB; nominal value 47 Ω ±10%.

CONFIDENCE:
High for the diagnos

In [56]:
# Step 53 — Inspect Qwen3 generation / stopping configuration

print("=" * 80)
print("TOKENIZER STOP TOKENS")
print("=" * 80)

print("EOS token:", repr(tokenizer.eos_token))
print("EOS token ID:", tokenizer.eos_token_id)

print("PAD token:", repr(tokenizer.pad_token))
print("PAD token ID:", tokenizer.pad_token_id)


print("\n" + "=" * 80)
print("MODEL GENERATION CONFIG")
print("=" * 80)

print("EOS token ID:",
      apex_model.generation_config.eos_token_id)

print("PAD token ID:",
      apex_model.generation_config.pad_token_id)

print("Max new tokens:",
      apex_model.generation_config.max_new_tokens)

print("\n" + "=" * 80)
print("SPECIAL TOKENS")
print("=" * 80)

for name, token_id in tokenizer.special_tokens_map.items():
    print(f"{name}: {token_id}")

TOKENIZER STOP TOKENS
EOS token: '<|im_end|>'
EOS token ID: 151645
PAD token: '<|endoftext|>'
PAD token ID: 151643

MODEL GENERATION CONFIG
EOS token ID: [151645, 151643]
PAD token ID: 151643
Max new tokens: None

SPECIAL TOKENS
eos_token: <|im_end|>
pad_token: <|endoftext|>


In [57]:
# Step 54 — Controlled APEX generation with explicit chat EOS

def apex_generate_controlled(user_prompt, max_new_tokens=450):

    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": user_prompt
        }
    ]

    prompt_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        prompt_text,
        return_tensors="pt"
    ).to(apex_model.device)

    with torch.no_grad():

        outputs = apex_model.generate(
            **inputs,

            max_new_tokens=max_new_tokens,

            do_sample=False,

            # Explicit Qwen chat stopping
            eos_token_id=tokenizer.convert_tokens_to_ids("<|im_end|>"),
            pad_token_id=tokenizer.pad_token_id,

            # Reduce repetition
            repetition_penalty=1.10,

            use_cache=True,
        )

    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    response = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

    return response.strip()


print("=" * 80)
print("CONTROLLED GENERATION TEST")
print("=" * 80)

response = apex_generate_controlled(
    unseen_test_1,
    max_new_tokens=450
)

print(response)

CONTROLLED GENERATION TEST
<think>

</think>

DIAGNOSTIC REASONING:

F001 — DC Link Voltage Too High
EVIDENCE STATUS:
Based only on the supplied maintenance manual and observed condition.

CAUSE ANALYSIS:
The fault is described in the maintenance manual as occurring due to rapid deceleration of the backgauge axis causing excessive regenerative energy to return to the DC link bus.

CONCLUSION:
The fault is likely related to the backgauge deceleration setting rather than the physical condition of the braking resistor.

DECISION LOGIC:
The maintenance manual provides a sequence of corrective steps based on the condition of the braking resistor and the deceleration ramp.

ACTION SELECTION:
The first step in the manual recommends measuring the mains supply at the main isolator terminals.

CONFIDENCE:
High for the recommended action based on the supplied maintenance manual.

DECISION:
Measure the mains supply at the main isolator terminals — 400 V ±10% (360–440 V).

EVIDENCE STATUS:
Based on

In [58]:
# Step 55 — Inspect repeated structures in APEX training answers

print("=" * 80)
print("TRAINING ANSWER REPETITION CHECK")
print("=" * 80)

for record in apex_training_records:

    answer = record["answer"]

    # Count important structural labels
    labels_to_check = [
        "EVIDENCE STATUS:",
        "CONFIDENCE:",
        "DECISION:",
        "CONCLUSION:",
        "ACTION:",
        "SAFETY:"
    ]

    counts = {
        label: answer.count(label)
        for label in labels_to_check
    }

    repeated = {
        label: count
        for label, count in counts.items()
        if count > 1
    }

    print(
        f'\nScenario {record["scenario_id"]:02d} | '
        f'{record["type"]} | {record["fault"]}'
    )

    if repeated:
        print("⚠️ Repeated labels:", repeated)
    else:
        print("✅ No repeated structural labels")

print("\n" + "=" * 80)
print("CHECK COMPLETE")
print("=" * 80)

TRAINING ANSWER REPETITION CHECK

Scenario 01 | direct_diagnosis | F001
✅ No repeated structural labels

Scenario 02 | fault_disambiguation | F001
✅ No repeated structural labels

Scenario 03 | insufficient_evidence | F001
✅ No repeated structural labels

Scenario 04 | direct_diagnosis | F002
✅ No repeated structural labels

Scenario 05 | fault_disambiguation | F002
✅ No repeated structural labels

Scenario 06 | safety_critical | F020
✅ No repeated structural labels

Scenario 07 | safety_critical | F030
✅ No repeated structural labels

Scenario 08 | root_cause | F011
✅ No repeated structural labels

Scenario 09 | root_cause | F015
✅ No repeated structural labels

Scenario 10 | procedure | F040
✅ No repeated structural labels

Scenario 11 | data_integrity | F045
✅ No repeated structural labels

Scenario 12 | procedure | F050
✅ No repeated structural labels

Scenario 13 | root_cause | F055
✅ No repeated structural labels

Scenario 14 | constraint | F060
✅ No repeated structural labels

S

In [59]:
# Step 56 — Unseen F002 evaluation

unseen_test_2 = """Machine:
DeltaWorks DX-200 Automated Press Brake

Observed fault:
F002 — DC Link Voltage Too Low

Evidence from the maintenance manual:
F002 — DC Link Voltage Too Low

Description:
The DC link bus has fallen below the minimum operating threshold.

Probable Causes:
1. Incoming mains phase loss.
2. A blown input fuse.
3. Failed rectifier diode or undersized incoming cable.

Corrective Steps:
1. Measure all three mains phases at the main isolator; all must be present
   and within 360–440 V AC.
2. Check input fuses for continuity and inspect for signs of overheating.
3. If mains and fuses are healthy, perform a rectifier diode test.
4. Verify incoming cable cross-section against the machine specification.

Question:
The operator reports that the machine suddenly stopped with F002.
What should the technician check first, and how should the diagnosis proceed?
Do not assume which electrical component has failed.
"""

response = apex_generate_controlled(
    unseen_test_2,
    max_new_tokens=350
)

print("=" * 80)
print("APEX UNSEEN TEST 2 — F002")
print("=" * 80)
print(response)

APEX UNSEEN TEST 2 — F002
<think>

</think>

FAULT:
F002 — DC Link Voltage Too Low

DIAGNOSTIC PATH:

1. Check the condition of the incoming mains supply.
   - Confirm that all three phases are present and within the specified voltage range (360–440 V AC).
   - Use a multimeter to measure each phase at the main isolator.

EVIDENCE STATUS:
Based only on the supplied maintenance manual.

DECISION LOGIC:
Only follow the documented diagnostic sequence.

CONCLUSION:
Follow the documented diagnostic procedure in order.

CONFIDENCE:
High for the documented procedure.

REASONING:
The supplied maintenance manual provides the correct diagnostic path.

SAFETY:
Always ensure that the equipment is isolated before testing.

DOCUMENTATION:
Record the results of each step according to the maintenance manual.

DECISION:
Continue following the documented procedure until the root cause is identified.

CONFIDENCE:
High for the documented procedure.

EVIDENCE STATUS:
Based only on the supplied maintenance 

In [60]:
# Step 57 — Diagnose repetition with a short generation limit

short_response = apex_generate_controlled(
    unseen_test_2,
    max_new_tokens=180
)

print("=" * 80)
print("SHORT GENERATION TEST — F002")
print("=" * 80)

print(short_response)

print("\n" + "=" * 80)
print("OUTPUT TOKEN COUNT")
print("=" * 80)

output_tokens = tokenizer(
    short_response,
    add_special_tokens=False
)["input_ids"]

print("Generated tokens:", len(output_tokens))

SHORT GENERATION TEST — F002
<think>

</think>

FAULT:
F002 — DC Link Voltage Too Low

DIAGNOSTIC PATH:

1. Check the condition of the incoming mains supply.
   - Confirm that all three phases are present and within the specified voltage range (360–440 V AC).
   - Use a multimeter to measure each phase at the main isolator.

EVIDENCE STATUS:
Based only on the supplied maintenance manual.

DECISION LOGIC:
Only follow the documented diagnostic sequence.

CONCLUSION:
Follow the documented diagnostic procedure in order.

CONFIDENCE:
High for the documented procedure.

REASONING:
The supplied maintenance manual provides the correct diagnostic path.

SAFETY:
Always ensure that the equipment is isolated before testing.

DOCUMENTATION:
Record the results of each step according to the maintenance manual.

DECISION:
Continue following the documented procedure until the root cause is identified

OUTPUT TOKEN COUNT
Generated tokens: 180


In [61]:
# Step 58 — Inspect all APEX training targets

print("=" * 80)
print("APEX TRAINING TARGET REVIEW")
print("=" * 80)

for record in apex_training_records:

    print("\n" + "-" * 80)
    print(
        f'SCENARIO {record["scenario_id"]:02d} | '
        f'{record["type"]} | '
        f'{record["fault"]}'
    )
    print("-" * 80)

    print(record["answer"])

print("\n" + "=" * 80)
print("END OF TRAINING TARGET REVIEW")
print("=" * 80)

APEX TRAINING TARGET REVIEW

--------------------------------------------------------------------------------
SCENARIO 01 | direct_diagnosis | F001
--------------------------------------------------------------------------------
FAULT:
F001 — DC Link Voltage Too High

DIAGNOSIS:
The manual identifies these probable causes:
1. The braking resistor on the servo amplifier has gone open-circuit or its connecting cable is 
broken.
2. The backgauge deceleration ramp (parameter BG.DECEL in the drive menu) is set too 
aggressively short.
3. Mains supply voltage is already at the high end of tolerance (>440 V AC) before 
regeneration occurs.

DIAGNOSTIC SEQUENCE:
1. Measure the mains supply at the main isolator terminals — must be 400 V ±10% (360–440 
V). If high, notify the facility electrician before proceeding.
2. Power down (LOTO) and measure the braking resistor resistance at the servo amplifier 
terminals R+ and RB; the nominal value is 47 Ω ±10%. Replace if open or out of tolerance 
(Par

In [62]:
# Step 59 — Clean repetitive meta-sections from APEX targets

import re

def clean_training_answer(answer):

    # Remove generic confidence blocks
    answer = re.sub(
        r"\n*CONFIDENCE:\s*.*?(?=\n[A-Z][A-Z /_-]+:|\Z)",
        "\n",
        answer,
        flags=re.S
    )

    # Remove generic evidence-status blocks
    answer = re.sub(
        r"\n*EVIDENCE STATUS:\s*.*?(?=\n[A-Z][A-Z /_-]+:|\Z)",
        "\n",
        answer,
        flags=re.S
    )

    # Remove generic documentation blocks
    answer = re.sub(
        r"\n*DOCUMENTATION:\s*.*?(?=\n[A-Z][A-Z /_-]+:|\Z)",
        "\n",
        answer,
        flags=re.S
    )

    # Clean excessive blank lines
    answer = re.sub(r"\n{3,}", "\n\n", answer)

    return answer.strip()


# ------------------------------------------------------------
# Create cleaned training records
# ------------------------------------------------------------

cleaned_apex_training_records = []

for record in apex_training_records:

    cleaned_answer = clean_training_answer(
        record["answer"]
    )

    cleaned_apex_training_records.append({
        **record,
        "answer": cleaned_answer
    })


# ------------------------------------------------------------
# Compare before vs after
# ------------------------------------------------------------

print("=" * 80)
print("CLEANED APEX TARGETS")
print("=" * 80)

for original, cleaned in zip(
    apex_training_records,
    cleaned_apex_training_records
):

    print(
        f'\nScenario {original["scenario_id"]:02d} | '
        f'{original["fault"]}'
    )

    print(
        "Original:",
        len(original["answer"]),
        "chars"
    )

    print(
        "Cleaned :",
        len(cleaned["answer"]),
        "chars"
    )

    print(
        "Removed :",
        len(original["answer"]) - len(cleaned["answer"]),
        "chars"
    )


# ------------------------------------------------------------
# Check that important diagnostic content remains
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("CONTENT VALIDATION")
print("=" * 80)

checks = {
    "18 records": len(cleaned_apex_training_records) == 18,

    "no empty answers": all(
        x["answer"].strip()
        for x in cleaned_apex_training_records
    ),

    "no confidence blocks": all(
        "CONFIDENCE:" not in x["answer"]
        for x in cleaned_apex_training_records
    ),

    "no evidence status blocks": all(
        "EVIDENCE STATUS:" not in x["answer"]
        for x in cleaned_apex_training_records
    ),

    "diagnostic content retained": all(
        len(x["answer"]) > 300
        for x in cleaned_apex_training_records
    )
}

for name, result in checks.items():
    print(
        f'{"✅" if result else "❌"} {name}'
    )


print("\n" + "=" * 80)
print("SAMPLE CLEANED ANSWER — F001")
print("=" * 80)

print(
    cleaned_apex_training_records[0]["answer"]
)

CLEANED APEX TARGETS

Scenario 01 | F001
Original: 1847 chars
Cleaned : 1702 chars
Removed : 145 chars

Scenario 02 | F001
Original: 1417 chars
Cleaned : 1233 chars
Removed : 184 chars

Scenario 03 | F001
Original: 1112 chars
Cleaned : 904 chars
Removed : 208 chars

Scenario 04 | F002
Original: 1185 chars
Cleaned : 1046 chars
Removed : 139 chars

Scenario 05 | F002
Original: 1303 chars
Cleaned : 1143 chars
Removed : 160 chars

Scenario 06 | F020
Original: 1495 chars
Cleaned : 1342 chars
Removed : 153 chars

Scenario 07 | F030
Original: 2626 chars
Cleaned : 2473 chars
Removed : 153 chars

Scenario 08 | F011
Original: 2142 chars
Cleaned : 1930 chars
Removed : 212 chars

Scenario 09 | F015
Original: 1394 chars
Cleaned : 1182 chars
Removed : 212 chars

Scenario 10 | F040
Original: 1042 chars
Cleaned : 977 chars
Removed : 65 chars

Scenario 11 | F045
Original: 1355 chars
Cleaned : 1160 chars
Removed : 195 chars

Scenario 12 | F050
Original: 1068 chars
Cleaned : 1003 chars
Removed : 65 chars

In [63]:
# Step 60 — Build concise, non-repetitive APEX training targets

def compact_answer(answer):

    # Remove meta sections that encourage repetitive generation
    answer = re.sub(
        r"\n*CONFIDENCE:\s*.*?(?=\n[A-Z][A-Z /_-]+:|\Z)",
        "",
        answer,
        flags=re.S
    )

    answer = re.sub(
        r"\n*EVIDENCE STATUS:\s*.*?(?=\n[A-Z][A-Z /_-]+:|\Z)",
        "",
        answer,
        flags=re.S
    )

    answer = re.sub(
        r"\n*DOCUMENTATION:\s*.*?(?=\n[A-Z][A-Z /_-]+:|\Z)",
        "",
        answer,
        flags=re.S
    )

    # Remove generic boilerplate safety sections
    answer = re.sub(
        r"\n*SAFETY:\s*Follow any explicit shutdown, isolation, LOTO, or escalation instruction contained in the documented steps\.",
        "",
        answer,
        flags=re.S
    )

    # Remove generic evidence-only conclusion language
    answer = re.sub(
        r"\n*EVIDENCE STATUS:\s*.*",
        "",
        answer,
        flags=re.S
    )

    # Clean whitespace
    answer = re.sub(r"[ \t]+\n", "\n", answer)
    answer = re.sub(r"\n{3,}", "\n\n", answer)

    return answer.strip()


# Build new records
compact_apex_records = []

for record in apex_training_records:

    cleaned = compact_answer(record["answer"])

    compact_apex_records.append({
        **record,
        "answer": cleaned
    })


# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

print("=" * 80)
print("COMPACT APEX TARGET VALIDATION")
print("=" * 80)

for record in compact_apex_records:

    answer = record["answer"]

    print(
        f'\nScenario {record["scenario_id"]:02d} | '
        f'{record["fault"]}'
    )

    print("Characters:", len(answer))

    print(
        "Confidence:",
        "❌" if "CONFIDENCE:" in answer else "✅"
    )

    print(
        "Evidence Status:",
        "❌" if "EVIDENCE STATUS:" in answer else "✅"
    )

    print(
        "Documentation:",
        "❌" if "DOCUMENTATION:" in answer else "✅"
    )


# ------------------------------------------------------------
# Show representative targets
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("SAMPLE — F001")
print("=" * 80)

print(compact_apex_records[0]["answer"])


print("\n" + "=" * 80)
print("SAMPLE — F002")
print("=" * 80)

print(compact_apex_records[3]["answer"])


print("\n" + "=" * 80)
print("SAMPLE — F030")
print("=" * 80)

print(compact_apex_records[6]["answer"])

COMPACT APEX TARGET VALIDATION

Scenario 01 | F001
Characters: 1694
Confidence: ✅
Evidence Status: ✅
Documentation: ✅

Scenario 02 | F001
Characters: 1228
Confidence: ✅
Evidence Status: ✅
Documentation: ✅

Scenario 03 | F001
Characters: 901
Confidence: ✅
Evidence Status: ✅
Documentation: ✅

Scenario 04 | F002
Characters: 1044
Confidence: ✅
Evidence Status: ✅
Documentation: ✅

Scenario 05 | F002
Characters: 1140
Confidence: ✅
Evidence Status: ✅
Documentation: ✅

Scenario 06 | F020
Characters: 1337
Confidence: ✅
Evidence Status: ✅
Documentation: ✅

Scenario 07 | F030
Characters: 2457
Confidence: ✅
Evidence Status: ✅
Documentation: ✅

Scenario 08 | F011
Characters: 1920
Confidence: ✅
Evidence Status: ✅
Documentation: ✅

Scenario 09 | F015
Characters: 1180
Confidence: ✅
Evidence Status: ✅
Documentation: ✅

Scenario 10 | F040
Characters: 858
Confidence: ✅
Evidence Status: ✅
Documentation: ✅

Scenario 11 | F045
Characters: 1157
Confidence: ✅
Evidence Status: ✅
Documentation: ✅

Scenario 12 |

In [64]:
# Step 61 — Analyze remaining structural patterns in APEX targets

from collections import Counter

print("=" * 80)
print("REMAINING STRUCTURAL PATTERN ANALYSIS")
print("=" * 80)

all_headers = Counter()

for record in compact_apex_records:

    answer = record["answer"]

    # Capture section headers such as:
    # FAULT:
    # DIAGNOSIS:
    # DIAGNOSTIC SEQUENCE:
    headers = re.findall(
        r"(?m)^([A-Z][A-Z0-9 /_-]+):\s*$",
        answer
    )

    all_headers.update(headers)

print("\nHeader frequency across 18 targets:")
for header, count in all_headers.most_common():
    print(f"{count:2d}x  {header}")


print("\n" + "=" * 80)
print("TARGET LENGTH DISTRIBUTION")
print("=" * 80)

lengths = [
    len(record["answer"])
    for record in compact_apex_records
]

print("Minimum:", min(lengths))
print("Maximum:", max(lengths))
print("Average:", round(sum(lengths) / len(lengths), 1))


print("\n" + "=" * 80)
print("ALL UNIQUE HEADERS")
print("=" * 80)

print(sorted(all_headers.keys()))


print("\n" + "=" * 80)
print("SCENARIO STRUCTURES")
print("=" * 80)

for record in compact_apex_records:

    headers = re.findall(
        r"(?m)^([A-Z][A-Z0-9 /_-]+):\s*$",
        record["answer"]
    )

    print(
        f'Scenario {record["scenario_id"]:02d} | '
        f'{record["fault"]} | '
        f'{headers}'
    )

REMAINING STRUCTURAL PATTERN ANALYSIS

Header frequency across 18 targets:
18x  CONCLUSION
 9x  DECISION LOGIC
 9x  DOCUMENTED CAUSES
 5x  FAULT
 5x  REASONING
 4x  SAFETY-CRITICAL FAULT
 4x  WHAT THE MANUAL ESTABLISHES
 3x  ROOT-CAUSE ANALYSIS
 2x  DIAGNOSTIC SEQUENCE
 2x  SAFETY
 2x  DIAGNOSTIC APPROACH
 2x  REQUIRED RESPONSE
 2x  DOCUMENTED CORRECTIVE SEQUENCE
 2x  SAFETY PRIORITY
 2x  EVIDENCE INTERPRETATION
 2x  DIAGNOSTIC PATH
 2x  PROCEDURE
 2x  PURPOSE
 2x  STEPS
 2x  DATA-INTEGRITY FAULT
 2x  CORRECTIVE PROCEDURE
 2x  DOCUMENTED CORRECTIVE STEPS
 2x  SAFETY REQUIREMENT
 1x  DIAGNOSIS
 1x  FAULT DISTINCTION
 1x  KEY DIFFERENCE
 1x  WHAT CAN BE CONCLUDED
 1x  WHAT CANNOT YET BE CONCLUDED
 1x  DOCUMENTED POSSIBILITIES
 1x  NEXT ACTION
 1x  FAULT DISAMBIGUATION
 1x  KEY DISTINCTION
 1x  HOW TO DISTINGUISH
 1x  WHAT THE FAULT INDICATES
 1x  IMPORTANT
 1x  DOCUMENTED POSSIBLE CAUSES
 1x  DIAGNOSTIC STEPS
 1x  CONSTRAINT CHECK
 1x  DOCUMENTED CONDITION
 1x  REQUIRED CONSTRAINTS
 1x  

In [65]:
# Step 62 — Rebuild APEX dataset from compact targets

from datasets import Dataset

# Convert compact records into the training message format
compact_dataset = Dataset.from_list([
    {
        "scenario_id": record["scenario_id"],
        "fault": record["fault"],
        "scenario_type": record["scenario_type"],
        "messages": [
            {
                "role": "system",
                "content": SYSTEM_PROMPT
            },
            {
                "role": "user",
                "content": record["prompt"]
            },
            {
                "role": "assistant",
                "content": record["answer"]
            }
        ]
    }
    for record in compact_apex_records
])


# ------------------------------------------------------------
# Apply Qwen chat template
# ------------------------------------------------------------

def format_compact_example(example):

    formatted = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False
    )

    return {
        "text": formatted
    }


formatted_compact_dataset = compact_dataset.map(
    format_compact_example
)


# ------------------------------------------------------------
# Inspect dataset
# ------------------------------------------------------------

print("=" * 80)
print("COMPACT APEX DATASET")
print("=" * 80)

print("Rows:", len(formatted_compact_dataset))
print(
    "Columns:",
    formatted_compact_dataset.column_names
)


# ------------------------------------------------------------
# Token statistics
# ------------------------------------------------------------

token_lengths = []

for example in formatted_compact_dataset:

    tokens = tokenizer(
        example["text"],
        add_special_tokens=False
    )["input_ids"]

    token_lengths.append(len(tokens))


print("\n" + "=" * 80)
print("TOKEN STATISTICS")
print("=" * 80)

print("Minimum:", min(token_lengths))
print("Maximum:", max(token_lengths))
print("Average:", round(sum(token_lengths) / len(token_lengths), 1))


# ------------------------------------------------------------
# Show one formatted example
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("FORMATTED EXAMPLE — SCENARIO 01")
print("=" * 80)

print(formatted_compact_dataset[0]["text"][:5000])

KeyError: 'scenario_type'

In [66]:
# Step 62 — Rebuild APEX dataset from compact targets
# Corrected: compact_apex_records does not contain scenario_type

from datasets import Dataset

compact_dataset = Dataset.from_list([
    {
        "scenario_id": record["scenario_id"],
        "fault": record["fault"],
        "messages": [
            {
                "role": "system",
                "content": SYSTEM_PROMPT
            },
            {
                "role": "user",
                "content": record["prompt"]
            },
            {
                "role": "assistant",
                "content": record["answer"]
            }
        ]
    }
    for record in compact_apex_records
])


# ------------------------------------------------------------
# Apply Qwen chat template
# ------------------------------------------------------------

def format_compact_example(example):

    formatted = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False
    )

    return {
        "text": formatted
    }


formatted_compact_dataset = compact_dataset.map(
    format_compact_example
)


# ------------------------------------------------------------
# Dataset check
# ------------------------------------------------------------

print("=" * 80)
print("COMPACT APEX DATASET")
print("=" * 80)

print("Rows:", len(formatted_compact_dataset))
print("Columns:", formatted_compact_dataset.column_names)


# ------------------------------------------------------------
# Token statistics
# ------------------------------------------------------------

token_lengths = []

for example in formatted_compact_dataset:

    tokens = tokenizer(
        example["text"],
        add_special_tokens=False
    )["input_ids"]

    token_lengths.append(len(tokens))


print("\n" + "=" * 80)
print("TOKEN STATISTICS")
print("=" * 80)

print("Minimum:", min(token_lengths))
print("Maximum:", max(token_lengths))
print("Average:", round(sum(token_lengths) / len(token_lengths), 1))


# ------------------------------------------------------------
# Show formatted example
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("FORMATTED EXAMPLE — SCENARIO 01")
print("=" * 80)

print(formatted_compact_dataset[0]["text"][:5000])

Map:   0%|          | 0/18 [00:00<?, ? examples/s]

COMPACT APEX DATASET
Rows: 18
Columns: ['scenario_id', 'fault', 'messages', 'text']

TOKEN STATISTICS
Minimum: 475
Maximum: 1688
Average: 784.9

FORMATTED EXAMPLE — SCENARIO 01
<|im_start|>system
You are APEX, an industrial troubleshooting AI.

Use supplied evidence as the source of truth.
Separate documented facts from inference.
Follow conditional diagnostic sequences.
Preserve explicit safety requirements.
Never invent technical facts.
Never claim an exact failed component without sufficient evidence.
If evidence is insufficient, state what is known and what cannot be concluded.<|im_end|>
<|im_start|>user
Machine:
DeltaWorks DX-200 Automated Press Brake

Observed fault:
F001 — DC Link Voltage Too High

Question:
What should be checked first, and what is the prescribed troubleshooting sequence?

Evidence from the maintenance manual:
F001 — DC Link Voltage Too High 
Description: The DC link bus inside the servo amplifier powering the backgauge axes has risen 
above 780 V DC. This is d

In [67]:
# Step 63 — Assistant-only labels for compact APEX dataset

def prepare_compact_assistant_only(example):

    text = example["text"]

    # Locate the assistant response start
    assistant_marker = "<|im_start|>assistant\n"

    if assistant_marker not in text:
        raise ValueError("Assistant marker not found")

    assistant_start = text.index(assistant_marker) + len(assistant_marker)

    # Tokenize the complete conversation
    tokenized = tokenizer(
        text,
        add_special_tokens=False,
        truncation=False
    )

    input_ids = tokenized["input_ids"]
    attention_mask = tokenized["attention_mask"]

    # Tokenize everything before the assistant answer
    prefix_ids = tokenizer(
        text[:assistant_start],
        add_special_tokens=False
    )["input_ids"]

    prefix_length = len(prefix_ids)

    # Assistant-only loss
    labels = [-100] * prefix_length + input_ids[prefix_length:]

    # Safety check
    if len(labels) != len(input_ids):
        raise ValueError("Label length mismatch")

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels
    }


assistant_only_dataset = formatted_compact_dataset.map(
    prepare_compact_assistant_only,
    remove_columns=["messages", "text"]
)


# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

print("=" * 80)
print("ASSISTANT-ONLY DATASET")
print("=" * 80)

print("Rows:", len(assistant_only_dataset))
print("Columns:", assistant_only_dataset.column_names)


print("\n" + "=" * 80)
print("LABEL STATISTICS")
print("=" * 80)

for i in range(len(assistant_only_dataset)):

    labels = assistant_only_dataset[i]["labels"]

    ignored = sum(1 for x in labels if x == -100)
    trainable = sum(1 for x in labels if x != -100)

    print(
        f"Scenario {i+1:02d} | "
        f"Total: {len(labels):4d} | "
        f"Ignored: {ignored:4d} | "
        f"Trainable: {trainable:4d}"
    )


# ------------------------------------------------------------
# Verify first trainable text
# ------------------------------------------------------------

first_labels = assistant_only_dataset[0]["labels"]

first_trainable = next(
    i for i, x in enumerate(first_labels)
    if x != -100
)

decoded_target = tokenizer.decode(
    assistant_only_dataset[0]["input_ids"][first_trainable:first_trainable + 120],
    skip_special_tokens=False
)

print("\n" + "=" * 80)
print("FIRST TRAINABLE TOKENS")
print("=" * 80)

print("First trainable position:", first_trainable)
print("Decoded target preview:")
print(decoded_target)

Map:   0%|          | 0/18 [00:00<?, ? examples/s]

ASSISTANT-ONLY DATASET
Rows: 18
Columns: ['scenario_id', 'fault', 'input_ids', 'attention_mask', 'labels']

LABEL STATISTICS
Scenario 01 | Total:  955 | Ignored:  546 | Trainable:  409
Scenario 02 | Total:  981 | Ignored:  698 | Trainable:  283
Scenario 03 | Total:  746 | Ignored:  545 | Trainable:  201
Scenario 04 | Total:  560 | Ignored:  317 | Trainable:  243
Scenario 05 | Total: 1582 | Ignored: 1328 | Trainable:  254
Scenario 06 | Total:  635 | Ignored:  337 | Trainable:  298
Scenario 07 | Total: 1235 | Ignored:  643 | Trainable:  592
Scenario 08 | Total: 1029 | Ignored:  592 | Trainable:  437
Scenario 09 | Total:  574 | Ignored:  319 | Trainable:  255
Scenario 10 | Total:  515 | Ignored:  329 | Trainable:  186
Scenario 11 | Total:  550 | Ignored:  296 | Trainable:  254
Scenario 12 | Total:  492 | Ignored:  306 | Trainable:  186
Scenario 13 | Total:  492 | Ignored:  299 | Trainable:  193
Scenario 14 | Total:  485 | Ignored:  274 | Trainable:  211
Scenario 15 | Total:  475 | Ignored

In [69]:
# Step 64 — Check trainable-token balance across APEX examples

trainable_lengths = []

for i in range(len(assistant_only_dataset)):

    labels = assistant_only_dataset[i]["labels"]

    trainable = sum(
        1 for token in labels
        if token != -100
    )

    trainable_lengths.append(trainable)


print("=" * 80)
print("TRAINABLE TOKEN BALANCE")
print("=" * 80)

print("Minimum:", min(trainable_lengths))
print("Maximum:", max(trainable_lengths))
print("Average:", round(sum(trainable_lengths) / len(trainable_lengths), 1))
print("Total trainable tokens:", sum(trainable_lengths))


print("\n" + "=" * 80)
print("SORTED BY TRAINABLE TOKEN COUNT")
print("=" * 80)

for idx in sorted(
    range(len(trainable_lengths)),
    key=lambda i: trainable_lengths[i],
    reverse=True
):

    print(
        f"Scenario {idx+1:02d} | "
        f"{compact_apex_records[idx]['fault']:4s} | "
        f"{trainable_lengths[idx]:4d} tokens"
    )


print("\n" + "=" * 80)
print("LONGEST EXAMPLES")
print("=" * 80)

longest = sorted(
    range(len(trainable_lengths)),
    key=lambda i: trainable_lengths[i],
    reverse=True
)[:3]

for idx in longest:
    print(
        f"Scenario {idx+1:02d} | "
        f"Trainable tokens: {trainable_lengths[idx]}"
    )

TRAINABLE TOKEN BALANCE
Minimum: 186
Maximum: 592
Average: 294.3
Total trainable tokens: 5298

SORTED BY TRAINABLE TOKEN COUNT
Scenario 07 | F030 |  592 tokens
Scenario 18 | F099 |  568 tokens
Scenario 08 | F011 |  437 tokens
Scenario 01 | F001 |  409 tokens
Scenario 16 | F080 |  302 tokens
Scenario 06 | F020 |  298 tokens
Scenario 02 | F001 |  283 tokens
Scenario 09 | F015 |  255 tokens
Scenario 05 | F002 |  254 tokens
Scenario 11 | F045 |  254 tokens
Scenario 04 | F002 |  243 tokens
Scenario 17 | F090 |  220 tokens
Scenario 14 | F060 |  211 tokens
Scenario 15 | F070 |  206 tokens
Scenario 03 | F001 |  201 tokens
Scenario 13 | F055 |  193 tokens
Scenario 10 | F040 |  186 tokens
Scenario 12 | F050 |  186 tokens

LONGEST EXAMPLES
Scenario 07 | Trainable tokens: 592
Scenario 18 | Trainable tokens: 568
Scenario 08 | Trainable tokens: 437


In [71]:
# Step 65 — Attach the final APEX LoRA adapter cleanly

from peft import LoraConfig, get_peft_model, PeftModel
from transformers import TrainingArguments

# ------------------------------------------------------------
# Verify current model state
# ------------------------------------------------------------

print("=" * 80)
print("MODEL STATE")
print("=" * 80)

print("Model class:", model.__class__.__name__)
print("Already PeftModel:", isinstance(model, PeftModel))


# ------------------------------------------------------------
# Attach LoRA only if not already attached
# ------------------------------------------------------------

if not isinstance(model, PeftModel):

    apex_lora_config = LoraConfig(
        r=16,
        lora_alpha=32,
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM",
        target_modules=[
            "q_proj",
            "k_proj",
            "v_proj",
            "o_proj",
            "gate_proj",
            "up_proj",
            "down_proj",
        ],
    )

    model = get_peft_model(
        model,
        apex_lora_config
    )

    print("\n✅ Fresh LoRA adapter attached.")

else:

    print("\n⚠️ LoRA adapter already attached.")
    print("Keeping the existing adapter to avoid stacking adapters.")


# ------------------------------------------------------------
# Trainable parameter check
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("TRAINABLE PARAMETERS")
print("=" * 80)

model.print_trainable_parameters()


# ------------------------------------------------------------
# Training configuration
# ------------------------------------------------------------

training_args = TrainingArguments(
    output_dir="./apex_model_v2",

    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,

    learning_rate=2e-4,
    num_train_epochs=3,

    gradient_checkpointing=True,

    fp16=True,

    optim="paged_adamw_8bit",

    logging_steps=1,
    save_strategy="no",
    report_to="none",

    seed=42,

    remove_unused_columns=False,
)


print("\n" + "=" * 80)
print("FINAL TRAINING CONFIG")
print("=" * 80)

print("Examples:", len(assistant_only_dataset))
print("Epochs:", training_args.num_train_epochs)
print("Batch size:", training_args.per_device_train_batch_size)
print("Gradient accumulation:", training_args.gradient_accumulation_steps)
print(
    "Effective batch size:",
    training_args.per_device_train_batch_size
    * training_args.gradient_accumulation_steps
)
print("Learning rate:", training_args.learning_rate)
print("FP16:", training_args.fp16)
print("Gradient checkpointing:", training_args.gradient_checkpointing)
print("Optimizer:", training_args.optim)
print("Output:", training_args.output_dir)

MODEL STATE
Model class: Qwen3ForCausalLM
Already PeftModel: False


/usr/local/lib/python3.13/dist-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/peft/tuners/tuners_utils.py:305: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(



✅ Fresh LoRA adapter attached.

TRAINABLE PARAMETERS
trainable params: 33,030,144 || all params: 4,055,498,240 || trainable%: 0.8145

FINAL TRAINING CONFIG
Examples: 18
Epochs: 3
Batch size: 1
Gradient accumulation: 4
Effective batch size: 4
Learning rate: 0.0002
FP16: True
Gradient checkpointing: True
Optimizer: OptimizerNames.PAGED_ADAMW_8BIT
Output: ./apex_model_v2


In [72]:
# Step 66 — Inspect hidden PEFT state before final training

print("=" * 80)
print("PEFT STATE INSPECTION")
print("=" * 80)

print("Model class:", model.__class__.__name__)
print("Has peft_config:", hasattr(model, "peft_config"))

if hasattr(model, "peft_config"):
    print("peft_config type:", type(model.peft_config))
    print("peft_config:", model.peft_config)

print("\nNamed modules containing 'lora':")

lora_modules = [
    name
    for name, module in model.named_modules()
    if "lora" in name.lower()
]

print("Count:", len(lora_modules))

for name in lora_modules[:20]:
    print(name)

print("\n" + "=" * 80)
print("MODEL MEMORY")
print("=" * 80)

if torch.cuda.is_available():
    free, total = torch.cuda.mem_get_info()
    print("Free VRAM:", round(free / 1024**3, 2), "GB")
    print("Used VRAM:", round((total - free) / 1024**3, 2), "GB")

PEFT STATE INSPECTION
Model class: PeftModelForCausalLM
Has peft_config: True
peft_config type: <class 'dict'>
peft_config: {'default': LoraConfig(task_type='CAUSAL_LM', peft_type=<PeftType.LORA: 'LORA'>, auto_mapping=None, peft_version='0.20.0', base_model_name_or_path='Qwen/Qwen3-4B', revision=None, inference_mode=False, r=16, target_modules={'up_proj', 'o_proj', 'down_proj', 'k_proj', 'v_proj', 'q_proj', 'gate_proj'}, exclude_modules=None, lora_alpha=32, lora_dropout=0.05, fan_in_fan_out=False, bias='none', use_rslora=False, modules_to_save=None, init_lora_weights=True, layers_to_transform=None, layers_pattern=None, rank_pattern={}, alpha_pattern={}, megatron_config=None, megatron_core='megatron.core', trainable_token_indices=None, loftq_config={}, eva_config=None, corda_config=None, lora_ga_config=None, use_dora=False, velora_config=None, alora_invocation_tokens=None, use_qalora=False, qalora_group_size=16, monteclora_config=None, layer_replication=None, runtime_config=LoraRuntimeC

In [74]:
# Step 67 — Reload clean 4-bit Qwen3-4B base model
# Corrected: recreate bnb_config first

import gc
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

# ------------------------------------------------------------
# Recreate the 4-bit quantization configuration
# ------------------------------------------------------------

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)


# ------------------------------------------------------------
# Reload clean base model
# ------------------------------------------------------------

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
)


# ------------------------------------------------------------
# Verify clean state
# ------------------------------------------------------------

print("=" * 80)
print("CLEAN BASE MODEL")
print("=" * 80)

print("Model class:", model.__class__.__name__)
print("Is PeftModel:", isinstance(model, PeftModel))
print("Has peft_config:", hasattr(model, "peft_config"))

if hasattr(model, "peft_config"):
    print("⚠️ Unexpected PEFT state detected.")
else:
    print("✅ No PEFT configuration detected.")


# ------------------------------------------------------------
# VRAM check
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("VRAM AFTER RELOAD")
print("=" * 80)

if torch.cuda.is_available():
    free, total = torch.cuda.mem_get_info()

    print("Free VRAM:", round(free / 1024**3, 2), "GB")
    print("Used VRAM:", round((total - free) / 1024**3, 2), "GB")

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLEAN BASE MODEL
Model class: Qwen3ForCausalLM
Is PeftModel: False
Has peft_config: False
✅ No PEFT configuration detected.

VRAM AFTER RELOAD
Free VRAM: 8.23 GB
Used VRAM: 6.33 GB


In [75]:
# Step 68 — Attach ONE clean LoRA adapter

from peft import LoraConfig, get_peft_model, PeftModel

apex_lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
)

model = get_peft_model(
    model,
    apex_lora_config
)

print("=" * 80)
print("FINAL APEX ADAPTER")
print("=" * 80)

print("Model class:", model.__class__.__name__)
print("Is PeftModel:", isinstance(model, PeftModel))

print("\nTrainable parameters:")
model.print_trainable_parameters()

print("\nAdapter names:")
print(list(model.peft_config.keys()))

print("\n" + "=" * 80)
print("VRAM")
print("=" * 80)

if torch.cuda.is_available():
    free, total = torch.cuda.mem_get_info()

    print("Free VRAM:", round(free / 1024**3, 2), "GB")
    print("Used VRAM:", round((total - free) / 1024**3, 2), "GB")

FINAL APEX ADAPTER
Model class: PeftModelForCausalLM
Is PeftModel: True

Trainable parameters:
trainable params: 33,030,144 || all params: 4,055,498,240 || trainable%: 0.8145

Adapter names:
['default']

VRAM
Free VRAM: 8.11 GB
Used VRAM: 6.45 GB


In [76]:
# Step 69 — Create the final APEX Trainer

from transformers import Trainer, DataCollatorForSeq2Seq

# ------------------------------------------------------------
# Data collator
# ------------------------------------------------------------

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True,
    label_pad_token_id=-100,
    return_tensors="pt",
)


# ------------------------------------------------------------
# Trainer
# ------------------------------------------------------------

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=assistant_only_dataset,
    data_collator=data_collator,
)


# ------------------------------------------------------------
# Verify Trainer
# ------------------------------------------------------------

print("=" * 80)
print("APEX TRAINER READY")
print("=" * 80)

print("Trainer created:", trainer is not None)
print("Training examples:", len(trainer.train_dataset))
print("Model:", trainer.model.__class__.__name__)
print("Adapter names:", list(model.peft_config.keys()))

print("\nTrainable parameters:")
model.print_trainable_parameters()

print("\nVRAM:")

if torch.cuda.is_available():
    free, total = torch.cuda.mem_get_info()
    print("Free VRAM:", round(free / 1024**3, 2), "GB")
    print("Used VRAM:", round((total - free) / 1024**3, 2), "GB")

APEX TRAINER READY
Trainer created: True
Training examples: 18
Model: PeftModelForCausalLM
Adapter names: ['default']

Trainable parameters:
trainable params: 33,030,144 || all params: 4,055,498,240 || trainable%: 0.8145

VRAM:
Free VRAM: 8.11 GB
Used VRAM: 6.45 GB


In [77]:
# Step 70 — Train APEX v2

print("=" * 80)
print("STARTING APEX V2 TRAINING")
print("=" * 80)

train_result = trainer.train()

print("\n" + "=" * 80)
print("APEX V2 TRAINING COMPLETE")
print("=" * 80)

print("Training loss:", train_result.training_loss)
print("Training steps:", train_result.global_step)
print("Training runtime:", train_result.metrics.get("train_runtime"))
print("Samples per second:", train_result.metrics.get("train_samples_per_second"))

STARTING APEX V2 TRAINING


ValueError: Unable to create tensor, you should probably activate truncation and/or padding with 'padding=True' 'truncation=True' to have batched tensors with the same length. Perhaps your features (`fault` in this case) have excessive nesting (inputs type `list` where type `int` is expected).

In [78]:
# Step 70 — Remove metadata columns before training

training_dataset = assistant_only_dataset.remove_columns(
    ["scenario_id", "fault"]
)

print("=" * 80)
print("TRAINING DATASET")
print("=" * 80)

print("Rows:", len(training_dataset))
print("Columns:", training_dataset.column_names)

# Verify that only numeric model inputs remain
for column in training_dataset.column_names:
    sample = training_dataset[0][column]

    print(
        f"{column:20s} | "
        f"type={type(sample).__name__} | "
        f"length={len(sample)}"
    )

print("\n" + "=" * 80)
print("READY FOR TRAINING")
print("=" * 80)

print("✅ Metadata columns removed")
print("✅ input_ids retained")
print("✅ attention_mask retained")
print("✅ labels retained")

TRAINING DATASET
Rows: 18
Columns: ['input_ids', 'attention_mask', 'labels']
input_ids            | type=list | length=955
attention_mask       | type=list | length=955
labels               | type=list | length=955

READY FOR TRAINING
✅ Metadata columns removed
✅ input_ids retained
✅ attention_mask retained
✅ labels retained


In [79]:
# Step 71 — Replace Trainer dataset with clean numeric-only dataset

trainer.train_dataset = training_dataset

print("=" * 80)
print("TRAINER DATASET FIXED")
print("=" * 80)

print("Training examples:", len(trainer.train_dataset))
print("Columns:", trainer.train_dataset.column_names)

# Confirm no string metadata remains
print("\nFirst example field types:")

for key, value in trainer.train_dataset[0].items():
    print(
        f"{key:20s} | "
        f"type={type(value).__name__} | "
        f"length={len(value)}"
    )

print("\n✅ Trainer now uses input_ids + attention_mask + labels only.")

TRAINER DATASET FIXED
Training examples: 18
Columns: ['input_ids', 'attention_mask', 'labels']

First example field types:
input_ids            | type=list | length=955
attention_mask       | type=list | length=955
labels               | type=list | length=955

✅ Trainer now uses input_ids + attention_mask + labels only.


In [80]:
# Step 72 — Train APEX v2

print("=" * 80)
print("STARTING APEX V2 TRAINING")
print("=" * 80)

train_result = trainer.train()

print("\n" + "=" * 80)
print("APEX V2 TRAINING COMPLETE")
print("=" * 80)

print("Training loss:", train_result.training_loss)
print("Training steps:", train_result.global_step)
print("Training runtime:", train_result.metrics.get("train_runtime"))
print(
    "Samples per second:",
    train_result.metrics.get("train_samples_per_second")
)

STARTING APEX V2 TRAINING


Step,Training Loss
1,1.373201
2,1.537529
3,1.227035
4,0.859909
5,0.779391
6,0.770798
7,0.643414
8,0.704096
9,0.897216
10,0.575656



APEX V2 TRAINING COMPLETE
Training loss: 0.8172749678293864
Training steps: 15
Training runtime: 162.2485
Samples per second: 0.333


In [81]:
# Step 73 — Save APEX v2 adapter

import os

FINAL_ADAPTER_PATH = "./apex_model_v2/final_adapter"

os.makedirs(FINAL_ADAPTER_PATH, exist_ok=True)

model.save_pretrained(FINAL_ADAPTER_PATH)
tokenizer.save_pretrained(FINAL_ADAPTER_PATH)

print("=" * 80)
print("APEX V2 ADAPTER SAVED")
print("=" * 80)

print("Path:", FINAL_ADAPTER_PATH)

print("\nFiles:")

for filename in sorted(os.listdir(FINAL_ADAPTER_PATH)):
    filepath = os.path.join(FINAL_ADAPTER_PATH, filename)
    size_kb = os.path.getsize(filepath) / 1024

    print(
        f"{filename:35s} "
        f"{size_kb:10.1f} KB"
    )

print("\n" + "=" * 80)
print("SAVE STATUS")
print("=" * 80)

required_files = [
    "adapter_config.json",
    "adapter_model.safetensors",
]

for filename in required_files:
    exists = os.path.exists(
        os.path.join(FINAL_ADAPTER_PATH, filename)
    )
    print(
        f'{"✅" if exists else "❌"} {filename}'
    )

APEX V2 ADAPTER SAVED
Path: ./apex_model_v2/final_adapter

Files:
README.md                                  5.1 KB
adapter_config.json                        1.1 KB
adapter_model.safetensors             129089.7 KB
chat_template.jinja                        4.1 KB
tokenizer.json                         11154.9 KB
tokenizer_config.json                      0.7 KB

SAVE STATUS
✅ adapter_config.json
✅ adapter_model.safetensors


In [82]:
# Step 74 — Verify APEX V2 adapter reload

from peft import PeftModel

print("=" * 80)
print("LOADING SAVED APEX V2 ADAPTER")
print("=" * 80)

# Load the adapter onto the current clean base model
reloaded_model = PeftModel.from_pretrained(
    model,
    FINAL_ADAPTER_PATH,
    is_trainable=False,
)

print("\nModel class:", type(reloaded_model).__name__)
print("Is PeftModel:", isinstance(reloaded_model, PeftModel))

print("\nAdapter names:", list(reloaded_model.peft_config.keys()))

print("\nTrainable parameters:")
reloaded_model.print_trainable_parameters()

print("\n" + "=" * 80)
print("RELOAD STATUS")
print("=" * 80)

print("✅ Saved adapter loaded successfully")

LOADING SAVED APEX V2 ADAPTER


/usr/local/lib/python3.13/dist-packages/peft/tuners/tuners_utils.py:305: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(



Model class: PeftModelForCausalLM
Is PeftModel: True

Adapter names: ['default']

Trainable parameters:
trainable params: 0 || all params: 4,055,498,240 || trainable%: 0.0000

RELOAD STATUS
✅ Saved adapter loaded successfully


/usr/local/lib/python3.13/dist-packages/peft/peft_model.py:665: UserWarning: Found missing adapter keys while loading the checkpoint: ['base_model.model.base_model.model.model.layers.0.self_attn.q_proj.lora_A.default.weight', 'base_model.model.base_model.model.model.layers.0.self_attn.q_proj.lora_B.default.weight', 'base_model.model.base_model.model.model.layers.0.self_attn.k_proj.lora_A.default.weight', 'base_model.model.base_model.model.model.layers.0.self_attn.k_proj.lora_B.default.weight', 'base_model.model.base_model.model.model.layers.0.self_attn.v_proj.lora_A.default.weight', 'base_model.model.base_model.model.model.layers.0.self_attn.v_proj.lora_B.default.weight', 'base_model.model.base_model.model.model.layers.0.self_attn.o_proj.lora_A.default.weight', 'base_model.model.base_model.model.model.layers.0.self_attn.o_proj.lora_B.default.weight', 'base_model.model.base_model.model.model.layers.0.mlp.gate_proj.lora_A.default.weight', 'base_model.model.base_model.model.model.layers.0

In [83]:
# Step 75 — Clean reload of saved APEX V2 adapter

import torch
from transformers import AutoModelForCausalLM
from peft import PeftModel

print("=" * 80)
print("CLEAN APEX V2 RELOAD")
print("=" * 80)

# Free the incorrectly stacked reload model
try:
    del reloaded_model
except:
    pass

torch.cuda.empty_cache()

# Recreate the 4-bit configuration
clean_bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

# Load a completely fresh Qwen3-4B base model
clean_base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=clean_bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

# Attach the SAVED APEX adapter
clean_apex_model = PeftModel.from_pretrained(
    clean_base_model,
    FINAL_ADAPTER_PATH,
    is_trainable=False,
)

print("\nModel class:", type(clean_apex_model).__name__)
print("Is PeftModel:", isinstance(clean_apex_model, PeftModel))

print("\nAdapter names:", list(clean_apex_model.peft_config.keys()))

print("\nTrainable parameters:")
clean_apex_model.print_trainable_parameters()

print("\n" + "=" * 80)
print("MODEL STRUCTURE CHECK")
print("=" * 80)

print("Base model:", type(clean_base_model).__name__)
print("APEX model:", type(clean_apex_model).__name__)

print("\n" + "=" * 80)
print("STATUS")
print("=" * 80)

print("✅ Fresh base model loaded")
print("✅ Saved adapter attached to clean base")
print("⚠️ No nested adapter reload")

CLEAN APEX V2 RELOAD


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]


Model class: PeftModelForCausalLM
Is PeftModel: True

Adapter names: ['default']

Trainable parameters:
trainable params: 0 || all params: 4,055,498,240 || trainable%: 0.0000

MODEL STRUCTURE CHECK
Base model: Qwen3ForCausalLM
APEX model: PeftModelForCausalLM

STATUS
✅ Fresh base model loaded
✅ Saved adapter attached to clean base
⚠️ No nested adapter reload


In [84]:
# Step 76 — APEX V2 inference test: complex fault disambiguation

import torch

clean_apex_model.eval()

test_prompt = """You are APEX, the high-tier industrial troubleshooting reasoning model.

Machine:
Delta DX-200 Automated Press Brake

Operator reports:
"The machine shows F001 during a backgauge move. The fault appeared after a rapid stop. Plant voltage has not yet been measured."

Retrieved manual evidence:

F001 — DC Link Voltage Too High
- DC link bus inside the servo amplifier powering the backgauge axes is above 780 V DC.
- This fault is distinct from F003 mains overvoltage.
- A likely cause is regenerative energy from a rapidly decelerating backgauge axis that cannot be dissipated fast enough.
- Probable causes:
  1. Braking resistor open or cable broken.
  2. BG.DECEL too aggressive / too short.
  3. Mains voltage already high, above 440 V AC, before regeneration.
- Corrective sequence:
  1. Measure mains at the main isolator. Required: 400 V ±10% (360–440 V).
  2. Power down and apply LOTO. Measure the braking resistor at R+ and RB. Required: 47 Ω ±10%.
  3. If the resistor is healthy, increase BG.DECEL by 25%.
  4. If the fault persists with normal mains and a good resistor, request DeltaWorks service for DC-link capacitor capacitance testing.

F003 — Mains Overvoltage
- Mains voltage above 440 V AC for more than 500 ms.
- This is a separate fault from F001.

Safety:
- Apply LOTO before electrical resistance measurements.
- Do not operate the machine if mains voltage is outside the specified range.

Question:
Determine what can be concluded from the evidence, distinguish F001 from F003, identify the most appropriate diagnostic sequence, and clearly state anything that cannot yet be concluded."""

messages = [
    {
        "role": "system",
        "content": "You are APEX, a high-tier industrial troubleshooting assistant. Reason from the supplied evidence. Do not invent machine-specific facts. Distinguish documented facts from inference and state when evidence is insufficient."
    },
    {
        "role": "user",
        "content": test_prompt
    }
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt"
).to(clean_apex_model.device)

with torch.no_grad():
    outputs = clean_apex_model.generate(
        inputs,
        max_new_tokens=700,
        do_sample=False,
        temperature=0.0,
        pad_token_id=tokenizer.eos_token_id,
    )

generated = outputs[0][inputs.shape[-1]:]

response = tokenizer.decode(
    generated,
    skip_special_tokens=True
)

print("=" * 80)
print("APEX V2 TEST RESPONSE")
print("=" * 80)
print(response)

[transformers] The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


AttributeError: 

In [85]:
# Step 76 — Corrected APEX V2 inference test

import torch

clean_apex_model.eval()

test_prompt = """You are APEX, the high-tier industrial troubleshooting reasoning model.

Machine:
Delta DX-200 Automated Press Brake

Operator reports:
"The machine shows F001 during a backgauge move. The fault appeared after a rapid stop. Plant voltage has not yet been measured."

Retrieved manual evidence:

F001 — DC Link Voltage Too High
- DC link bus inside the servo amplifier powering the backgauge axes is above 780 V DC.
- This fault is distinct from F003 mains overvoltage.
- A likely cause is regenerative energy from a rapidly decelerating backgauge axis that cannot be dissipated fast enough.
- Probable causes:
  1. Braking resistor open or cable broken.
  2. BG.DECEL too aggressive / too short.
  3. Mains voltage already high, above 440 V AC, before regeneration.
- Corrective sequence:
  1. Measure mains at the main isolator. Required: 400 V ±10% (360–440 V).
  2. Power down and apply LOTO. Measure the braking resistor at R+ and RB. Required: 47 Ω ±10%.
  3. If the resistor is healthy, increase BG.DECEL by 25%.
  4. If the fault persists with normal mains and a good resistor, request DeltaWorks service for DC-link capacitor capacitance testing.

F003 — Mains Overvoltage
- Mains voltage above 440 V AC for more than 500 ms.
- This is a separate fault from F001.

Safety:
- Apply LOTO before electrical resistance measurements.
- Do not operate the machine if mains voltage is outside the specified range.

Question:
Determine what can be concluded from the evidence, distinguish F001 from F003, identify the most appropriate diagnostic sequence, and clearly state anything that cannot yet be concluded."""

messages = [
    {
        "role": "system",
        "content": (
            "You are APEX, a high-tier industrial troubleshooting assistant. "
            "Reason only from supplied evidence. Do not invent machine-specific facts. "
            "Distinguish documented facts from inference and state when evidence is insufficient."
        )
    },
    {
        "role": "user",
        "content": test_prompt
    }
]

# IMPORTANT:
# Return only the tensor instead of a BatchEncoding object.
input_ids = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt"
).to(clean_apex_model.device)

print("Input tensor shape:", input_ids.shape)

with torch.no_grad():
    outputs = clean_apex_model.generate(
        input_ids=input_ids,
        max_new_tokens=700,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )

generated = outputs[0][input_ids.shape[-1]:]

response = tokenizer.decode(
    generated,
    skip_special_tokens=True
)

print("\n" + "=" * 80)
print("APEX V2 TEST RESPONSE")
print("=" * 80)
print(response)

AttributeError: 

In [86]:
# Step 76 — APEX V2 inference test
# Explicitly extract input_ids from BatchEncoding

import torch

clean_apex_model.eval()

test_prompt = """You are APEX, the high-tier industrial troubleshooting reasoning model.

Machine:
Delta DX-200 Automated Press Brake

Operator reports:
"The machine shows F001 during a backgauge move. The fault appeared after a rapid stop. Plant voltage has not yet been measured."

Retrieved manual evidence:

F001 — DC Link Voltage Too High
- DC link bus inside the servo amplifier powering the backgauge axes is above 780 V DC.
- This fault is distinct from F003 mains overvoltage.
- A likely cause is regenerative energy from a rapidly decelerating backgauge axis that cannot be dissipated fast enough.
- Probable causes:
  1. Braking resistor open or cable broken.
  2. BG.DECEL too aggressive / too short.
  3. Mains voltage already high, above 440 V AC, before regeneration.
- Corrective sequence:
  1. Measure mains at the main isolator. Required: 400 V ±10% (360–440 V).
  2. Power down and apply LOTO. Measure the braking resistor at R+ and RB. Required: 47 Ω ±10%.
  3. If the resistor is healthy, increase BG.DECEL by 25%.
  4. If the fault persists with normal mains and a good resistor, request DeltaWorks service for DC-link capacitor capacitance testing.

F003 — Mains Overvoltage
- Mains voltage above 440 V AC for more than 500 ms.
- This is a separate fault from F001.

Safety:
- Apply LOTO before electrical resistance measurements.
- Do not operate the machine if mains voltage is outside the specified range.

Question:
Determine what can be concluded from the evidence, distinguish F001 from F003, identify the most appropriate diagnostic sequence, and clearly state anything that cannot yet be concluded."""

messages = [
    {
        "role": "system",
        "content": (
            "You are APEX, a high-tier industrial troubleshooting assistant. "
            "Reason only from supplied evidence. Do not invent machine-specific facts. "
            "Distinguish documented facts from inference and state when evidence is insufficient."
        )
    },
    {
        "role": "user",
        "content": test_prompt
    }
]

# Qwen chat template
encoded = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt"
)

# Explicitly obtain the tensor
if hasattr(encoded, "input_ids"):
    input_ids = encoded["input_ids"]
else:
    input_ids = encoded

input_ids = input_ids.to(clean_apex_model.device)

print("Input type:", type(input_ids).__name__)
print("Input tensor shape:", input_ids.shape)

# Generate
with torch.no_grad():
    outputs = clean_apex_model.generate(
        input_ids=input_ids,
        max_new_tokens=700,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )

# Remove prompt tokens
generated = outputs[0][input_ids.shape[-1]:]

response = tokenizer.decode(
    generated,
    skip_special_tokens=True
)

print("\n" + "=" * 80)
print("APEX V2 TEST RESPONSE")
print("=" * 80)
print(response)

Input type: Tensor
Input tensor shape: torch.Size([1, 470])

APEX V2 TEST RESPONSE
<think>

</think>

FAULT CODE:
F001 — DC Link Voltage Too High

CONCLUSION:
The fault code F001 is the correct one to apply.

REASONING:
The manual evidence explicitly defines F001 as the correct fault code for the described condition.

DIAGNOSTIC SEQUENCE:
1. Measure mains at the main isolator. Required: 400 V ±10% (360–440 V).
2. Power down and apply LOTO. Measure the braking resistor at R+ and RB. Required: 47 Ω ±10%.
3. If the resistor is healthy, increase BG.DECEL by 25%.
4. If the fault persists with normal mains and a good resistor, request DeltaWorks service for DC-link capacitor capacitance testing.

REASONING:
The manual evidence provides the correct diagnostic sequence for F001.

CONCLUSION:
F003 — Mains Overvoltage
- Mains voltage above 440 V AC for more than 500 ms.
- This is a separate fault from F001.

REASONING:
The manual evidence explicitly defines F003 as a separate fault code.

CONCLU

In [87]:
# Step 77 — APEX V2 generation stability test

clean_apex_model.eval()

with torch.no_grad():
    outputs = clean_apex_model.generate(
        input_ids=input_ids,
        max_new_tokens=500,
        do_sample=False,

        # Prevent the model from repeatedly copying the same phrases
        repetition_penalty=1.15,
        no_repeat_ngram_size=4,

        pad_token_id=tokenizer.eos_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

generated = outputs[0][input_ids.shape[-1]:]

response = tokenizer.decode(
    generated,
    skip_special_tokens=True
)

print("=" * 80)
print("APEX V2 STABILITY TEST")
print("=" * 80)
print(response)

APEX V2 STABILITY TEST
<think>

</think>

FAULT CODE ANALYSIS:
F002 — DC LINK VOLTAGE TOO HIGH
- Fault code F002 indicates an issue with the DC link voltage in the power supply system.
- The fault should not be confused with F004 which relates to motor current overload.

DIAGNOSTIC SEQUENCE:
1. Check the DC link bus voltage using the multimeter.
   - Expected value: 785 V DC ±10%
   - Document the actual reading.

CONCLUSION:
The fault condition must be investigated according to the documented procedure rather than assumed based on the reported symptom alone.

DOCUMENTED CAUSES:
1. Brakes resistor failure
2. Deceleration setting too low
3. Main voltage out of specification

DECISION LOGIC:
Use the documented corrective steps as the basis for diagnosis instead of making assumptions about the root cause.

SAFETY:
Apply LOTO before performing any electrical measurement.

REASONING:
Do not assume the fault condition without verifying it against the documented criteria. Use the provided inf

In [88]:
# Step 78 — APEX V2 seen-example sanity test

clean_apex_model.eval()

seen_prompt = """You are APEX, the high-tier industrial troubleshooting reasoning model.

Machine:
Delta DX-200 Automated Press Brake

Fault:
F001 — DC Link Voltage Too High

Manual evidence:
- DC link bus inside the servo amplifier powering the backgauge axes is above 780 V DC.
- This fault is distinct from F003 mains overvoltage.
- Probable causes:
  1. Braking resistor open or cable broken.
  2. BG.DECEL too aggressive / too short.
  3. Mains voltage already high, above 440 V AC before regeneration.
- Corrective sequence:
  1. Measure mains at the main isolator. Required: 400 V ±10% (360–440 V).
  2. Power down and apply LOTO. Measure braking resistor at R+ and RB. Required: 47 Ω ±10%.
  3. If healthy, increase BG.DECEL by 25%.
  4. If the fault persists with normal mains and a good resistor, request DeltaWorks service for DC-link capacitor capacitance testing.

Question:
Give the documented diagnostic sequence for F001. Do not introduce facts that are not present in the evidence."""

messages_seen = [
    {
        "role": "system",
        "content": (
            "You are APEX, a high-tier industrial troubleshooting assistant. "
            "Use only the supplied evidence. Do not invent values, fault codes, "
            "components, or procedures."
        )
    },
    {
        "role": "user",
        "content": seen_prompt
    }
]

encoded_seen = tokenizer.apply_chat_template(
    messages_seen,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt"
)

input_seen = (
    encoded_seen["input_ids"]
    if hasattr(encoded_seen, "input_ids")
    else encoded_seen
)

input_seen = input_seen.to(clean_apex_model.device)

with torch.no_grad():
    output_seen = clean_apex_model.generate(
        input_ids=input_seen,
        max_new_tokens=350,
        do_sample=False,
        repetition_penalty=1.15,
        no_repeat_ngram_size=4,
        pad_token_id=tokenizer.eos_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

generated_seen = output_seen[0][input_seen.shape[-1]:]

response_seen = tokenizer.decode(
    generated_seen,
    skip_special_tokens=True
)

print("=" * 80)
print("APEX V2 SEEN-EXAMPLE TEST")
print("=" * 80)
print(response_seen)

APEX V2 SEEN-EXAMPLE TEST
<think>

</think>

FAULT CODE:
F01 — "DC Link Voltage Too Low"

DIAGNOSTIC SEQUENCE:

1. Check the DC link bus voltage on the servo amplifier. Required: below 785 V DC.
2. Determine if this fault is different from F03 mains under-voltage.
3. Possible causes:
   - Faulty DC link capacitor.
   - Shorted DC link bus.
   - Fault in the motor control circuitry.
4. Corrective steps:
   - Replace the DC link capacitor if faulty.
   - Inspect the DC link wiring for shorts.
   - Repair any faults in the motor drive system.

CONCLUSION:
The correct procedure should be based directly on the documented fault code and its associated manual evidence. The provided answer must match exactly what was presented in the original question.


In [89]:
# Step 79 — Inspect the exact F001 training example

print("=" * 80)
print("F001 TRAINING EXAMPLE")
print("=" * 80)

# Find the first F001 scenario
for i, example in enumerate(scenarios):
    if example["fault"] == "F001":
        print("\nScenario index:", i)
        print("Scenario ID:", example["scenario_id"])
        print("Fault:", example["fault"])

        print("\n" + "-" * 80)
        print("PROMPT")
        print("-" * 80)
        print(example["prompt"])

        print("\n" + "-" * 80)
        print("TARGET")
        print("-" * 80)
        print(example["answer"])

        break
else:
    print("❌ F001 scenario not found")

F001 TRAINING EXAMPLE


NameError: name 'scenarios' is not defined

In [90]:
# Step 79 — Inspect the actual tokenized training sample

print("=" * 80)
print("TRAINING DATASET INSPECTION")
print("=" * 80)

print("Rows:", len(training_dataset))
print("Columns:", training_dataset.column_names)

# Decode every training example and look for F001
found = False

for i in range(len(training_dataset)):
    sample = training_dataset[i]

    # Decode the full sequence
    decoded = tokenizer.decode(
        sample["input_ids"],
        skip_special_tokens=False
    )

    if "F001" in decoded:
        found = True

        print("\n" + "=" * 80)
        print("FOUND F001 TRAINING SAMPLE")
        print("=" * 80)

        print("Dataset index:", i)

        print("\n" + "-" * 80)
        print("DECODED TRAINING SAMPLE")
        print("-" * 80)

        print(decoded)

        print("\n" + "-" * 80)
        print("LABEL MASK CHECK")
        print("-" * 80)

        labels = sample["labels"]

        trainable_tokens = sum(
            1 for x in labels if x != -100
        )

        ignored_tokens = sum(
            1 for x in labels if x == -100
        )

        print("Total tokens:", len(labels))
        print("Ignored tokens:", ignored_tokens)
        print("Trainable tokens:", trainable_tokens)

        break

if not found:
    print("\n❌ No training sample containing F001 was found.")

TRAINING DATASET INSPECTION
Rows: 18
Columns: ['input_ids', 'attention_mask', 'labels']

FOUND F001 TRAINING SAMPLE
Dataset index: 0

--------------------------------------------------------------------------------
DECODED TRAINING SAMPLE
--------------------------------------------------------------------------------
<|im_start|>system
You are APEX, an industrial troubleshooting AI.

Use supplied evidence as the source of truth.
Separate documented facts from inference.
Follow conditional diagnostic sequences.
Preserve explicit safety requirements.
Never invent technical facts.
Never claim an exact failed component without sufficient evidence.
If evidence is insufficient, state what is known and what cannot be concluded.<|im_end|>
<|im_start|>user
Machine:
DeltaWorks DX-200 Automated Press Brake

Observed fault:
F001 — DC Link Voltage Too High

Question:
What should be checked first, and what is the prescribed troubleshooting sequence?

Evidence from the maintenance manual:
F001 — DC 

In [91]:
# Step 80 — Verify assistant-only labels exactly
print("=" * 80)
print("ASSISTANT LABEL VERIFICATION")
print("=" * 80)

idx = 0
sample = training_dataset[idx]

input_ids = sample["input_ids"]
labels = sample["labels"]

# Decode only tokens that are actually trainable
trainable_ids = [
    token_id
    for token_id, label in zip(input_ids, labels)
    if label != -100
]

trainable_text = tokenizer.decode(
    trainable_ids,
    skip_special_tokens=False
)

print("Trainable token count:", len(trainable_ids))
print("\n" + "-" * 80)
print("ACTUAL TRAINABLE TARGET")
print("-" * 80)
print(trainable_text)

print("\n" + "-" * 80)
print("CHECKS")
print("-" * 80)

checks = {
    "Contains F001": "F001" in trainable_text,
    "Contains correct title": "DC Link Voltage Too High" in trainable_text,
    "Contains 400 V ±10%": "400 V ±10%" in trainable_text,
    "Contains 47 Ω ±10%": "47 Ω ±10%" in trainable_text,
    "Contains BG.DECEL": "BG.DECEL" in trainable_text,
    "Contains LOTO": "LOTO" in trainable_text,
    "Contains F01": "F01" in trainable_text,
    "Contains 785 V": "785 V" in trainable_text,
    "Contains F004": "F004" in trainable_text,
}

for name, result in checks.items():
    print(f"{name:<35}: {result}")

print("\n" + "=" * 80)

ASSISTANT LABEL VERIFICATION
Trainable token count: 409

--------------------------------------------------------------------------------
ACTUAL TRAINABLE TARGET
--------------------------------------------------------------------------------
<think>

</think>

FAULT:
F001 — DC Link Voltage Too High

DIAGNOSIS:
The manual identifies these probable causes:
1. The braking resistor on the servo amplifier has gone open-circuit or its connecting cable is
broken.
2. The backgauge deceleration ramp (parameter BG.DECEL in the drive menu) is set too
aggressively short.
3. Mains supply voltage is already at the high end of tolerance (>440 V AC) before
regeneration occurs.

DIAGNOSTIC SEQUENCE:
1. Measure the mains supply at the main isolator terminals — must be 400 V ±10% (360–440
V). If high, notify the facility electrician before proceeding.
2. Power down (LOTO) and measure the braking resistor resistance at the servo amplifier
terminals R+ and RB; the nominal value is 47 Ω ±10%. Replace if op

In [92]:
# Step 81 — Compare clean base vs trained APEX adapter
print("=" * 80)
print("BASE MODEL vs APEX ADAPTER")
print("=" * 80)

test_text = """Machine: DeltaWorks DX-200 Automated Press Brake

Observed fault: F001 — DC Link Voltage Too High

What should be checked first?"""

messages = [
    {
        "role": "system",
        "content": (
            "You are APEX, an industrial troubleshooting AI. "
            "Use supplied evidence as the source of truth. "
            "Never invent technical facts."
        ),
    },
    {
        "role": "user",
        "content": test_text,
    },
]

encoded = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    return_tensors="pt",
)

if hasattr(encoded, "input_ids"):
    input_ids = encoded["input_ids"]
else:
    input_ids = encoded

input_ids = input_ids.to(clean_apex_model.device)

# ---------------------------------------------------------
# BASE MODEL
# ---------------------------------------------------------
print("\n" + "-" * 80)
print("1. CLEAN BASE MODEL")
print("-" * 80)

with torch.no_grad():
    base_output = clean_base_model.generate(
        input_ids=input_ids,
        max_new_tokens=120,
        do_sample=False,
        repetition_penalty=1.15,
        no_repeat_ngram_size=4,
    )

base_new_tokens = base_output[0][input_ids.shape[1]:]

print(
    tokenizer.decode(
        base_new_tokens,
        skip_special_tokens=False
    )
)

# ---------------------------------------------------------
# APEX ADAPTER
# ---------------------------------------------------------
print("\n" + "-" * 80)
print("2. APEX ADAPTER")
print("-" * 80)

with torch.no_grad():
    apex_output = clean_apex_model.generate(
        input_ids=input_ids,
        max_new_tokens=120,
        do_sample=False,
        repetition_penalty=1.15,
        no_repeat_ngram_size=4,
    )

apex_new_tokens = apex_output[0][input_ids.shape[1]:]

print(
    tokenizer.decode(
        apex_new_tokens,
        skip_special_tokens=False
    )
)

# ---------------------------------------------------------
# ADAPTER STATUS
# ---------------------------------------------------------
print("\n" + "-" * 80)
print("ADAPTER STATUS")
print("-" * 80)

print("Base model class:", type(clean_base_model).__name__)
print("APEX model class:", type(clean_apex_model).__name__)
print("Adapter names:", clean_apex_model.peft_config.keys())

trainable = sum(
    p.numel()
    for p in clean_apex_model.parameters()
    if p.requires_grad
)

print("Trainable parameters:", trainable)
print("=" * 80)

BASE MODEL vs APEX ADAPTER

--------------------------------------------------------------------------------
1. CLEAN BASE MODEL
--------------------------------------------------------------------------------
<think>

</think>

FAULT CODE:
F001 - DC LINK VOLTAGE TOO HIGH

CAUSES:
1. POWER SUPPLY FAULT
2. MOTOR DRIVE FAULT
3. INVERTER FAULT
4. CIRCUIT BOARD FAULT

PROCEDURE:
1. CHECK THE POWER SUPPLY
   a. VERIFY THE INPUT LINE VOLTAGE
   b. CHECK FOR ANY SHORTS OR OVERLOADS ON THE POWER LINE
   c. CHECK THE OUTPUT LINE VOLTAGES OF THE POWER SUPPLIES
   d. CHECK THE CURRENT LIMITING

--------------------------------------------------------------------------------
2. APEX ADAPTER
--------------------------------------------------------------------------------
<think>

</think>

FAULT CODE:
F001 - DC LINK VOLTAGE TOO HIGH

CAUSES:
1. POWER SUPPLY FAULT
2. MOTOR DRIVE FAULT
3. INVERTER FAULT
4. CIRCUIT BOARD FAULT

PROCEDURE:
1. CHECK THE POWER SUPPLY
   a. VERIFY THE INPUT LINE VOLTAGE
 

In [93]:
# Step 82 — Inspect actual saved LoRA weights
print("=" * 80)
print("APEX ADAPTER WEIGHT INSPECTION")
print("=" * 80)

adapter_path = FINAL_ADAPTER_PATH

print("Adapter path:", adapter_path)

# Load the saved adapter state directly
from safetensors.torch import load_file

adapter_file = os.path.join(
    adapter_path,
    "adapter_model.safetensors"
)

state = load_file(adapter_file)

print("Number of tensors:", len(state))

print("\n" + "-" * 80)
print("FIRST 10 ADAPTER TENSORS")
print("-" * 80)

for i, (name, tensor) in enumerate(state.items()):
    if i >= 10:
        break

    print(
        f"{name}\n"
        f"  shape={tuple(tensor.shape)}"
        f"  dtype={tensor.dtype}"
        f"  mean={tensor.float().mean().item():.8f}"
        f"  std={tensor.float().std().item():.8f}"
        f"  abs_mean={tensor.float().abs().mean().item():.8f}"
        f"  nonzero={torch.count_nonzero(tensor).item()}"
    )

print("\n" + "-" * 80)
print("GLOBAL ADAPTER STATISTICS")
print("-" * 80)

total_params = 0
total_nonzero = 0
sum_abs = 0.0

for tensor in state.values():
    t = tensor.float()
    total_params += t.numel()
    total_nonzero += torch.count_nonzero(t).item()
    sum_abs += t.abs().sum().item()

print("Total adapter parameters:", total_params)
print("Non-zero parameters:", total_nonzero)
print("Non-zero ratio:", total_nonzero / total_params)
print("Mean absolute weight:", sum_abs / total_params)

print("\n" + "=" * 80)

APEX ADAPTER WEIGHT INSPECTION
Adapter path: ./apex_model_v2/final_adapter
Number of tensors: 504

--------------------------------------------------------------------------------
FIRST 10 ADAPTER TENSORS
--------------------------------------------------------------------------------
base_model.model.model.layers.0.mlp.down_proj.lora_A.weight
  shape=(16, 9728)  dtype=torch.float32  mean=0.00000486  std=0.00588731  abs_mean=0.00508567  nonzero=155648
base_model.model.model.layers.0.mlp.down_proj.lora_B.weight
  shape=(2560, 16)  dtype=torch.float32  mean=0.00000190  std=0.00071023  abs_mean=0.00060447  nonzero=40960
base_model.model.model.layers.0.mlp.gate_proj.lora_A.weight
  shape=(16, 2560)  dtype=torch.float32  mean=-0.00002610  std=0.01148728  abs_mean=0.00994279  nonzero=40960
base_model.model.model.layers.0.mlp.gate_proj.lora_B.weight
  shape=(9728, 16)  dtype=torch.float32  mean=0.00000273  std=0.00070937  abs_mean=0.00060505  nonzero=155648
base_model.model.model.layers.0.mlp

In [94]:
# Step 83 — Verify that the loaded LoRA adapter changes model outputs
print("=" * 80)
print("MEASURING LORA EFFECT")
print("=" * 80)

# Use the same test input
test_text = """Machine: DeltaWorks DX-200 Automated Press Brake

Observed fault: F001 — DC Link Voltage Too High

What should be checked first?"""

messages = [
    {
        "role": "system",
        "content": (
            "You are APEX, an industrial troubleshooting AI. "
            "Use supplied evidence as the source of truth. "
            "Never invent technical facts."
        ),
    },
    {
        "role": "user",
        "content": test_text,
    },
]

encoded = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    return_tensors="pt",
)

if hasattr(encoded, "input_ids"):
    input_ids = encoded["input_ids"]
else:
    input_ids = encoded

input_ids = input_ids.to(clean_apex_model.device)

# Get logits from clean base
with torch.no_grad():
    base_logits = clean_base_model(
        input_ids=input_ids
    ).logits[:, -1, :].float()

# Get logits from APEX adapter
with torch.no_grad():
    apex_logits = clean_apex_model(
        input_ids=input_ids
    ).logits[:, -1, :].float()

# Compare
difference = (apex_logits - base_logits).abs()

print("\nBase logits shape:", tuple(base_logits.shape))
print("APEX logits shape:", tuple(apex_logits.shape))

print("\n" + "-" * 80)
print("LOGIT DIFFERENCE")
print("-" * 80)

print("Maximum absolute difference:", difference.max().item())
print("Mean absolute difference:", difference.mean().item())
print("Number of different values:", torch.count_nonzero(difference).item())
print(
    "Percentage different:",
    100 * torch.count_nonzero(difference).item() / difference.numel()
)

# Compare top predicted tokens
base_top = torch.topk(base_logits, 10, dim=-1).indices[0]
apex_top = torch.topk(apex_logits, 10, dim=-1).indices[0]

print("\n" + "-" * 80)
print("TOP 10 NEXT TOKENS")
print("-" * 80)

print("\nBASE:")
for token_id in base_top:
    print(
        token_id.item(),
        repr(tokenizer.decode([token_id.item()]))
    )

print("\nAPEX:")
for token_id in apex_top:
    print(
        token_id.item(),
        repr(tokenizer.decode([token_id.item()]))
    )

print("\n" + "=" * 80)

MEASURING LORA EFFECT

Base logits shape: (1, 151936)
APEX logits shape: (1, 151936)

--------------------------------------------------------------------------------
LOGIT DIFFERENCE
--------------------------------------------------------------------------------
Maximum absolute difference: 0.0
Mean absolute difference: 0.0
Number of different values: 0
Percentage different: 0.0

--------------------------------------------------------------------------------
TOP 10 NEXT TOKENS
--------------------------------------------------------------------------------

BASE:
151667 '<think>'
151668 '</think>'
80445 '\t\n\t\n\t\n\t\n'
151645 '<|im_end|>'
50950 '\xa0\n'
64321 'itä'
151644 '<|im_start|>'
24015 'ông'
3224 '.\r\n'
8997 '。\n'

APEX:
151667 '<think>'
151668 '</think>'
80445 '\t\n\t\n\t\n\t\n'
151645 '<|im_end|>'
50950 '\xa0\n'
64321 'itä'
151644 '<|im_start|>'
24015 'ông'
3224 '.\r\n'
8997 '。\n'



In [95]:
# Step 84 — Check PEFT adapter activation state
print("=" * 80)
print("PEFT ADAPTER ACTIVATION CHECK")
print("=" * 80)

print("Model:", type(clean_apex_model).__name__)
print("Active adapter:", clean_apex_model.active_adapter)
print("Available adapters:", list(clean_apex_model.peft_config.keys()))

# Check whether adapters are globally disabled
print("\nDisable-adapter state:")
print(clean_apex_model._disable_adapters)

# Inspect a few actual LoRA modules
print("\n" + "-" * 80)
print("LORA MODULE CHECK")
print("-" * 80)

checked = 0

for name, module in clean_apex_model.named_modules():
    if hasattr(module, "lora_A") and hasattr(module, "lora_B"):
        print("\nModule:", name)
        print("  LoRA A adapters:", list(module.lora_A.keys()))
        print("  LoRA B adapters:", list(module.lora_B.keys()))

        if hasattr(module, "disable_adapters"):
            print("  disable_adapters:", module.disable_adapters)

        if hasattr(module, "active_adapters"):
            print("  active_adapters:", module.active_adapters)

        if hasattr(module, "scaling"):
            print("  scaling:", module.scaling)

        checked += 1

        if checked >= 3:
            break

print("\nLoRA modules inspected:", checked)
print("=" * 80)


PEFT ADAPTER ACTIVATION CHECK
Model: PeftModelForCausalLM
Active adapter: default
Available adapters: ['default']

Disable-adapter state:


AttributeError: 'Qwen3ForCausalLM' object has no attribute '_disable_adapters'

In [96]:
# Step 85 — Inspect actual loaded LoRA modules
print("=" * 80)
print("LOADED LORA MODULE INSPECTION")
print("=" * 80)

found = 0

for name, module in clean_apex_model.named_modules():
    if hasattr(module, "lora_A") and hasattr(module, "lora_B"):

        print("\n" + "-" * 80)
        print("Module:", name)
        print("-" * 80)

        print("LoRA A adapters:", list(module.lora_A.keys()))
        print("LoRA B adapters:", list(module.lora_B.keys()))

        adapter_name = "default"

        A = module.lora_A[adapter_name].weight.detach().float()
        B = module.lora_B[adapter_name].weight.detach().float()

        print("A shape:", tuple(A.shape))
        print("B shape:", tuple(B.shape))

        print("A mean abs:", A.abs().mean().item())
        print("B mean abs:", B.abs().mean().item())

        if hasattr(module, "scaling"):
            print("Scaling:", module.scaling)

        if hasattr(module, "disable_adapters"):
            print("disable_adapters:", module.disable_adapters)

        if hasattr(module, "active_adapters"):
            print("active_adapters:", module.active_adapters)

        found += 1

        if found >= 3:
            break

print("\n" + "=" * 80)
print("LoRA modules inspected:", found)
print("=" * 80)

LOADED LORA MODULE INSPECTION

--------------------------------------------------------------------------------
Module: base_model.model.model.layers.0.self_attn.q_proj
--------------------------------------------------------------------------------
LoRA A adapters: ['default']
LoRA B adapters: ['default']
A shape: (16, 2560)
B shape: (4096, 16)
A mean abs: 0.00989482831209898
B mean abs: 0.000608623493462801
Scaling: {'default': 2.0}
disable_adapters: False
active_adapters: ['default']

--------------------------------------------------------------------------------
Module: base_model.model.model.layers.0.self_attn.k_proj
--------------------------------------------------------------------------------
LoRA A adapters: ['default']
LoRA B adapters: ['default']
A shape: (16, 2560)
B shape: (1024, 16)
A mean abs: 0.00986738782376051
B mean abs: 0.0005905110738240182
Scaling: {'default': 2.0}
disable_adapters: False
active_adapters: ['default']

--------------------------------------------

In [97]:
# Step 86 — Directly verify adapter ON vs OFF
print("=" * 80)
print("DIRECT LORA ON/OFF TEST")
print("=" * 80)

test_text = """Machine: DeltaWorks DX-200 Automated Press Brake

Observed fault: F001 — DC Link Voltage Too High

What should be checked first?"""

messages = [
    {
        "role": "system",
        "content": (
            "You are APEX, an industrial troubleshooting AI. "
            "Use supplied evidence as the source of truth. "
            "Never invent technical facts."
        ),
    },
    {
        "role": "user",
        "content": test_text,
    },
]

encoded = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    return_tensors="pt",
)

if hasattr(encoded, "input_ids"):
    input_ids = encoded["input_ids"]
else:
    input_ids = encoded

input_ids = input_ids.to(clean_apex_model.device)

# ---------------------------------------------------------
# ADAPTER ON
# ---------------------------------------------------------
clean_apex_model.set_adapter("default")

with torch.no_grad():
    logits_on = clean_apex_model(
        input_ids=input_ids
    ).logits[:, -1, :].float()

# ---------------------------------------------------------
# ADAPTER OFF
# ---------------------------------------------------------
with clean_apex_model.disable_adapter():
    with torch.no_grad():
        logits_off = clean_apex_model(
            input_ids=input_ids
        ).logits[:, -1, :].float()

# ---------------------------------------------------------
# COMPARE
# ---------------------------------------------------------
difference = (logits_on - logits_off).abs()

print("\nAdapter ON/OFF difference:")
print("Maximum:", difference.max().item())
print("Mean:", difference.mean().item())
print("Non-zero:", torch.count_nonzero(difference).item())

print("\n" + "-" * 80)
print("TOP TOKEN WITH ADAPTER ON")
print("-" * 80)

top_on = torch.topk(logits_on, 10, dim=-1).indices[0]

for token_id in top_on:
    print(
        token_id.item(),
        repr(tokenizer.decode([token_id.item()]))
    )

print("\n" + "-" * 80)
print("TOP TOKEN WITH ADAPTER OFF")
print("-" * 80)

top_off = torch.topk(logits_off, 10, dim=-1).indices[0]

for token_id in top_off:
    print(
        token_id.item(),
        repr(tokenizer.decode([token_id.item()]))
    )

print("\n" + "=" * 80)

DIRECT LORA ON/OFF TEST

Adapter ON/OFF difference:
Maximum: 24.765625
Mean: 4.390427589416504
Non-zero: 151869

--------------------------------------------------------------------------------
TOP TOKEN WITH ADAPTER ON
--------------------------------------------------------------------------------
151667 '<think>'
151668 '</think>'
80445 '\t\n\t\n\t\n\t\n'
151645 '<|im_end|>'
50950 '\xa0\n'
64321 'itä'
151644 '<|im_start|>'
24015 'ông'
3224 '.\r\n'
8997 '。\n'

--------------------------------------------------------------------------------
TOP TOKEN WITH ADAPTER OFF
--------------------------------------------------------------------------------
151667 '<think>'
151668 '</think>'
32803 'aines'
151644 '<|im_start|>'
9684 'ени'
125032 'ệnh'
15835 'ICES'
51677 'icamente'
94963 'ức'
85070 'AINED'



In [98]:
# Step 87 — Exact training-example reproduction test
print("=" * 80)
print("APEX TRAINING-MEMORIZATION TEST")
print("=" * 80)

# Recover the original F001 training example
original_sample = assistant_only_dataset[0]

decoded_training = tokenizer.decode(
    original_sample["input_ids"],
    skip_special_tokens=False
)

# Extract the user portion from the original training example
user_start = decoded_training.find("<|im_start|>user\n")
user_end = decoded_training.find("<|im_end|>", user_start)

user_text = decoded_training[
    user_start + len("<|im_start|>user\n"):
    user_end
]

print("\nTraining example user prompt length:", len(user_text))

# Build the exact same conversation format
messages = [
    {
        "role": "system",
        "content": (
            "You are APEX, an industrial troubleshooting AI.\n\n"
            "Use supplied evidence as the source of truth.\n"
            "Separate documented facts from inference.\n"
            "Follow conditional diagnostic sequences.\n"
            "Preserve explicit safety requirements.\n"
            "Never invent technical facts.\n"
            "Never claim an exact failed component without sufficient evidence.\n"
            "If evidence is insufficient, state what is known and what cannot be concluded."
        ),
    },
    {
        "role": "user",
        "content": user_text,
    },
]

encoded = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    return_tensors="pt",
)

if hasattr(encoded, "input_ids"):
    input_ids = encoded["input_ids"]
else:
    input_ids = encoded

input_ids = input_ids.to(clean_apex_model.device)

# Generate
with torch.no_grad():
    output = clean_apex_model.generate(
        input_ids=input_ids,
        max_new_tokens=450,
        do_sample=False,
        repetition_penalty=1.15,
        no_repeat_ngram_size=4,
    )

generated = output[0][input_ids.shape[1]:]

generated_text = tokenizer.decode(
    generated,
    skip_special_tokens=False
)

print("\n" + "-" * 80)
print("APEX OUTPUT ON EXACT TRAINING EXAMPLE")
print("-" * 80)
print(generated_text)

print("\n" + "-" * 80)
print("EXPECTED TRAINING TARGET")
print("-" * 80)

# Extract assistant target from the original formatted sample
assistant_start = decoded_training.find("<|im_start|>assistant\n")

expected_text = decoded_training[
    assistant_start + len("<|im_start|>assistant\n"):
]

print(expected_text)

print("\n" + "=" * 80)

APEX TRAINING-MEMORIZATION TEST

Training example user prompt length: 1802

--------------------------------------------------------------------------------
APEX OUTPUT ON EXACT TRAINING EXAMPLE
--------------------------------------------------------------------------------
<think>

</think>

FAULT CODE:
F01 — Fault Code Description

TRIAGE SEQUENCE:
The correct procedure for this fault code is outlined in the relevant section of the maintenance manual.

DOCUMENTED FAULT CAUSES:
The following causes have been identified that could lead to this fault:

CAUSE 1:
Braking resistor failure

CAUSE 2:
Backgauge decel ramp setting

CAUSE3:
Mains supply voltage above limit

DIAGNOSTIC ORDER:
The appropriate order of investigation is given below:

STEP 1:
Check the mains supply voltage using the specified method.

STEP 2:
Measure the braking resistor's resistance.

STEP 3:
Adjust the backgague decel ramp parameter.

STEP 4:
Investigate the capacitors if necessary.

CONCLUSION:
Do not assume any

In [99]:
# Step 88 — Inspect exactly where the assistant labels begin/end
print("=" * 80)
print("LABEL BOUNDARY INSPECTION")
print("=" * 80)

sample = training_dataset[0]

input_ids = sample["input_ids"]
labels = sample["labels"]

# Find first trainable token
first_trainable = next(
    i for i, x in enumerate(labels)
    if x != -100
)

# Find last trainable token
last_trainable = max(
    i for i, x in enumerate(labels)
    if x != -100
)

print("First trainable index:", first_trainable)
print("Last trainable index:", last_trainable)

print("\n" + "-" * 80)
print("TOKENS AROUND LABEL START")
print("-" * 80)

start = max(0, first_trainable - 15)
end = min(len(input_ids), first_trainable + 25)

for i in range(start, end):
    token = tokenizer.decode([input_ids[i]], skip_special_tokens=False)
    marker = " <-- TRAINABLE START" if i == first_trainable else ""
    print(
        f"{i:4d} | label={labels[i]:6d} | {repr(token)}{marker}"
    )

print("\n" + "-" * 80)
print("TOKENS AROUND LABEL END")
print("-" * 80)

start = max(0, last_trainable - 20)
end = min(len(input_ids), last_trainable + 10)

for i in range(start, end):
    token = tokenizer.decode([input_ids[i]], skip_special_tokens=False)
    marker = " <-- LAST TRAINABLE" if i == last_trainable else ""
    print(
        f"{i:4d} | label={labels[i]:6d} | {repr(token)}{marker}"
    )

print("\n" + "-" * 80)
print("FINAL LABEL TOKEN")
print("-" * 80)

final_token = tokenizer.decode(
    [input_ids[last_trainable]],
    skip_special_tokens=False
)

print("Token:", repr(final_token))
print("Token ID:", input_ids[last_trainable])

print("\n" + "=" * 80)

LABEL BOUNDARY INSPECTION
First trainable index: 546
Last trainable index: 954

--------------------------------------------------------------------------------
TOKENS AROUND LABEL START
--------------------------------------------------------------------------------
 531 | label=  -100 | '-S'
 532 | label=  -100 | 'CH'
 533 | label=  -100 | '-D'
 534 | label=  -100 | 'X'
 535 | label=  -100 | '2'
 536 | label=  -100 | '0'
 537 | label=  -100 | '0'
 538 | label=  -100 | ' Sheet'
 539 | label=  -100 | ' '
 540 | label=  -100 | '5'
 541 | label=  -100 | '<|im_end|>'
 542 | label=  -100 | '\n'
 543 | label=  -100 | '<|im_start|>'
 544 | label=  -100 | 'assistant'
 545 | label=  -100 | '\n'
 546 | label=151667 | '<think>' <-- TRAINABLE START
 547 | label=   271 | '\n\n'
 548 | label=151668 | '</think>'
 549 | label=   271 | '\n\n'
 550 | label=  5291 | 'FAULT'
 551 | label=   510 | ':\n'
 552 | label=    37 | 'F'
 553 | label=    15 | '0'
 554 | label=    15 | '0'
 555 | label=    16 | '1'

In [100]:
# Step 89 — Full-evidence APEX test
print("=" * 80)
print("FULL EVIDENCE TEST")
print("=" * 80)

messages = [
    {
        "role": "system",
        "content": """You are APEX, an industrial troubleshooting AI.

Use supplied evidence as the source of truth.
Separate documented facts from inference.
Follow conditional diagnostic sequences.
Preserve explicit safety requirements.
Never invent technical facts.
Never claim an exact failed component without sufficient evidence.
If evidence is insufficient, state what is known and what cannot be concluded."""
    },
    {
        "role": "user",
        "content": """Machine:
DeltaWorks DX-200 Automated Press Brake

Observed fault:
F001 — DC Link Voltage Too High

Question:
What should be checked first, and what is the prescribed troubleshooting sequence?

Evidence from the maintenance manual:
F001 — DC Link Voltage Too High
Description: The DC link bus inside the servo amplifier powering the backgauge axes has risen above 780 V DC. This is distinct from mains overvoltage (see F003). The condition is caused by regenerative energy returning from a rapidly decelerating backgauge axis being unable to dissipate fast enough. The servo amplifier shuts down all backgauge axes to protect the power stage.

Probable Causes:
1. The braking resistor on the servo amplifier has gone open-circuit or its connecting cable is broken.
2. The backgauge deceleration ramp (parameter BG.DECEL in the drive menu) is set too aggressively short.
3. Mains supply voltage is already at the high end of tolerance (>440 V AC) before regeneration occurs.

Corrective Steps:
1. Measure the mains supply at the main isolator terminals — must be 400 V ±10% (360–440 V). If high, notify the facility electrician before proceeding.
2. Power down (LOTO) and measure the braking resistor resistance at the servo amplifier terminals R+ and RB; nominal value is 47 Ω ±10%. Replace if open or out of tolerance (Part No. DW-BRK-RES).
3. If the resistor is healthy, increase BG.DECEL from the current value by 25% (e.g. 0.2 s → 0.25 s).
4. If the fault persists at normal mains voltage with a good braking resistor, the DC link capacitors may be degraded — request DeltaWorks service for a capacitance test.

Reference: Section 7.2, Servo Amplifier; Electrical Schematic DW-SCH-DX200 Sheet 5"""
    }
]

encoded = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    return_tensors="pt"
)

if hasattr(encoded, "input_ids"):
    input_ids = encoded["input_ids"]
else:
    input_ids = encoded

input_ids = input_ids.to(clean_apex_model.device)

with torch.no_grad():
    output = clean_apex_model.generate(
        input_ids=input_ids,
        max_new_tokens=350,
        do_sample=False,
        repetition_penalty=1.15,
        no_repeat_ngram_size=4,
    )

generated = output[0][input_ids.shape[1]:]

print("\n" + "-" * 80)
print("APEX OUTPUT")
print("-" * 80)

print(
    tokenizer.decode(
        generated,
        skip_special_tokens=False
    )
)

print("\n" + "=" * 80)

FULL EVIDENCE TEST

--------------------------------------------------------------------------------
APEX OUTPUT
--------------------------------------------------------------------------------
<think>

</think>

FAULT CODE:
F01 — Fault Code: DC Link VoltageToo High

DOCUMENTED CAUSES:
The following causes have been identified in the relevant section of the maintenance manual:

1. Braking resistor failure or loose connection
2. Deceleration parameter setting that is too aggressive
3. Mismatched mains supply voltage

DIAGNOSTIC SEQUENCE:
Apply the documented procedure in order until the root cause is found.

PROCEDURE:
1. Check the mains supply voltage using the specified method.
   - If it exceeds the acceptable range, contact the electrical team.
   - Otherwise, continue to the next step.

2. Use LOTO procedures to isolate the machine.
   - Test the braking resistor's continuity between R+ and Rb.
     - If there is no continuity, replace the resistor.
     - Else, proceed to the next

In [101]:
# Step 90 — Compare training loss: base vs APEX
print("=" * 80)
print("BASE vs APEX TRAINING LOSS")
print("=" * 80)

sample = training_dataset[0]

input_ids = torch.tensor(
    [sample["input_ids"]],
    dtype=torch.long,
    device=clean_apex_model.device
)

attention_mask = torch.tensor(
    [sample["attention_mask"]],
    dtype=torch.long,
    device=clean_apex_model.device
)

labels = torch.tensor(
    [sample["labels"]],
    dtype=torch.long,
    device=clean_apex_model.device
)

# ---------------------------------------------------------
# BASE MODEL LOSS
# ---------------------------------------------------------
with torch.no_grad():
    base_result = clean_base_model(
        input_ids=input_ids,
        attention_mask=attention_mask,
        labels=labels,
    )

# ---------------------------------------------------------
# APEX LOSS
# ---------------------------------------------------------
clean_apex_model.set_adapter("default")

with torch.no_grad():
    apex_result = clean_apex_model(
        input_ids=input_ids,
        attention_mask=attention_mask,
        labels=labels,
    )

print("\nBase model loss:", float(base_result.loss))
print("APEX model loss:", float(apex_result.loss))

print("\nLoss improvement:")
print(
    float(base_result.loss) - float(apex_result.loss)
)

print("\nPercentage improvement:")
if float(base_result.loss) != 0:
    print(
        100
        * (
            float(base_result.loss) - float(apex_result.loss)
        )
        / float(base_result.loss)
    )
else:
    print("N/A")

print("\n" + "=" * 80)

BASE vs APEX TRAINING LOSS

Base model loss: 0.3945554196834564
APEX model loss: 0.3945554196834564

Loss improvement:
0.0

Percentage improvement:
0.0



In [102]:
# Step 91 — Reset CUDA state before APEX v3
import gc
import torch

print("=" * 80)
print("RESETTING APEX TRAINING STATE")
print("=" * 80)

# Delete training/inference model objects from previous run
for var_name in [
    "trainer",
    "model",
    "clean_apex_model",
    "clean_base_model",
]:
    if var_name in globals():
        try:
            del globals()[var_name]
            print("Deleted:", var_name)
        except:
            pass

gc.collect()
torch.cuda.empty_cache()
torch.cuda.ipc_collect()

print("\nCUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print(
        f"GPU: {torch.cuda.get_device_name(0)}"
    )
    print(
        f"VRAM allocated: "
        f"{torch.cuda.memory_allocated() / 1024**3:.2f} GB"
    )
    print(
        f"VRAM reserved: "
        f"{torch.cuda.memory_reserved() / 1024**3:.2f} GB"
    )
    print(
        f"VRAM free: "
        f"{(torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated()) / 1024**3:.2f} GB"
    )

print("\n" + "=" * 80)
print("RESET COMPLETE")
print("=" * 80)

RESETTING APEX TRAINING STATE
Deleted: trainer
Deleted: model
Deleted: clean_apex_model
Deleted: clean_base_model

CUDA available: True
GPU: Tesla T4
VRAM allocated: 9.43 GB
VRAM reserved: 9.83 GB
VRAM free: 5.13 GB

RESET COMPLETE


In [103]:
# Step 92 — APEX v3 configuration
from peft import LoraConfig

APEX_V3_LORA_CONFIG = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
)

# Training hyperparameters for v3
APEX_V3_LEARNING_RATE = 5e-5
APEX_V3_EPOCHS = 3
APEX_V3_BATCH_SIZE = 1
APEX_V3_GRAD_ACCUMULATION = 4

print("=" * 80)
print("APEX V3 CONFIGURATION")
print("=" * 80)

print("\nLoRA:")
print("  Rank:", APEX_V3_LORA_CONFIG.r)
print("  Alpha:", APEX_V3_LORA_CONFIG.lora_alpha)
print("  Dropout:", APEX_V3_LORA_CONFIG.lora_dropout)
print("  Target modules:", len(APEX_V3_LORA_CONFIG.target_modules))

print("\nTraining:")
print("  Learning rate:", APEX_V3_LEARNING_RATE)
print("  Epochs:", APEX_V3_EPOCHS)
print("  Batch size:", APEX_V3_BATCH_SIZE)
print("  Gradient accumulation:", APEX_V3_GRAD_ACCUMULATION)

print("\n" + "=" * 80)
print("CONFIG READY")
print("=" * 80)

APEX V3 CONFIGURATION

LoRA:
  Rank: 16
  Alpha: 32
  Dropout: 0.05
  Target modules: 7

Training:
  Learning rate: 5e-05
  Epochs: 3
  Batch size: 1
  Gradient accumulation: 4

CONFIG READY


In [104]:
# Step 93 — Load a completely fresh Qwen3-4B base for APEX v3

import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

MODEL_NAME = "Qwen/Qwen3-4B"

clean_bnb_config_v3 = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print("=" * 80)
print("LOADING FRESH APEX V3 BASE MODEL")
print("=" * 80)

apex_v3_base = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=clean_bnb_config_v3,
    device_map="auto",
    trust_remote_code=True,
)

print("\nModel class:", type(apex_v3_base).__name__)
print("Is PEFT model:", "PeftModel" in type(apex_v3_base).__name__)

print("\n" + "-" * 80)
print("GPU MEMORY")
print("-" * 80)

print(
    f"Allocated: "
    f"{torch.cuda.memory_allocated() / 1024**3:.2f} GB"
)

print(
    f"Reserved: "
    f"{torch.cuda.memory_reserved() / 1024**3:.2f} GB"
)

print("\n" + "=" * 80)
print("FRESH BASE READY")
print("=" * 80)

LOADING FRESH APEX V3 BASE MODEL


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]


Model class: Qwen3ForCausalLM
Is PEFT model: False

--------------------------------------------------------------------------------
GPU MEMORY
--------------------------------------------------------------------------------
Allocated: 11.93 GB
Reserved: 11.99 GB

FRESH BASE READY


In [105]:
# Step 94 — Attach fresh APEX v3 LoRA adapter

from peft import get_peft_model

print("=" * 80)
print("ATTACHING APEX V3 LORA")
print("=" * 80)

apex_v3_model = get_peft_model(
    apex_v3_base,
    APEX_V3_LORA_CONFIG
)

print("\nModel class:", type(apex_v3_model).__name__)
print("Is PEFT model:", "PeftModel" in type(apex_v3_model).__name__)
print("Adapter names:", list(apex_v3_model.peft_config.keys()))

print("\n" + "-" * 80)
print("TRAINABLE PARAMETERS")
print("-" * 80)

trainable_params = sum(
    p.numel()
    for p in apex_v3_model.parameters()
    if p.requires_grad
)

total_params = sum(
    p.numel()
    for p in apex_v3_model.parameters()
)

print("Trainable:", trainable_params)
print("Total:", total_params)
print(
    "Trainable %:",
    100 * trainable_params / total_params
)

print("\n" + "-" * 80)
print("GPU MEMORY")
print("-" * 80)

print(
    f"Allocated: "
    f"{torch.cuda.memory_allocated() / 1024**3:.2f} GB"
)

print(
    f"Reserved: "
    f"{torch.cuda.memory_reserved() / 1024**3:.2f} GB"
)

print("\n" + "=" * 80)
print("APEX V3 ADAPTER READY")
print("=" * 80)

ATTACHING APEX V3 LORA

Model class: PeftModelForCausalLM
Is PEFT model: True
Adapter names: ['default']

--------------------------------------------------------------------------------
TRAINABLE PARAMETERS
--------------------------------------------------------------------------------
Trainable: 33030144
Total: 2238840320
Trainable %: 1.4753237962053498

--------------------------------------------------------------------------------
GPU MEMORY
--------------------------------------------------------------------------------
Allocated: 12.05 GB
Reserved: 12.12 GB

APEX V3 ADAPTER READY


In [107]:
# Step 95 — Check which APEX dataset objects still exist

print("=" * 80)
print("APEX DATASET STATE")
print("=" * 80)

for name in [
    "assistant_only_dataset",
    "training_dataset",
    "fault_df",
    "full_text",
]:
    exists = name in globals()
    print(f"{name:<25}: {'AVAILABLE' if exists else 'MISSING'}")

if "assistant_only_dataset" in globals():
    print("\nassistant_only_dataset:")
    print("  Rows:", len(assistant_only_dataset))
    print("  Columns:", assistant_only_dataset.column_names)

if "training_dataset" in globals():
    print("\ntraining_dataset:")
    print("  Rows:", len(training_dataset))
    print("  Columns:", training_dataset.column_names)

if "fault_df" in globals():
    print("\nfault_df:")
    print("  Shape:", fault_df.shape)
    print("  Columns:", list(fault_df.columns))

print("\n" + "=" * 80)

APEX DATASET STATE
assistant_only_dataset   : AVAILABLE
training_dataset         : AVAILABLE
fault_df                 : AVAILABLE
full_text                : AVAILABLE

assistant_only_dataset:
  Rows: 18
  Columns: ['scenario_id', 'fault', 'input_ids', 'attention_mask', 'labels']

training_dataset:
  Rows: 18
  Columns: ['input_ids', 'attention_mask', 'labels']

fault_df:
  Shape: (21, 3)
  Columns: ['code', 'title', 'section']



In [108]:
# Step 96 — Inspect all existing APEX training targets

print("=" * 80)
print("EXISTING APEX TRAINING TARGETS")
print("=" * 80)

for i in range(len(assistant_only_dataset)):

    sample = assistant_only_dataset[i]

    input_ids = sample["input_ids"]
    labels = sample["labels"]

    # Decode only trainable tokens
    target_ids = [
        token_id
        for token_id, label in zip(input_ids, labels)
        if label != -100
    ]

    target_text = tokenizer.decode(
        target_ids,
        skip_special_tokens=False
    )

    # Remove template markers for easier inspection
    target_text = target_text.replace(
        "<|im_end|>", ""
    ).strip()

    print("\n" + "=" * 80)
    print(f"TARGET {i + 1}")
    print("=" * 80)
    print(target_text)

print("\n" + "=" * 80)
print("INSPECTION COMPLETE")
print("=" * 80)

EXISTING APEX TRAINING TARGETS

TARGET 1
<think>

</think>

FAULT:
F001 — DC Link Voltage Too High

DIAGNOSIS:
The manual identifies these probable causes:
1. The braking resistor on the servo amplifier has gone open-circuit or its connecting cable is
broken.
2. The backgauge deceleration ramp (parameter BG.DECEL in the drive menu) is set too
aggressively short.
3. Mains supply voltage is already at the high end of tolerance (>440 V AC) before
regeneration occurs.

DIAGNOSTIC SEQUENCE:
1. Measure the mains supply at the main isolator terminals — must be 400 V ±10% (360–440
V). If high, notify the facility electrician before proceeding.
2. Power down (LOTO) and measure the braking resistor resistance at the servo amplifier
terminals R+ and RB; the nominal value is 47 Ω ±10%. Replace if open or out of tolerance
(Part No. DW-BRK-RES).
3. If the resistor is healthy, open the servo drive parameter menu and increase BG.DECEL
from the current value by 25% (e.g., 0.2 s → 0.25 s) to reduce peak

In [109]:
# Step 97 — Reconstruct clean scenario records from the existing dataset

import re

apex_v3_records = []

for i in range(len(assistant_only_dataset)):
    sample = assistant_only_dataset[i]

    input_ids = sample["input_ids"]
    labels = sample["labels"]

    decoded = tokenizer.decode(
        input_ids,
        skip_special_tokens=False
    )

    # Extract user message
    user_start = decoded.find("<|im_start|>user\n")
    user_end = decoded.find("<|im_end|>", user_start)

    user_text = decoded[
        user_start + len("<|im_start|>user\n"):
        user_end
    ].strip()

    # Extract assistant target
    assistant_start = decoded.find("<|im_start|>assistant\n")
    assistant_end = decoded.find("<|im_end|>", assistant_start)

    target_text = decoded[
        assistant_start + len("<|im_start|>assistant\n"):
        assistant_end
    ].strip()

    # Remove Qwen thinking wrapper from target
    target_text = re.sub(
        r"^<think>\s*</think>\s*",
        "",
        target_text,
        flags=re.DOTALL
    ).strip()

    apex_v3_records.append({
        "index": i,
        "user": user_text,
        "target": target_text,
    })

print("=" * 80)
print("APEX V3 RECORDS")
print("=" * 80)

print("Records:", len(apex_v3_records))

for record in apex_v3_records:
    print(
        f"\n{record['index'] + 1:02d} | "
        f"user={len(record['user'])} chars | "
        f"target={len(record['target'])} chars"
    )

print("\n" + "=" * 80)
print("RECONSTRUCTION COMPLETE")
print("=" * 80)

APEX V3 RECORDS
Records: 18

01 | user=1802 chars | target=1694 chars

02 | user=2405 chars | target=1228 chars

03 | user=1791 chars | target=901 chars

04 | user=948 chars | target=1044 chars

05 | user=4804 chars | target=1140 chars

06 | user=1106 chars | target=1337 chars

07 | user=2229 chars | target=2457 chars

08 | user=2210 chars | target=1920 chars

09 | user=1046 chars | target=1180 chars

10 | user=1039 chars | target=858 chars

11 | user=905 chars | target=1157 chars

12 | user=994 chars | target=884 chars

13 | user=937 chars | target=868 chars

14 | user=761 chars | target=928 chars

15 | user=822 chars | target=1027 chars

16 | user=1071 chars | target=1340 chars

17 | user=807 chars | target=1026 chars

18 | user=3994 chars | target=2465 chars

RECONSTRUCTION COMPLETE


In [110]:
print("=" * 100)
print("APEX V3 TRAINING PAIRS")
print("=" * 100)

for r in apex_v3_records:
    print(f"\n{'='*100}")
    print(f"SCENARIO {r['index'] + 1:02d}")
    print(f"{'='*100}")

    print("\nUSER:")
    print(r["user"])

    print("\nTARGET:")
    print(r["target"])

APEX V3 TRAINING PAIRS

SCENARIO 01

USER:
Machine:
DeltaWorks DX-200 Automated Press Brake

Observed fault:
F001 — DC Link Voltage Too High

Question:
What should be checked first, and what is the prescribed troubleshooting sequence?

Evidence from the maintenance manual:
F001 — DC Link Voltage Too High 
Description: The DC link bus inside the servo amplifier powering the backgauge axes has risen 
above 780 V DC. This is distinct from mains overvoltage (see F003). The condition is caused by 
regenerative energy returning from a rapidly decelerating backgauge axis being unable to dissipate 
fast enough. The servo amplifier shuts down all backgauge axes to protect the power stage. 
Probable Causes: 
 The braking resistor on the servo amplifier has gone open-circuit or its connecting cable is 
broken. 
 The backgauge deceleration ramp (parameter BG.DECEL in the drive menu) is set too 
aggressively short. 

 Mains supply voltage is already at the high end of tolerance (>440 V AC) befor

In [111]:
from difflib import SequenceMatcher
import re

print("=" * 100)
print("V3 EVIDENCE → TARGET OVERLAP")
print("=" * 100)

for r in apex_v3_records:
    user = r["user"]
    target = r["target"]

    # Evidence is everything after the evidence header
    evidence_match = re.search(
        r"Evidence from the maintenance manual:\s*(.*?)(?:\n\nTARGET:|\Z)",
        user,
        flags=re.DOTALL | re.IGNORECASE
    )

    evidence = evidence_match.group(1).strip() if evidence_match else ""

    # Normalize text
    def normalize(text):
        text = text.lower()
        text = re.sub(r"\s+", " ", text)
        return text.strip()

    e_norm = normalize(evidence)
    t_norm = normalize(target)

    # Overall similarity
    similarity = SequenceMatcher(None, e_norm, t_norm).ratio()

    # Sentence-level exact/near overlap
    evidence_sentences = [
        s.strip() for s in re.split(r"(?<=[.!?])\s+", evidence)
        if len(s.strip()) > 30
    ]

    target_sentences = [
        s.strip() for s in re.split(r"(?<=[.!?])\s+", target)
        if len(s.strip()) > 30
    ]

    copied = 0
    for ts in target_sentences:
        best = max(
            [SequenceMatcher(None, normalize(ts), normalize(es)).ratio()
             for es in evidence_sentences],
            default=0
        )
        if best >= 0.85:
            copied += 1

    copy_ratio = copied / max(len(target_sentences), 1)

    print(
        f"{r['index']+1:02d} | "
        f"overall_similarity={similarity:.2f} | "
        f"near_copied_sentences={copy_ratio:.0%} | "
        f"target_sentences={len(target_sentences)}"
    )

print("\n" + "=" * 100)
print("OVERLAP CHECK COMPLETE")
print("=" * 100)

V3 EVIDENCE → TARGET OVERLAP
01 | overall_similarity=0.53 | near_copied_sentences=56% | target_sentences=16
02 | overall_similarity=0.26 | near_copied_sentences=30% | target_sentences=10
03 | overall_similarity=0.13 | near_copied_sentences=38% | target_sentences=8
04 | overall_similarity=0.57 | near_copied_sentences=50% | target_sentences=12
05 | overall_similarity=0.20 | near_copied_sentences=50% | target_sentences=10
06 | overall_similarity=0.53 | near_copied_sentences=54% | target_sentences=13
07 | overall_similarity=0.83 | near_copied_sentences=68% | target_sentences=19
08 | overall_similarity=0.69 | near_copied_sentences=64% | target_sentences=14
09 | overall_similarity=0.46 | near_copied_sentences=50% | target_sentences=12
10 | overall_similarity=0.07 | near_copied_sentences=44% | target_sentences=9
11 | overall_similarity=0.06 | near_copied_sentences=33% | target_sentences=12
12 | overall_similarity=0.09 | near_copied_sentences=44% | target_sentences=9
13 | overall_similarity=0.

In [112]:
f001 = apex_v3_records[0]

reasoning_target = """FAULT IDENTIFIED:
F001 — DC Link Voltage Too High

INTERPRETATION:
The manual indicates that the DC link voltage is above its specified limit because regenerative energy from the backgauge is not being dissipated quickly enough. The fault code alone does not identify which probable cause is responsible.

REASONING:
The documented possibilities are a failed/open braking resistor or cable, an overly aggressive BG.DECEL setting, or mains voltage already above the permitted range. These must be distinguished by measurement rather than assumption.

DIAGNOSTIC ORDER:
1. Check mains voltage at the main isolator. The permitted range is 360–440 V AC.
2. If mains voltage is acceptable, apply LOTO and check the braking resistor between R+ and RB. The documented value is 47 Ω ±10%.
3. If the resistor is healthy, increase BG.DECEL by 25% as specified.
4. If the fault remains with normal mains and a healthy resistor, the manual directs service to test the DC link capacitors.

DECISION:
Do not name a failed component until the corresponding check provides evidence. Stop and escalate where the manual requires electrician or service involvement.

SAFETY:
Follow the documented isolation and LOTO requirements before performing electrical measurements or component checks."""

print("ORIGINAL TARGET:")
print(f001["target"])

print("\n" + "=" * 100)
print("REASONING-STYLE TARGET:")
print("=" * 100)
print(reasoning_target)

ORIGINAL TARGET:
FAULT:
F001 — DC Link Voltage Too High

DIAGNOSIS:
The manual identifies these probable causes:
1. The braking resistor on the servo amplifier has gone open-circuit or its connecting cable is
broken.
2. The backgauge deceleration ramp (parameter BG.DECEL in the drive menu) is set too
aggressively short.
3. Mains supply voltage is already at the high end of tolerance (>440 V AC) before
regeneration occurs.

DIAGNOSTIC SEQUENCE:
1. Measure the mains supply at the main isolator terminals — must be 400 V ±10% (360–440
V). If high, notify the facility electrician before proceeding.
2. Power down (LOTO) and measure the braking resistor resistance at the servo amplifier
terminals R+ and RB; the nominal value is 47 Ω ±10%. Replace if open or out of tolerance
(Part No. DW-BRK-RES).
3. If the resistor is healthy, open the servo drive parameter menu and increase BG.DECEL
from the current value by 25% (e.g., 0.2 s → 0.25 s) to reduce peak regenerative current.
4. If the fault pers

In [113]:
# Build reasoning-focused targets for all 18 APEX scenarios

reasoning_targets = [
"""FAULT IDENTIFIED:
F001 — DC Link Voltage Too High

INTERPRETATION:
The DC link voltage has exceeded its specified limit because regenerative energy from the backgauge is not being dissipated quickly enough. The fault code alone does not identify the failed component.

REASONING:
The documented possibilities are a failed/open braking resistor or cable, an overly aggressive BG.DECEL setting, or mains voltage already above the permitted range. These possibilities must be distinguished using the documented checks.

DIAGNOSTIC ORDER:
1. Check mains voltage at the main isolator. The permitted range is 360–440 V AC.
2. If acceptable, apply LOTO and measure the braking resistor between R+ and RB. The nominal value is 47 Ω ±10%.
3. If the resistor is healthy, increase BG.DECEL by 25%.
4. If the fault persists with normal mains and a healthy resistor, request a DC link capacitor capacitance test from DeltaWorks service.

DECISION:
Do not identify a specific failed component until the corresponding diagnostic evidence supports it.

SAFETY:
Follow the documented isolation, LOTO, and escalation requirements before performing the relevant checks.""",

"""FAULT DISTINCTION:
F001 — DC Link Voltage Too High
F003 — Mains Overvoltage

INTERPRETATION:
F001 concerns excessive DC-link voltage inside the servo amplifier, associated with regenerative energy from the backgauge. F003 concerns incoming mains voltage exceeding 440 V AC for more than 500 ms.

REASONING:
Although both conditions involve excessive voltage, they occur at different points in the electrical system and have different documented causes and diagnostic paths.

DIAGNOSTIC APPROACH:
Use the fault definition and measured machine condition to determine which path applies. For F001, investigate the DC-link/regenerative condition and its documented causes. For F003, measure all three mains phases and investigate the incoming supply.

DECISION:
Do not replace one fault definition with the other simply because both involve voltage. The observed evidence must support the selected diagnosis.""",

"""FAULT IDENTIFIED:
F001 — DC Link Voltage Too High

WHAT IS ESTABLISHED:
The fault code establishes that the documented F001 condition is present.

WHAT IS NOT ESTABLISHED:
The fault code alone does not prove whether the braking resistor, BG.DECEL setting, mains supply, or DC-link capacitors are responsible.

REASONING:
The manual provides several possible causes, so selecting one without diagnostic evidence would be an unsupported conclusion.

NEXT ACTION:
Perform the documented checks in sequence and use their results to narrow the possible cause.

DECISION:
Do not claim an exact failed component until the relevant diagnostic evidence confirms it.""",

"""FAULT IDENTIFIED:
F002 — DC Link Voltage Too Low

INTERPRETATION:
The DC link has fallen below 500 V DC, indicating that the mains supply is insufficient to maintain the regulated bus.

DOCUMENTED CAUSES:
Phase loss, main rectifier diode failure, or an undersized supply cable causing voltage drop under load.

DIAGNOSTIC ORDER:
1. Measure all three phase voltages at the drive input.
2. Check fuses F1, F2, and F3 in the drive input fuse holder.
3. If phases and fuses are healthy, request a rectifier diode test from a qualified engineer.
4. Verify that the supply cable meets the installation requirement of at least 16 mm² per phase.

DECISION:
Use the result of each check to determine which documented cause remains plausible. Do not identify a failed component before the evidence supports it.""",

"""FAULT DISAMBIGUATION:
F002 — DC Link Voltage Too Low
F099 — Power Supply Phase Failure

INTERPRETATION:
F002 describes a low DC-link condition below 500 V DC. F099 specifically indicates loss of one or more incoming supply phases or phase-sequence reversal detected by the three-phase monitoring relay.

REASONING:
Phase loss is a documented possible cause of F002, so the two faults can be related. However, they are not interchangeable fault definitions.

HOW TO DISTINGUISH:
Check the actual three-phase supply condition. For F002, the manual directs measurement at the drive input and additional checks of fuses, rectifier, and cable sizing. For F099, the manual directs checking the three incoming phases at the main isolator and then the main supply fuses and switching components.

DECISION:
Use the measured electrical condition and the specific fault definition to select the appropriate diagnostic path.""",

"""SAFETY-CRITICAL FAULT:
F020 — Light Curtain Fault

INTERPRETATION:
The safety light curtain controller has reported an internal device fault rather than a simple beam interruption.

REASONING:
The documented possibilities are damaged or misaligned emitter/receiver heads, contamination of the optical lenses, or an internal controller fault.

DIAGNOSTIC ORDER:
1. Clean the emitter and receiver lenses.
2. Check alignment; the receiver green LED should be solid green.
3. Power-cycle the light curtain through the documented safety relay supply breaker.
4. If the fault persists, use the deTec4 diagnostic LED information and replace the controller if indicated.

SAFETY DECISION:
The safety condition must not be bypassed, overridden, short-circuited, or otherwise defeated. Production should not resume merely to avoid the fault.""",

"""SAFETY-CRITICAL FAULT:
F030 — Door Safety Switch Open

INTERPRETATION:
The rear guard door is detected as open or the safety switch circuit is open. This prevents a ram cycle because the backgauge-accessible area presents a crushing hazard. The manual defines this as a hard safety stop.

REASONING:
The possible causes are an unlatched door, a missing or damaged Schmersal AZ 16 actuator key, or damaged/loose wiring.

DIAGNOSTIC ORDER:
1. Confirm the rear guard door is fully closed and latched.
2. Check the Rear Door safety input on HMI Diagnostics; it should indicate the closed state.
3. Inspect the AZ 16 actuator key and replace it if damaged.
4. If the door and actuator are intact, apply LOTO and check cable continuity to safety relay K2.

SAFETY DECISION:
Do not short-circuit, tape, bypass, or override the safety switch. The documented hard safety stop must remain effective during diagnosis.""",

"""FAULT IDENTIFIED:
F011 — Temperature Sensor Short Circuit

INTERPRETATION:
The machine has detected a PT100 resistance below the documented short-circuit threshold. This establishes an electrical fault, but not which physical component caused it.

REASONING:
The possible causes are crushed or pinched wiring, fluid contamination in the sensor terminal head, or an internally shorted PT100 sensor.

EVIDENCE NEEDED:
1. Identify the affected channel through HMI Diagnostics.
2. Apply LOTO, disconnect the sensor cable at the control cabinet, and measure resistance.
3. A healthy PT100 is approximately 100 Ω at 0°C and 109 Ω at 25°C; below 10 Ω confirms a short.
4. If the cable is shorted, repair or replace the cable. If the cable is correct but the fault persists when reconnected, the sensor itself is indicated as failed.

DECISION:
The root cause should be selected from the measurement results rather than inferred from F011 alone.""",

"""FAULT IDENTIFIED:
F015 — Hydraulic Oil Temperature High

INTERPRETATION:
The hydraulic oil has exceeded 60°C, restricting machine operation to slow speed.

REASONING:
The documented causes are an oil cooler problem, insufficient cooling-water flow, or continuous relief-valve bypassing.

DIAGNOSTIC ORDER:
1. Check the oil cooler for blockage or verify cooling-water flow where applicable.
2. Monitor the temperature trend; if it rises rapidly, stop the HPU.
3. Listen for continuous relief-valve bypass noise and check the pressure setpoint if present.
4. Allow the oil to cool below 50°C before full-speed operation.

DECISION:
Do not select a single root cause until the corresponding diagnostic evidence supports it.""",

"""PROCEDURE:
F040 — Backgauge Home Position Loss

INTERPRETATION:
The backgauge reference position has been invalidated and must be re-established through the documented homing procedure.

PROCEDURE:
1. Press HOME ALL AXES on the HMI.
2. Confirm visually that the backgauge reaches the hard stop and retreats to the home-position marker.
3. If homing fails, inspect the X-axis reference-mark sensor for contamination.
4. Re-enter the backgauge calibration offsets from the machine data sheet if required.

DECISION:
Follow the sequence and use the observed homing result to determine the next step. Do not substitute an undocumented recovery procedure or bypass required safety controls.""",

"""DATA-INTEGRITY FAULT:
F045 — Tool Table Data Corrupt

INTERPRETATION:
The HMI tool and die table has failed its checksum check, indicating that the stored data cannot be assumed valid.

REASONING:
The documented causes are power loss during saving or an HMI storage-media fault. The fault code does not establish which cause occurred.

RECOVERY:
1. Restore the tool table from the most recent backup.
2. If no backup exists, re-enter dimensions from the physical tool documentation.
3. Create a new backup immediately after restoration.
4. Contact DeltaWorks support if restores repeatedly fail because the SSD may require replacement.

DECISION:
Only use values supported by the available documentation. Do not invent or assume missing tool data.""",

"""PROCEDURE:
F050 — Bend Angle Sensor Fault

INTERPRETATION:
The laser-based angle measurement system has returned an out-of-range or invalid reading.

REASONING:
The documented possibilities include a contaminated sensor window, unsuitable sheet reflectivity, or a damaged sensor cable.

PROCEDURE:
1. Clean the sensor window.
2. Ensure the sheet surface is free of oil in the measurement zone.
3. If the material is very shiny or dark, adjust sensor sensitivity through the documented HMI setting.
4. If the sensor cannot be corrected before production, disable in-process measurement and use manual angle gauging as specified.

DECISION:
Follow the documented correction path and do not invent an alternative sensor adjustment.""",

"""ROOT-CAUSE ANALYSIS:
F055 — HPU Pressure Relief Valve Open

INTERPRETATION:
The hydraulic pressure has remained at the relief-valve setpoint of 210 bar for more than five seconds, indicating excessive bypass.

DOCUMENTED POSSIBILITIES:
1. Material is harder or thicker than the job specification.
2. Relief-valve setpoint has drifted low.
3. Proportional valve spool is sticking.

DIAGNOSTIC PATH:
1. Verify material grade and thickness against the HMI job specification.
2. Check the relief-valve setpoint at the hydraulic test point and adjust to 210 bar if drifted.
3. Inspect the proportional valve for contamination and clean it if stiction is present.

DECISION:
Do not declare a single root cause until the corresponding check confirms it.""",

"""CONSTRAINT CHECK:
F060 — Axis R (Ram Height) Out of Range

INTERPRETATION:
The R-axis has been commanded outside its documented 0–300 mm travel range.

REASONING:
The manual identifies an incorrect R-axis value for the installed die height or a die table that was not updated after a die change.

CORRECTION:
1. Enter the correct installed die height in HMI Tool Table > Bottom Die Height.
2. Recalculate the job sequence.
3. Confirm that the resulting R-axis target is within 0–300 mm.

DECISION:
Treat the documented travel range as a hard constraint. Do not guess an alternative value outside the specified range.""",

"""COMMUNICATION FAULT:
F070 — Network Communication Fault

INTERPRETATION:
The DX-200 has lost factory-network communication for more than ten seconds.

REASONING:
The documented possibilities are a damaged/disconnected network cable, a factory network-switch failure, or an IP configuration error after a firmware update.

DIAGNOSTIC ORDER:
1. Check the Ethernet cable between the control cabinet and network switch.
2. Ping the machine IP from a factory PC.
3. Compare the HMI network settings against the site network map.

DECISION:
Use the result of each check to distinguish the possible communication causes. Do not assume a failed cable, switch, or configuration without evidence.""",

"""SAFETY-CRITICAL FAULT:
F080 — Safety Relay Fault

INTERPRETATION:
The dual-channel Pilz PNOZ safety relay has detected an inconsistency between its monitoring channels.

REASONING:
The documented possibilities are a broken E-Stop/guard-switch input wire, a failed safety relay, or an incorrect reset sequence after an E-Stop.

DIAGNOSTIC ORDER:
1. Ensure all E-Stops are released and perform the documented manual reset.
2. Check the PNOZ LED diagnostics using the supplied Pilz documentation.
3. Measure continuity of the E-Stop and guard-switch wiring at the safety-relay inputs.
4. If all inputs are healthy and the fault remains, replace the safety relay as documented.

SAFETY DECISION:
Do not bypass or override the safety control merely to restore production. Resume production only after the documented safety condition has been resolved.""",

"""DATA-INTEGRITY FAULT:
F090 — Job Program CRC Error

INTERPRETATION:
The loaded job program has failed its CRC check, so the file may be corrupted or incompatible.

REASONING:
The documented possibilities are an incomplete/failed USB transfer or a file edited externally with incompatible encoding.

RECOVERY:
1. Reload the job from the USB drive or network share.
2. Verify the file on a PC using the DeltaWorks offline editor.
3. If the error persists, recreate the job on the machine HMI.

DECISION:
Treat the affected program as unverified until the documented checks establish that it is valid. Do not invent replacement program values.""",

"""SAFETY-CRITICAL FAULT:
F099 — Power Supply Phase Failure

INTERPRETATION:
The three-phase monitoring relay has detected loss of one or more incoming phases or phase-sequence reversal. The relay latches the fault because single-phase operation can rapidly overheat the 22 kW HPU motor.

DOCUMENTED POSSIBILITIES:
1. A blown main supply fuse.
2. Utility loss of a phase.
3. A faulty contact in the main isolator or K-MAIN contactor.

DIAGNOSTIC ORDER:
1. Measure all three incoming phases at the main isolator; each should be 230 V line-to-neutral or 400 V line-to-line.
2. Inspect F1, F2, and F3 and replace blown fuses only with the identically rated DW-FUSE-63A gG 63 A fuse.
3. If input phases are present but one is absent at the output, inspect the main isolator and K-MAIN contactor and contact a qualified electrician.
4. After the supply is confirmed healthy, manually reset the Finmotor relay.

SAFETY DECISION:
Do not reset the relay until all three phases are confirmed healthy. Do not use a higher-rated fuse. Production should resume only after the documented supply condition has been resolved."""
]

print("Reasoning targets created:", len(reasoning_targets))

for i, target in enumerate(reasoning_targets, 1):
    print(f"Scenario {i:02d}: {len(target)} chars")

Reasoning targets created: 18
Scenario 01: 1150 chars
Scenario 02: 904 chars
Scenario 03: 656 chars
Scenario 04: 800 chars
Scenario 05: 913 chars
Scenario 06: 831 chars
Scenario 07: 906 chars
Scenario 08: 937 chars
Scenario 09: 720 chars
Scenario 10: 685 chars
Scenario 11: 747 chars
Scenario 12: 728 chars
Scenario 13: 746 chars
Scenario 14: 616 chars
Scenario 15: 686 chars
Scenario 16: 846 chars
Scenario 17: 640 chars
Scenario 18: 1107 chars


In [114]:
from difflib import SequenceMatcher
import re

print("=" * 100)
print("APEX V3 REASONING TARGET OVERLAP")
print("=" * 100)

for i, r in enumerate(apex_v3_records):
    evidence_match = re.search(
        r"Evidence from the maintenance manual:\s*(.*?)(?:\n\nTARGET:|\Z)",
        r["user"],
        flags=re.DOTALL | re.IGNORECASE
    )

    evidence = evidence_match.group(1).strip() if evidence_match else ""
    target = reasoning_targets[i]

    def normalize(text):
        text = text.lower()
        text = re.sub(r"\s+", " ", text)
        return text.strip()

    e_norm = normalize(evidence)
    t_norm = normalize(target)

    similarity = SequenceMatcher(None, e_norm, t_norm).ratio()

    evidence_sentences = [
        s.strip() for s in re.split(r"(?<=[.!?])\s+", evidence)
        if len(s.strip()) > 30
    ]

    target_sentences = [
        s.strip() for s in re.split(r"(?<=[.!?])\s+", target)
        if len(s.strip()) > 30
    ]

    copied = 0

    for ts in target_sentences:
        best = max(
            [
                SequenceMatcher(
                    None,
                    normalize(ts),
                    normalize(es)
                ).ratio()
                for es in evidence_sentences
            ],
            default=0
        )

        if best >= 0.85:
            copied += 1

    copy_ratio = copied / max(len(target_sentences), 1)

    print(
        f"{i+1:02d} | "
        f"similarity={similarity:.2f} | "
        f"near_copied={copy_ratio:.0%} | "
        f"target_sentences={len(target_sentences)}"
    )

print("\n" + "=" * 100)
print("REASONING OVERLAP CHECK COMPLETE")
print("=" * 100)

APEX V3 REASONING TARGET OVERLAP
01 | similarity=0.13 | near_copied=0% | target_sentences=12
02 | similarity=0.06 | near_copied=0% | target_sentences=8
03 | similarity=0.06 | near_copied=0% | target_sentences=5
04 | similarity=0.43 | near_copied=38% | target_sentences=8
05 | similarity=0.03 | near_copied=0% | target_sentences=8
06 | similarity=0.05 | near_copied=0% | target_sentences=8
07 | similarity=0.10 | near_copied=0% | target_sentences=10
08 | similarity=0.07 | near_copied=0% | target_sentences=9
09 | similarity=0.42 | near_copied=29% | target_sentences=7
10 | similarity=0.32 | near_copied=14% | target_sentences=7
11 | similarity=0.14 | near_copied=22% | target_sentences=9
12 | similarity=0.11 | near_copied=33% | target_sentences=6
13 | similarity=0.33 | near_copied=38% | target_sentences=8
14 | similarity=0.20 | near_copied=17% | target_sentences=6
15 | similarity=0.13 | near_copied=14% | target_sentences=7
16 | similarity=0.14 | near_copied=12% | target_sentences=8
17 | similar

In [115]:
from datasets import Dataset

v3_records = []

for i, r in enumerate(apex_v3_records):
    v3_records.append({
        "scenario_id": i + 1,
        "user": r["user"],
        "target": reasoning_targets[i],
    })

v3_raw_dataset = Dataset.from_list(v3_records)

print("=" * 100)
print("APEX V3 DATASET")
print("=" * 100)
print("Rows:", len(v3_raw_dataset))
print("Columns:", v3_raw_dataset.column_names)

print("\nTarget lengths:")
for i, target in enumerate(reasoning_targets, 1):
    print(f"{i:02d}: {len(target)} chars")

print("\nTotal target characters:", sum(len(x) for x in reasoning_targets))
print("Average target characters:", round(
    sum(len(x) for x in reasoning_targets) / len(reasoning_targets), 1
))

print("\n" + "=" * 100)
print("V3 DATASET READY")
print("=" * 100)

APEX V3 DATASET
Rows: 18
Columns: ['scenario_id', 'user', 'target']

Target lengths:
01: 1150 chars
02: 904 chars
03: 656 chars
04: 800 chars
05: 913 chars
06: 831 chars
07: 906 chars
08: 937 chars
09: 720 chars
10: 685 chars
11: 747 chars
12: 728 chars
13: 746 chars
14: 616 chars
15: 686 chars
16: 846 chars
17: 640 chars
18: 1107 chars

Total target characters: 14618
Average target characters: 812.1

V3 DATASET READY


In [116]:
from datasets import Dataset

def format_apex_v3(example):
    messages = [
        {
            "role": "system",
            "content": (
                "You are APEX, an industrial troubleshooting reasoning model. "
                "Use the provided maintenance evidence as the source of truth. "
                "Reason from evidence, distinguish documented facts from hypotheses, "
                "compare plausible causes when necessary, follow documented diagnostic "
                "order, respect safety requirements, and do not invent machine facts."
            )
        },
        {
            "role": "user",
            "content": example["user"]
        },
        {
            "role": "assistant",
            "content": example["target"]
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False
    )

    return {"text": text}


# Use the tokenizer already loaded with Qwen3-4B
v3_formatted_dataset = v3_raw_dataset.map(
    format_apex_v3,
    remove_columns=["scenario_id", "user", "target"]
)

print("=" * 100)
print("APEX V3 CHAT-TEMPLATE DATASET")
print("=" * 100)
print("Rows:", len(v3_formatted_dataset))
print("Columns:", v3_formatted_dataset.column_names)

print("\nFirst formatted example:")
print("-" * 100)
print(v3_formatted_dataset[0]["text"][:4000])
print("-" * 100)

print("\nFormatting complete.")

Map:   0%|          | 0/18 [00:00<?, ? examples/s]

APEX V3 CHAT-TEMPLATE DATASET
Rows: 18
Columns: ['text']

First formatted example:
----------------------------------------------------------------------------------------------------
<|im_start|>system
You are APEX, an industrial troubleshooting reasoning model. Use the provided maintenance evidence as the source of truth. Reason from evidence, distinguish documented facts from hypotheses, compare plausible causes when necessary, follow documented diagnostic order, respect safety requirements, and do not invent machine facts.<|im_end|>
<|im_start|>user
Machine:
DeltaWorks DX-200 Automated Press Brake

Observed fault:
F001 — DC Link Voltage Too High

Question:
What should be checked first, and what is the prescribed troubleshooting sequence?

Evidence from the maintenance manual:
F001 — DC Link Voltage Too High 
Description: The DC link bus inside the servo amplifier powering the backgauge axes has risen 
above 780 V DC. This is distinct from mains overvoltage (see F003). The condition

In [117]:
MAX_LENGTH = 2048

def tokenize_apex_v3(example):
    text = example["text"]

    # Tokenize the complete conversation
    encoded = tokenizer(
        text,
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False,
    )

    input_ids = encoded["input_ids"]
    attention_mask = encoded["attention_mask"]

    # Locate the assistant boundary
    assistant_marker = "<|im_start|>assistant\n"
    marker_ids = tokenizer(
        assistant_marker,
        add_special_tokens=False
    )["input_ids"]

    # Find assistant marker inside the tokenized sequence
    assistant_start = None

    for i in range(len(input_ids) - len(marker_ids) + 1):
        if input_ids[i:i + len(marker_ids)] == marker_ids:
            assistant_start = i + len(marker_ids)
            break

    if assistant_start is None:
        raise ValueError("Assistant marker not found in tokenized example.")

    # Mask everything before the assistant response
    labels = [-100] * assistant_start + input_ids[assistant_start:]

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels,
    }


v3_tokenized_dataset = v3_formatted_dataset.map(
    tokenize_apex_v3,
    remove_columns=["text"]
)

print("=" * 100)
print("APEX V3 TOKENIZATION")
print("=" * 100)
print("Rows:", len(v3_tokenized_dataset))
print("Columns:", v3_tokenized_dataset.column_names)

print("\nToken statistics:")
lengths = []

for i, row in enumerate(v3_tokenized_dataset):
    total = len(row["input_ids"])
    trainable = sum(x != -100 for x in row["labels"])
    ignored = sum(x == -100 for x in row["labels"])

    lengths.append(total)

    print(
        f"{i+1:02d} | "
        f"total={total:4d} | "
        f"ignored={ignored:4d} | "
        f"trainable={trainable:4d}"
    )

print("\n" + "-" * 100)
print("Min total tokens:", min(lengths))
print("Max total tokens:", max(lengths))
print("Average total tokens:", round(sum(lengths) / len(lengths), 1))
print("MAX_LENGTH:", MAX_LENGTH)

print("\nTokenization complete.")

Map:   0%|          | 0/18 [00:00<?, ? examples/s]

APEX V3 TOKENIZATION
Rows: 18
Columns: ['input_ids', 'attention_mask', 'labels']

Token statistics:
01 | total= 789 | ignored= 533 | trainable= 256
02 | total= 885 | ignored= 685 | trainable= 200
03 | total= 669 | ignored= 532 | trainable= 137
04 | total= 497 | ignored= 304 | trainable= 193
05 | total=1526 | ignored=1315 | trainable= 211
06 | total= 505 | ignored= 324 | trainable= 181
07 | total= 856 | ignored= 630 | trainable= 226
08 | total= 808 | ignored= 579 | trainable= 229
09 | total= 469 | ignored= 306 | trainable= 163
10 | total= 467 | ignored= 316 | trainable= 151
11 | total= 444 | ignored= 283 | trainable= 161
12 | total= 446 | ignored= 293 | trainable= 153
13 | total= 465 | ignored= 286 | trainable= 179
14 | total= 414 | ignored= 261 | trainable= 153
15 | total= 405 | ignored= 256 | trainable= 149
16 | total= 518 | ignored= 326 | trainable= 192
17 | total= 405 | ignored= 260 | trainable= 145
18 | total=1388 | ignored=1107 | trainable= 281

-----------------------------------

In [118]:
print("=" * 100)
print("APEX V3 ASSISTANT-ONLY LABEL CHECK")
print("=" * 100)

for idx in [0, 1, 17]:
    row = v3_tokenized_dataset[idx]

    input_ids = row["input_ids"]
    labels = row["labels"]

    # Recover only trainable tokens
    trainable_ids = [
        token_id
        for token_id, label in zip(input_ids, labels)
        if label != -100
    ]

    trainable_text = tokenizer.decode(
        trainable_ids,
        skip_special_tokens=False
    )

    ignored_count = sum(label == -100 for label in labels)
    trainable_count = sum(label != -100 for label in labels)

    print(f"\n{'=' * 100}")
    print(f"EXAMPLE {idx + 1}")
    print(f"{'=' * 100}")
    print(f"Total tokens:      {len(input_ids)}")
    print(f"Ignored tokens:    {ignored_count}")
    print(f"Trainable tokens:  {trainable_count}")

    print("\nTRAINABLE CONTENT:")
    print("-" * 100)
    print(trainable_text)
    print("-" * 100)

print("\n" + "=" * 100)
print("MASKING CHECK COMPLETE")
print("=" * 100)

APEX V3 ASSISTANT-ONLY LABEL CHECK

EXAMPLE 1
Total tokens:      789
Ignored tokens:    533
Trainable tokens:  256

TRAINABLE CONTENT:
----------------------------------------------------------------------------------------------------
<think>

</think>

FAULT IDENTIFIED:
F001 — DC Link Voltage Too High

INTERPRETATION:
The DC link voltage has exceeded its specified limit because regenerative energy from the backgauge is not being dissipated quickly enough. The fault code alone does not identify the failed component.

REASONING:
The documented possibilities are a failed/open braking resistor or cable, an overly aggressive BG.DECEL setting, or mains voltage already above the permitted range. These possibilities must be distinguished using the documented checks.

DIAGNOSTIC ORDER:
1. Check mains voltage at the main isolator. The permitted range is 360–440 V AC.
2. If acceptable, apply LOTO and measure the braking resistor between R+ and RB. The nominal value is 47 Ω ±10%.
3. If the resis

In [119]:
from datasets import Dataset

# Keep only the fields the Trainer should receive
apex_v3_train_dataset = v3_tokenized_dataset.remove_columns(
    ["scenario_id", "user"]
) if "scenario_id" in v3_tokenized_dataset.column_names else v3_tokenized_dataset

print("=" * 100)
print("APEX V3 TRAINING DATASET")
print("=" * 100)

print("Rows:", len(apex_v3_train_dataset))
print("Columns:", apex_v3_train_dataset.column_names)

# Verify every row has matching lengths
valid = True

for i, row in enumerate(apex_v3_train_dataset):
    input_len = len(row["input_ids"])
    mask_len = len(row["attention_mask"])
    label_len = len(row["labels"])

    if not (input_len == mask_len == label_len):
        valid = False
        print(
            f"ERROR row {i}: "
            f"input={input_len}, "
            f"attention={mask_len}, "
            f"labels={label_len}"
        )

print("\nLength consistency:", "PASS" if valid else "FAIL")

# Verify every example has trainable tokens
trainable_counts = [
    sum(label != -100 for label in row["labels"])
    for row in apex_v3_train_dataset
]

print(
    "Trainable-token check:",
    "PASS" if all(x > 0 for x in trainable_counts) else "FAIL"
)

print("\nTrainable tokens:")
print("Min:", min(trainable_counts))
print("Max:", max(trainable_counts))
print("Average:", round(sum(trainable_counts) / len(trainable_counts), 1))

print("\n" + "=" * 100)
print("READY FOR TRAINER")
print("=" * 100)

APEX V3 TRAINING DATASET
Rows: 18
Columns: ['input_ids', 'attention_mask', 'labels']

Length consistency: PASS
Trainable-token check: PASS

Trainable tokens:
Min: 137
Max: 281
Average: 186.7

READY FOR TRAINER


In [120]:
from transformers import TrainingArguments, DataCollatorForSeq2Seq, Trainer

APEX_V3_OUTPUT_DIR = "./apex_model_v3"

apex_v3_training_args = TrainingArguments(
    output_dir=APEX_V3_OUTPUT_DIR,

    # T4-friendly
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,

    # V3 learning setup
    learning_rate=5e-5,
    num_train_epochs=3,

    # Memory optimization
    gradient_checkpointing=True,
    fp16=True,

    # 8-bit optimizer
    optim="paged_adamw_8bit",

    # Logging
    logging_steps=1,
    save_strategy="no",
    report_to="none",

    # Reproducibility
    seed=42,

    # Important for our custom labels
    remove_unused_columns=False,
)

apex_v3_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=apex_v3_model,
    padding=True,
    label_pad_token_id=-100,
    return_tensors="pt",
)

print("=" * 100)
print("APEX V3 TRAINER CONFIG")
print("=" * 100)

print("Output directory:", APEX_V3_OUTPUT_DIR)
print("Batch size:", apex_v3_training_args.per_device_train_batch_size)
print("Gradient accumulation:", apex_v3_training_args.gradient_accumulation_steps)
print("Effective batch size:",
      apex_v3_training_args.per_device_train_batch_size *
      apex_v3_training_args.gradient_accumulation_steps)
print("Learning rate:", apex_v3_training_args.learning_rate)
print("Epochs:", apex_v3_training_args.num_train_epochs)
print("FP16:", apex_v3_training_args.fp16)
print("Gradient checkpointing:", apex_v3_training_args.gradient_checkpointing)
print("Optimizer:", apex_v3_training_args.optim)

print("\nCreating Trainer...")

apex_v3_trainer = Trainer(
    model=apex_v3_model,
    args=apex_v3_training_args,
    train_dataset=apex_v3_train_dataset,
    data_collator=apex_v3_collator,
)

print("Trainer created:", type(apex_v3_trainer).__name__)

print("\n" + "=" * 100)
print("TRAINER READY — TRAINING NOT STARTED")
print("=" * 100)

APEX V3 TRAINER CONFIG
Output directory: ./apex_model_v3
Batch size: 1
Gradient accumulation: 4
Effective batch size: 4
Learning rate: 5e-05
Epochs: 3
FP16: True
Gradient checkpointing: True
Optimizer: OptimizerNames.PAGED_ADAMW_8BIT

Creating Trainer...
Trainer created: Trainer

TRAINER READY — TRAINING NOT STARTED


In [121]:
import torch

print("=" * 100)
print("APEX V3 PRE-TRAINING SANITY CHECK")
print("=" * 100)

# Build exactly one batch using the Trainer's data collator
sample_batch = apex_v3_collator(
    [apex_v3_train_dataset[0]]
)

# Move tensors to the model device
model_device = next(apex_v3_model.parameters()).device

sample_batch = {
    k: v.to(model_device) if torch.is_tensor(v) else v
    for k, v in sample_batch.items()
}

print("Model device:", model_device)
print("Input shape:", tuple(sample_batch["input_ids"].shape))
print("Attention shape:", tuple(sample_batch["attention_mask"].shape))
print("Labels shape:", tuple(sample_batch["labels"].shape))

# Verify labels contain trainable tokens
trainable = (sample_batch["labels"] != -100).sum().item()
ignored = (sample_batch["labels"] == -100).sum().item()

print("Ignored label tokens:", ignored)
print("Trainable label tokens:", trainable)

# Forward pass only — NO optimizer step
apex_v3_model.eval()

with torch.no_grad():
    outputs = apex_v3_model(
        input_ids=sample_batch["input_ids"],
        attention_mask=sample_batch["attention_mask"],
        labels=sample_batch["labels"],
    )

loss = outputs.loss

print("\nForward pass: PASS")
print("Initial V3 loss:", float(loss))

print("\n" + "=" * 100)
print("PRE-TRAINING SANITY CHECK COMPLETE")
print("=" * 100)

APEX V3 PRE-TRAINING SANITY CHECK
Model device: cuda:0
Input shape: (1, 789)
Attention shape: (1, 789)
Labels shape: (1, 789)
Ignored label tokens: 533
Trainable label tokens: 256

Forward pass: PASS
Initial V3 loss: 2.729323148727417

PRE-TRAINING SANITY CHECK COMPLETE


In [124]:
import gc
import torch

print("=" * 100)
print("APEX V3 MEMORY RESET")
print("=" * 100)

# Stop the failed Trainer references
del apex_v3_trainer
gc.collect()

torch.cuda.empty_cache()
torch.cuda.ipc_collect()

print("Before configuration:")
print("Allocated:", round(torch.cuda.memory_allocated() / 1024**3, 2), "GB")
print("Reserved: ", round(torch.cuda.memory_reserved() / 1024**3, 2), "GB")
print("Free:     ", round(
    (torch.cuda.get_device_properties(0).total_memory -
     torch.cuda.memory_allocated()) / 1024**3, 2
), "GB")

# Disable KV cache during training.
# This is important when using gradient checkpointing.
apex_v3_model.config.use_cache = False

# Ensure gradient checkpointing is enabled on the model itself.
apex_v3_model.gradient_checkpointing_enable()

print("\nTraining memory settings:")
print("use_cache:", apex_v3_model.config.use_cache)
print("gradient_checkpointing:", True)

# Recreate the Trainer with a slightly smaller micro-batch footprint.
apex_v3_training_args = TrainingArguments(
    output_dir=APEX_V3_OUTPUT_DIR,

    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,

    learning_rate=5e-5,
    num_train_epochs=3,

    gradient_checkpointing=True,
    fp16=True,

    optim="paged_adamw_8bit",

    logging_steps=1,
    save_strategy="no",
    report_to="none",

    seed=42,
    remove_unused_columns=False,

    # Reduce memory retained by Trainer
    prediction_loss_only=True,
)

apex_v3_trainer = Trainer(
    model=apex_v3_model,
    args=apex_v3_training_args,
    train_dataset=apex_v3_train_dataset,
    data_collator=apex_v3_collator,
)

print("\nTrainer recreated successfully.")

print("\nAfter configuration:")
print("Allocated:", round(torch.cuda.memory_allocated() / 1024**3, 2), "GB")
print("Reserved: ", round(torch.cuda.memory_reserved() / 1024**3, 2), "GB")

print("\n" + "=" * 100)
print("APEX V3 MEMORY CONFIG READY")
print("=" * 100)

APEX V3 MEMORY RESET
Before configuration:
Allocated: 11.45 GB
Reserved:  13.23 GB
Free:      3.12 GB

Training memory settings:
use_cache: False
gradient_checkpointing: True

Trainer recreated successfully.

After configuration:
Allocated: 11.45 GB
Reserved:  13.23 GB

APEX V3 MEMORY CONFIG READY


In [125]:
MAX_LENGTH = 1536

def tokenize_apex_v3_1536(example):
    text = example["text"]

    encoded = tokenizer(
        text,
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False,
    )

    input_ids = encoded["input_ids"]
    attention_mask = encoded["attention_mask"]

    assistant_marker = "<|im_start|>assistant\n"
    marker_ids = tokenizer(
        assistant_marker,
        add_special_tokens=False
    )["input_ids"]

    assistant_start = None

    for i in range(len(input_ids) - len(marker_ids) + 1):
        if input_ids[i:i + len(marker_ids)] == marker_ids:
            assistant_start = i + len(marker_ids)
            break

    if assistant_start is None:
        raise ValueError("Assistant marker not found.")

    labels = [-100] * assistant_start + input_ids[assistant_start:]

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels,
    }


v3_tokenized_1536 = v3_formatted_dataset.map(
    tokenize_apex_v3_1536,
    remove_columns=["text"]
)

print("=" * 100)
print("APEX V3 — 1536 TOKEN CHECK")
print("=" * 100)

for i, row in enumerate(v3_tokenized_1536):
    total = len(row["input_ids"])
    trainable = sum(x != -100 for x in row["labels"])

    print(
        f"{i+1:02d} | "
        f"total={total:4d} | "
        f"trainable={trainable:3d}"
    )

print("\nMaximum sequence:", max(
    len(row["input_ids"]) for row in v3_tokenized_1536
))

print("All examples fit:",
      all(len(row["input_ids"]) <= MAX_LENGTH for row in v3_tokenized_1536))

print("\n" + "=" * 100)
print("1536-TOKEN DATASET READY")
print("=" * 100)

Map:   0%|          | 0/18 [00:00<?, ? examples/s]

APEX V3 — 1536 TOKEN CHECK
01 | total= 789 | trainable=256
02 | total= 885 | trainable=200
03 | total= 669 | trainable=137
04 | total= 497 | trainable=193
05 | total=1526 | trainable=211
06 | total= 505 | trainable=181
07 | total= 856 | trainable=226
08 | total= 808 | trainable=229
09 | total= 469 | trainable=163
10 | total= 467 | trainable=151
11 | total= 444 | trainable=161
12 | total= 446 | trainable=153
13 | total= 465 | trainable=179
14 | total= 414 | trainable=153
15 | total= 405 | trainable=149
16 | total= 518 | trainable=192
17 | total= 405 | trainable=145
18 | total=1388 | trainable=281

Maximum sequence: 1526
All examples fit: True

1536-TOKEN DATASET READY


In [126]:
# Use the 1536-token dataset for training
apex_v3_train_dataset = v3_tokenized_1536

# Recreate the collator
apex_v3_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=apex_v3_model,
    padding=True,
    label_pad_token_id=-100,
    return_tensors="pt",
)

# Recreate Trainer
apex_v3_trainer = Trainer(
    model=apex_v3_model,
    args=apex_v3_training_args,
    train_dataset=apex_v3_train_dataset,
    data_collator=apex_v3_collator,
)

print("=" * 100)
print("APEX V3 TRAINER UPDATED")
print("=" * 100)

print("Dataset rows:", len(apex_v3_train_dataset))
print("Maximum sequence:",
      max(len(x["input_ids"]) for x in apex_v3_train_dataset))
print("Learning rate:", apex_v3_training_args.learning_rate)
print("Epochs:", apex_v3_training_args.num_train_epochs)
print("Effective batch size:",
      apex_v3_training_args.per_device_train_batch_size *
      apex_v3_training_args.gradient_accumulation_steps)

print("\nTrainer dataset successfully switched to 1536-token version.")

print("\n" + "=" * 100)
print("READY FOR TRAINING")
print("=" * 100)

APEX V3 TRAINER UPDATED
Dataset rows: 18
Maximum sequence: 1526
Learning rate: 5e-05
Epochs: 3
Effective batch size: 4

Trainer dataset successfully switched to 1536-token version.

READY FOR TRAINING


In [127]:
print("=" * 100)
print("STARTING APEX V3 TRAINING — RETRY")
print("=" * 100)

apex_v3_model.train()

v3_train_result = apex_v3_trainer.train()

print("\n" + "=" * 100)
print("APEX V3 TRAINING COMPLETE")
print("=" * 100)

print("Final training loss:", v3_train_result.training_loss)
print("Training steps:", v3_train_result.global_step)
print("Training runtime:",
      v3_train_result.metrics.get("train_runtime"))


STARTING APEX V3 TRAINING — RETRY


OutOfMemoryError: CUDA out of memory. Tried to allocate 498.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 1.81 MiB is free. Including non-PyTorch memory, this process has 14.34 GiB memory in use. Of the allocated memory 13.44 GiB is allocated by PyTorch, and 785.19 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)

In [128]:
import torch
from transformers import Trainer

class APEXResponseOnlyTrainer(Trainer):

    def compute_loss(
        self,
        model,
        inputs,
        return_outputs=False,
        num_items_in_batch=None,
    ):
        labels = inputs["labels"]

        # The assistant response is at the end of every example.
        # Find the first trainable label position.
        trainable_positions = (labels != -100).nonzero(as_tuple=False)

        if trainable_positions.numel() == 0:
            raise ValueError("No trainable assistant tokens found.")

        # For batch size 1, find where the assistant response begins.
        response_start = trainable_positions[:, 1].min().item()

        # Number of tokens needed for the response.
        response_length = labels.shape[1] - response_start

        # Ask Qwen3 to return logits only for the response region.
        outputs = model(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            logits_to_keep=response_length,
        )

        logits = outputs.logits

        # Keep labels aligned with the returned final logits.
        response_labels = labels[:, -response_length:]

        # Standard causal language-model loss.
        shift_logits = logits[:, :-1, :].contiguous()
        shift_labels = response_labels[:, 1:].contiguous()

        loss = torch.nn.functional.cross_entropy(
            shift_logits.view(-1, shift_logits.size(-1)),
            shift_labels.view(-1),
            ignore_index=-100,
        )

        return (loss, outputs) if return_outputs else loss


# Recreate Trainer using the memory-efficient loss
apex_v3_trainer = APEXResponseOnlyTrainer(
    model=apex_v3_model,
    args=apex_v3_training_args,
    train_dataset=apex_v3_train_dataset,
    data_collator=apex_v3_collator,
)

print("=" * 100)
print("APEX V3 RESPONSE-ONLY TRAINER")
print("=" * 100)

print("Full context is still provided to the model.")
print("Logits are calculated only for the assistant response.")
print("Evidence tokens remain masked from the loss.")
print("\nTrainer:", type(apex_v3_trainer).__name__)
print("Dataset rows:", len(apex_v3_train_dataset))

print("\n" + "=" * 100)
print("MEMORY-EFFICIENT TRAINER READY")
print("=" * 100)


APEX V3 RESPONSE-ONLY TRAINER
Full context is still provided to the model.
Logits are calculated only for the assistant response.
Evidence tokens remain masked from the loss.

Trainer: APEXResponseOnlyTrainer
Dataset rows: 18

MEMORY-EFFICIENT TRAINER READY


In [129]:
import torch

print("=" * 100)
print("APEX V3 CUSTOM LOSS SANITY CHECK")
print("=" * 100)

# Get one collated training example
test_batch = apex_v3_collator(
    [apex_v3_train_dataset[0]]
)

model_device = next(apex_v3_model.parameters()).device

test_batch = {
    k: v.to(model_device) if torch.is_tensor(v) else v
    for k, v in test_batch.items()
}

apex_v3_model.eval()

# Run the exact custom loss calculation
with torch.no_grad():
    test_loss = apex_v3_trainer.compute_loss(
        apex_v3_model,
        test_batch
    )

print("Input shape:", tuple(test_batch["input_ids"].shape))
print("Label shape:", tuple(test_batch["labels"].shape))

print("Trainable tokens:",
      int((test_batch["labels"] != -100).sum()))

print("\nCustom response-only loss:", float(test_loss))

print(
    "Loss is finite:",
    bool(torch.isfinite(test_loss))
)

print("\n" + "=" * 100)
print("CUSTOM LOSS CHECK COMPLETE")
print("=" * 100)

APEX V3 CUSTOM LOSS SANITY CHECK


OutOfMemoryError: CUDA out of memory. Tried to allocate 742.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 499.81 MiB is free. Including non-PyTorch memory, this process has 13.86 GiB memory in use. Of the allocated memory 13.02 GiB is allocated by PyTorch, and 712.59 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)

In [130]:
import gc
import torch
from peft import LoraConfig, get_peft_model

print("=" * 100)
print("APEX V3 — MEMORY-SAFE LoRA REBUILD")
print("=" * 100)

# Remove failed trainer/model
try:
    del apex_v3_trainer
except:
    pass

try:
    del apex_v3_model
except:
    pass

gc.collect()
torch.cuda.empty_cache()
torch.cuda.ipc_collect()

print("Memory after cleanup:")
print("Allocated:", round(torch.cuda.memory_allocated() / 1024**3, 2), "GB")
print("Reserved: ", round(torch.cuda.memory_reserved() / 1024**3, 2), "GB")

# Attention-only LoRA
APEX_V3_LORA_CONFIG_SMALL = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
    ],
)

apex_v3_model = get_peft_model(
    apex_v3_base,
    APEX_V3_LORA_CONFIG_SMALL
)

# Training-safe settings
apex_v3_model.config.use_cache = False
apex_v3_model.gradient_checkpointing_enable()

trainable_params = sum(
    p.numel()
    for p in apex_v3_model.parameters()
    if p.requires_grad
)

total_params = sum(
    p.numel()
    for p in apex_v3_model.parameters()
)

print("\nModel:", type(apex_v3_model).__name__)
print("LoRA targets: q/k/v/o only")
print("Trainable parameters:", f"{trainable_params:,}")
print("Total parameters:", f"{total_params:,}")
print(
    "Trainable %:",
    round(100 * trainable_params / total_params, 3)
)

print("\nGPU memory:")
print("Allocated:", round(torch.cuda.memory_allocated() / 1024**3, 2), "GB")
print("Reserved: ", round(torch.cuda.memory_reserved() / 1024**3, 2), "GB")

print("\n" + "=" * 100)
print("SMALLER LoRA MODEL READY")
print("=" * 100)

APEX V3 — MEMORY-SAFE LoRA REBUILD
Memory after cleanup:
Allocated: 9.66 GB
Reserved:  12.67 GB


/usr/local/lib/python3.13/dist-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/peft/tuners/tuners_utils.py:305: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(



Model: PeftModelForCausalLM
LoRA targets: q/k/v/o only
Trainable parameters: 33,030,144
Total parameters: 2,238,840,320
Trainable %: 1.475

GPU memory:
Allocated: 9.66 GB
Reserved:  12.67 GB

SMALLER LoRA MODEL READY


In [131]:
import gc
import torch
from transformers import AutoModelForCausalLM
from peft import LoraConfig, get_peft_model

print("=" * 100)
print("APEX V3 — FRESH BASE + CLEAN LoRA")
print("=" * 100)

# Remove all previous PEFT/model references
for name in [
    "apex_v3_trainer",
    "apex_v3_model",
    "apex_v3_base",
]:
    if name in globals():
        del globals()[name]

gc.collect()
torch.cuda.empty_cache()
torch.cuda.ipc_collect()

print("Memory after cleanup:")
print("Allocated:", round(torch.cuda.memory_allocated() / 1024**3, 2), "GB")
print("Reserved: ", round(torch.cuda.memory_reserved() / 1024**3, 2), "GB")

# Recreate the quantization config
clean_bnb_config_v3 = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

# IMPORTANT: load directly from MODEL_NAME again
# This guarantees a completely clean, non-PEFT base.
apex_v3_base = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=clean_bnb_config_v3,
    device_map="auto",
    trust_remote_code=True,
)

print("\nFresh base loaded.")
print("Model class:", type(apex_v3_base).__name__)
print(
    "Is PEFT model:",
    hasattr(apex_v3_base, "peft_config")
)

# Attention-only LoRA
APEX_V3_LORA_CONFIG_SMALL = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
    ],
)

apex_v3_model = get_peft_model(
    apex_v3_base,
    APEX_V3_LORA_CONFIG_SMALL
)

# Training-safe settings
apex_v3_model.config.use_cache = False
apex_v3_model.gradient_checkpointing_enable()

trainable_params = sum(
    p.numel()
    for p in apex_v3_model.parameters()
    if p.requires_grad
)

total_params = sum(
    p.numel()
    for p in apex_v3_model.parameters()
)

print("\nClean LoRA attached.")
print("Model class:", type(apex_v3_model).__name__)
print("Is PEFT model:", hasattr(apex_v3_model, "peft_config"))
print("Adapter names:", list(apex_v3_model.peft_config.keys()))
print("Trainable parameters:", f"{trainable_params:,}")
print("Total parameters:", f"{total_params:,}")
print(
    "Trainable %:",
    round(100 * trainable_params / total_params, 3)
)

print("\nGPU memory:")
print("Allocated:", round(torch.cuda.memory_allocated() / 1024**3, 2), "GB")
print("Reserved: ", round(torch.cuda.memory_reserved() / 1024**3, 2), "GB")

print("\n" + "=" * 100)
print("CLEAN APEX V3 MODEL READY")
print("=" * 100)

APEX V3 — FRESH BASE + CLEAN LoRA
Memory after cleanup:
Allocated: 9.66 GB
Reserved:  12.66 GB


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]


Fresh base loaded.
Model class: Qwen3ForCausalLM
Is PEFT model: False

Clean LoRA attached.
Model class: PeftModelForCausalLM
Is PEFT model: True
Adapter names: ['default']
Trainable parameters: 11,796,480
Total parameters: 2,217,606,656
Trainable %: 0.532

GPU memory:
Allocated: 12.2 GB
Reserved:  12.72 GB

CLEAN APEX V3 MODEL READY


In [132]:
# Make sure the model is in training-safe mode
apex_v3_model.config.use_cache = False
apex_v3_model.gradient_checkpointing_enable()

# Recreate collator for the clean model
apex_v3_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=apex_v3_model,
    padding=True,
    label_pad_token_id=-100,
    return_tensors="pt",
)

# Recreate the memory-efficient Trainer
apex_v3_trainer = APEXResponseOnlyTrainer(
    model=apex_v3_model,
    args=apex_v3_training_args,
    train_dataset=apex_v3_train_dataset,
    data_collator=apex_v3_collator,
)

print("=" * 100)
print("APEX V3 — CLEAN TRAINER")
print("=" * 100)

print("Model:", type(apex_v3_model).__name__)
print("Adapter:", list(apex_v3_model.peft_config.keys()))
print("Trainable parameters:", f"{sum(p.numel() for p in apex_v3_model.parameters() if p.requires_grad):,}")
print("Dataset:", len(apex_v3_train_dataset), "examples")
print("Max sequence:", max(len(x["input_ids"]) for x in apex_v3_train_dataset))
print("Learning rate:", apex_v3_training_args.learning_rate)
print("Epochs:", apex_v3_training_args.num_train_epochs)

print("\nTrainer:", type(apex_v3_trainer).__name__)

print("\n" + "=" * 100)
print("CLEAN TRAINER READY")
print("=" * 100)

APEX V3 — CLEAN TRAINER
Model: PeftModelForCausalLM
Adapter: ['default']
Trainable parameters: 11,796,480
Dataset: 18 examples
Max sequence: 1526
Learning rate: 5e-05
Epochs: 3

Trainer: APEXResponseOnlyTrainer

CLEAN TRAINER READY


In [133]:
import torch
import gc

gc.collect()
torch.cuda.empty_cache()
torch.cuda.ipc_collect()

print("=" * 100)
print("APEX V3 — FINAL MEMORY TEST")
print("=" * 100)

test_batch = apex_v3_collator(
    [apex_v3_train_dataset[0]]
)

model_device = next(apex_v3_model.parameters()).device

test_batch = {
    k: v.to(model_device) if torch.is_tensor(v) else v
    for k, v in test_batch.items()
}

apex_v3_model.eval()

print("Before forward:")
print("Allocated:", round(torch.cuda.memory_allocated() / 1024**3, 2), "GB")
print("Free:",
      round(
          (torch.cuda.get_device_properties(0).total_memory -
           torch.cuda.memory_allocated()) / 1024**3,
          2
      ),
      "GB"
)

with torch.no_grad():
    test_loss = apex_v3_trainer.compute_loss(
        apex_v3_model,
        test_batch
    )

print("\nForward pass: PASS")
print("Input tokens:", test_batch["input_ids"].shape[1])
print("Trainable tokens:",
      int((test_batch["labels"] != -100).sum()))
print("Response-only loss:", float(test_loss))
print("Loss is finite:", bool(torch.isfinite(test_loss)))

print("\nAfter forward:")
print("Allocated:", round(torch.cuda.memory_allocated() / 1024**3, 2), "GB")

print("\n" + "=" * 100)
print("FINAL MEMORY TEST COMPLETE")
print("=" * 100)

APEX V3 — FINAL MEMORY TEST
Before forward:
Allocated: 12.2 GB
Free: 2.36 GB

Forward pass: PASS
Input tokens: 789
Trainable tokens: 256
Response-only loss: 2.734375
Loss is finite: True

After forward:
Allocated: 12.2 GB

FINAL MEMORY TEST COMPLETE


In [137]:
from transformers import AutoTokenizer

MODEL_NAME = "Qwen/Qwen3-4B"

apex_v3_tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True
)

if apex_v3_tokenizer.pad_token is None:
    apex_v3_tokenizer.pad_token = apex_v3_tokenizer.eos_token

print("Tokenizer loaded successfully")
print("Tokenizer class:", type(apex_v3_tokenizer).__name__)
print("Vocab size:", len(apex_v3_tokenizer))
print("Pad token:", apex_v3_tokenizer.pad_token)
print("EOS token:", apex_v3_tokenizer.eos_token)

Tokenizer loaded successfully
Tokenizer class: Qwen2Tokenizer
Vocab size: 151669
Pad token: <|endoftext|>
EOS token: <|im_end|>


In [138]:
print("=" * 100)
print("CHECKING SEQUENCE LENGTH REQUIREMENT")
print("=" * 100)

lengths = []

for i, row in enumerate(v3_formatted_dataset):
    ids = apex_v3_tokenizer(
        row["text"],
        add_special_tokens=False,
        truncation=False
    )["input_ids"]

    lengths.append(len(ids))
    print(f"Example {i+1:02d}: {len(ids)} tokens")

print("\n" + "-" * 100)
print("Maximum:", max(lengths))
print("Minimum:", min(lengths))
print("Average:", round(sum(lengths) / len(lengths), 1))

for limit in [1024, 768, 640, 512]:
    truncated = sum(x > limit for x in lengths)
    print(f"At {limit:4d} tokens -> {truncated}/{len(lengths)} examples exceed limit")

CHECKING SEQUENCE LENGTH REQUIREMENT
Example 01: 789 tokens
Example 02: 885 tokens
Example 03: 669 tokens
Example 04: 497 tokens
Example 05: 1526 tokens
Example 06: 505 tokens
Example 07: 856 tokens
Example 08: 808 tokens
Example 09: 469 tokens
Example 10: 467 tokens
Example 11: 444 tokens
Example 12: 446 tokens
Example 13: 465 tokens
Example 14: 414 tokens
Example 15: 405 tokens
Example 16: 518 tokens
Example 17: 405 tokens
Example 18: 1388 tokens

----------------------------------------------------------------------------------------------------
Maximum: 1526
Minimum: 405
Average: 664.2
At 1024 tokens -> 2/18 examples exceed limit
At  768 tokens -> 6/18 examples exceed limit
At  640 tokens -> 7/18 examples exceed limit
At  512 tokens -> 8/18 examples exceed limit


In [139]:
print("=" * 100)
print("CHECKING 1024-TOKEN TRUNCATION SAFETY")
print("=" * 100)

LIMIT = 1024

for i, row in enumerate(v3_formatted_dataset):
    full_ids = apex_v3_tokenizer(
        row["text"],
        add_special_tokens=False,
        truncation=False
    )["input_ids"]

    if len(full_ids) > LIMIT:
        truncated_text = apex_v3_tokenizer.decode(
            full_ids[LIMIT:],
            skip_special_tokens=False
        )

        print(f"\nExample {i+1:02d}")
        print(f"Full length: {len(full_ids)}")
        print(f"Tokens removed: {len(full_ids) - LIMIT}")
        print("Removed portion:")
        print("-" * 80)
        print(truncated_text[:2000])
        print("-" * 80)

CHECKING 1024-TOKEN TRUNCATION SAFETY

Example 05
Full length: 1526
Tokens removed: 502
Removed portion:
--------------------------------------------------------------------------------
 any hose 
showing cracks or weeping 
Annually Safety relay functional test and log per ISO 

Interval Task 
13849 maintenance records 
Annually Electrical insulation test (Megger) on HPU 
motor 

7. Spare Parts 
Part No. Description Qty to Stock 
DW-BRK-RES Braking Resistor, 47 Ω, 1 kW 1 
DW-PT100-HYD PT100 Temperature Sensor, 
Hydraulic Oil 
1 
DW-PT100-MOT PT100 Temperature Sensor, 
HPU Motor 
1 
DW-KEY-AZ16 Schmersal AZ 16 Actuator Key 2 
DW-PEDAL-01 Foot Pedal Assembly 1 
DW-PNOZ-01 Pilz PNOZ X3 Safety Relay 1 
DW-FUSE-63A Main Supply Fuse, 63 A gG 6 
DW-HYD-FILT Hydraulic Return Filter 
Element 
2 
DW-SEAL-HYD Hydraulic Cylinder Seal Kit 1 
DW-ENC-BG Backgauge Axis Encoder 
(universal, X/R/Z) 
1 
 
For parts and service, contact DeltaWorks Industries: +49 89 4567 8900 / service@deltaworks-
industr

In [140]:
print("=" * 100)
print("MEASURING CONTEXT AND ASSISTANT TOKEN LENGTHS")
print("=" * 100)

for i, row in enumerate(v3_formatted_dataset):
    text = row["text"]

    # Locate the assistant start marker
    marker = "<|im_start|>assistant"
    assistant_pos = text.rfind(marker)

    prefix = text[:assistant_pos]
    assistant = text[assistant_pos:]

    prefix_tokens = len(
        apex_v3_tokenizer(
            prefix,
            add_special_tokens=False,
            truncation=False
        )["input_ids"]
    )

    assistant_tokens = len(
        apex_v3_tokenizer(
            assistant,
            add_special_tokens=False,
            truncation=False
        )["input_ids"]
    )

    total = prefix_tokens + assistant_tokens

    print(
        f"Example {i+1:02d}: "
        f"context={prefix_tokens:4d} | "
        f"assistant={assistant_tokens:3d} | "
        f"total={total:4d}"
    )

MEASURING CONTEXT AND ASSISTANT TOKEN LENGTHS
Example 01: context= 530 | assistant=259 | total= 789
Example 02: context= 682 | assistant=203 | total= 885
Example 03: context= 529 | assistant=140 | total= 669
Example 04: context= 301 | assistant=196 | total= 497
Example 05: context=1312 | assistant=214 | total=1526
Example 06: context= 321 | assistant=184 | total= 505
Example 07: context= 627 | assistant=229 | total= 856
Example 08: context= 576 | assistant=232 | total= 808
Example 09: context= 303 | assistant=166 | total= 469
Example 10: context= 313 | assistant=154 | total= 467
Example 11: context= 280 | assistant=164 | total= 444
Example 12: context= 290 | assistant=156 | total= 446
Example 13: context= 283 | assistant=182 | total= 465
Example 14: context= 258 | assistant=156 | total= 414
Example 15: context= 253 | assistant=152 | total= 405
Example 16: context= 323 | assistant=195 | total= 518
Example 17: context= 257 | assistant=148 | total= 405
Example 18: context=1104 | assistant

In [141]:
print("=" * 100)
print("INSPECTING LONG CONTEXT EXAMPLES")
print("=" * 100)

for idx in [4, 17]:  # Examples 05 and 18
    text = v3_formatted_dataset[idx]["text"]

    marker = "<|im_start|>assistant"
    assistant_pos = text.rfind(marker)

    context = text[:assistant_pos]

    print(f"\n{'=' * 100}")
    print(f"EXAMPLE {idx + 1:02d}")
    print(f"{'=' * 100}")

    # Remove chat-template wrappers so the actual prompt is easier to inspect
    print(context)

INSPECTING LONG CONTEXT EXAMPLES

EXAMPLE 05
<|im_start|>system
You are APEX, an industrial troubleshooting reasoning model. Use the provided maintenance evidence as the source of truth. Reason from evidence, distinguish documented facts from hypotheses, compare plausible causes when necessary, follow documented diagnostic order, respect safety requirements, and do not invent machine facts.<|im_end|>
<|im_start|>user
Machine:
DeltaWorks DX-200 Automated Press Brake

Observed fault:
F002 — DC Link Voltage Too Low

Question:
What is the relationship between these faults, and how are they different?

Evidence from the maintenance manual:
F002 — DC Link Voltage Too Low 
Description: The DC link voltage has fallen below 500 V DC, indicating the mains supply is 
insufficient to maintain the regulated bus. 
Probable Causes: 
 Phase loss on the input supply (check F099 for phase failure indication). 
 Main rectifier diode failure. 
 Supply cable undersized, causing voltage drop under load. 

In [142]:
print("=" * 100)
print("COMPACTING LONG TRAINING EXAMPLES")
print("=" * 100)

def rebuild_chat(system_text, user_text, assistant_text):
    return (
        "<|im_start|>system\n"
        + system_text.strip()
        + "<|im_end|>\n"
        "<|im_start|>user\n"
        + user_text.strip()
        + "<|im_end|>\n"
        "<|im_start|>assistant\n"
        + assistant_text.strip()
        + "<|im_end|>"
    )

system_text = (
    "You are APEX, an industrial troubleshooting reasoning model. "
    "Use the provided maintenance evidence as the source of truth. "
    "Reason from evidence, distinguish documented facts from hypotheses, "
    "compare plausible causes when necessary, follow documented diagnostic order, "
    "respect safety requirements, and do not invent machine facts."
)

# -------------------------
# Example 05
# -------------------------
ex5_assistant = reasoning_targets[4]

ex5_user = """
Machine:
DeltaWorks DX-200 Automated Press Brake

Observed fault:
F002 — DC Link Voltage Too Low

Question:
What is the relationship between these faults, and how are they different?

Evidence from the maintenance manual:

F002 — DC Link Voltage Too Low
Description: The DC link voltage has fallen below 500 V DC, indicating the mains supply is insufficient to maintain the regulated bus.

Probable Causes:
- Phase loss on the input supply (check F099 for phase failure indication).
- Main rectifier diode failure.
- Supply cable undersized, causing voltage drop under load.

Corrective Steps:
1. Measure all three phase voltages at the drive input terminals.
2. Check fuses F1, F2, F3 in the drive input fuse holder.
3. If all three phases are present and fuses intact, request a rectifier diode test from a qualified engineer.
4. Verify supply cable cross-section matches the installation drawing (minimum 16 mm² per phase).

Additional related fault evidence:

F099 — Power Supply Phase Failure
Description: The three-phase monitoring relay (Finmotor type FIN 3P) has detected the loss of one or more incoming supply phases, or a phase sequence reversal.

Probable Causes:
- A main supply fuse (F1, F2, or F3 in the main isolator) has blown.
- The utility supply has lost a phase.
- The main isolator or contactor (K-MAIN) has developed a faulty contact on one pole.

Corrective Steps:
1. Check all three incoming phase voltages at the main isolator input terminals; all three must read 230 V AC line-to-neutral (400 V line-to-line).
2. Inspect fuses F1, F2, and F3 in the main supply fuse holder.
3. If all three fuses and phases are present at the input but one phase is absent at the output, inspect the main isolator contacts and K-MAIN contactor.
4. After restoring the supply, manually reset the phase monitor relay only after all three phases are confirmed healthy.
"""

v3_formatted_dataset[4]["text"] = rebuild_chat(
    system_text,
    ex5_user,
    ex5_assistant
)

# -------------------------
# Example 18
# -------------------------
ex18_assistant = reasoning_targets[17]

ex18_user = """
Machine:
DeltaWorks DX-200 Automated Press Brake

Observed fault:
F099 — Power Supply Phase Failure

Question:
What are the documented causes and corrective steps for a phase failure?

Evidence from the maintenance manual:

F099 — Power Supply Phase Failure
Description: The three-phase monitoring relay (Finmotor type FIN 3P) has detected the loss of one or more incoming supply phases, or a phase sequence reversal. The machine shuts down the hydraulic pump motor and all servo axes immediately because single-phase operation of the 22 kW HPU motor can cause rapid overheating and motor winding failure. The phase monitor relay latches the fault and does not auto-reset when supply is restored.

Probable Causes:
- A main supply fuse (F1, F2, or F3 in the main isolator) has blown.
- The utility supply has lost a phase.
- The main isolator or contactor (K-MAIN) has developed a faulty contact on one pole.

Corrective Steps:
1. Check all three incoming phase voltages at the main isolator input terminals; all three must read 230 V AC line-to-neutral (400 V line-to-line). Note which phase is absent or low.
2. Inspect fuses F1, F2, and F3 and replace any blown fuse with the identically rated DW-FUSE-63A, gG 63 A fuse. Do not use a higher-rated fuse.
3. If all three fuses are intact and all three phases are present at the isolator input but one phase is absent at the output, inspect the main isolator contacts and K-MAIN contactor and contact a qualified electrician.
4. After restoring the supply, manually reset the phase monitor relay only after all three phases are confirmed healthy.
"""

v3_formatted_dataset[17]["text"] = rebuild_chat(
    system_text,
    ex18_user,
    ex18_assistant
)

# -------------------------
# Re-tokenize at 1024
# -------------------------
MAX_LENGTH = 1024

v3_1024_tokenized = []

for row in v3_formatted_dataset:
    ids = apex_v3_tokenizer(
        row["text"],
        add_special_tokens=False,
        truncation=False
    )["input_ids"]

    v3_1024_tokenized.append(ids)

print("\nNew sequence lengths:")
for i, ids in enumerate(v3_1024_tokenized):
    print(f"Example {i+1:02d}: {len(ids)} tokens")

print("\nMaximum:", max(map(len, v3_1024_tokenized)))
print("Examples over 1024:", sum(len(x) > 1024 for x in v3_1024_tokenized))

print("\n" + "=" * 100)
print("COMPACTION COMPLETE")
print("=" * 100)

COMPACTING LONG TRAINING EXAMPLES

New sequence lengths:
Example 01: 789 tokens
Example 02: 885 tokens
Example 03: 669 tokens
Example 04: 497 tokens
Example 05: 1526 tokens
Example 06: 505 tokens
Example 07: 856 tokens
Example 08: 808 tokens
Example 09: 469 tokens
Example 10: 467 tokens
Example 11: 444 tokens
Example 12: 446 tokens
Example 13: 465 tokens
Example 14: 414 tokens
Example 15: 405 tokens
Example 16: 518 tokens
Example 17: 405 tokens
Example 18: 1388 tokens

Maximum: 1526
Examples over 1024: 2

COMPACTION COMPLETE


In [143]:
print("=" * 100)
print("REBUILDING EXAMPLES 05 AND 18 PROPERLY")
print("=" * 100)

# Make a normal Python copy of all texts
v3_texts = [row["text"] for row in v3_formatted_dataset]

# Replace the two long examples
v3_texts[4] = rebuild_chat(
    system_text,
    ex5_user,
    reasoning_targets[4]
)

v3_texts[17] = rebuild_chat(
    system_text,
    ex18_user,
    reasoning_targets[17]
)

# Verify lengths BEFORE creating the training dataset
new_lengths = []

for i, text in enumerate(v3_texts):
    ids = apex_v3_tokenizer(
        text,
        add_special_tokens=False,
        truncation=False
    )["input_ids"]

    new_lengths.append(len(ids))

    if i in [4, 17]:
        print(f"\nExample {i+1:02d}")
        print("New length:", len(ids))
        print("Assistant target tokens:", len(
            apex_v3_tokenizer(
                reasoning_targets[i],
                add_special_tokens=False,
                truncation=False
            )["input_ids"]
        ))

print("\n" + "-" * 100)
print("ALL LENGTHS")
print("-" * 100)

for i, length in enumerate(new_lengths):
    print(f"Example {i+1:02d}: {length} tokens")

print("\nMaximum:", max(new_lengths))
print("Examples over 1024:", sum(x > 1024 for x in new_lengths))

REBUILDING EXAMPLES 05 AND 18 PROPERLY

Example 05
New length: 720
Assistant target tokens: 205

Example 18
New length: 719
Assistant target tokens: 275

----------------------------------------------------------------------------------------------------
ALL LENGTHS
----------------------------------------------------------------------------------------------------
Example 01: 789 tokens
Example 02: 885 tokens
Example 03: 669 tokens
Example 04: 497 tokens
Example 05: 720 tokens
Example 06: 505 tokens
Example 07: 856 tokens
Example 08: 808 tokens
Example 09: 469 tokens
Example 10: 467 tokens
Example 11: 444 tokens
Example 12: 446 tokens
Example 13: 465 tokens
Example 14: 414 tokens
Example 15: 405 tokens
Example 16: 518 tokens
Example 17: 405 tokens
Example 18: 719 tokens

Maximum: 885
Examples over 1024: 0


In [144]:
print("=" * 100)
print("BUILDING CLEAN APEX V3 TRAINING DATASET")
print("=" * 100)

MAX_LENGTH = 1024

tokenized_rows = []

for text in v3_texts:
    enc = apex_v3_tokenizer(
        text,
        add_special_tokens=False,
        truncation=True,
        max_length=MAX_LENGTH
    )

    input_ids = enc["input_ids"]
    attention_mask = enc["attention_mask"]

    # Find assistant response start
    assistant_marker_ids = apex_v3_tokenizer(
        "<|im_start|>assistant",
        add_special_tokens=False
    )["input_ids"]

    # Locate assistant marker
    response_start = None

    for j in range(len(input_ids) - len(assistant_marker_ids) + 1):
        if input_ids[j:j + len(assistant_marker_ids)] == assistant_marker_ids:
            response_start = j + len(assistant_marker_ids)
            break

    if response_start is None:
        raise ValueError("Assistant marker not found")

    # Mask everything before assistant response
    labels = [-100] * response_start + input_ids[response_start:]

    # Safety check
    trainable_tokens = sum(x != -100 for x in labels)

    if trainable_tokens == 0:
        raise ValueError("No trainable assistant tokens found")

    tokenized_rows.append({
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels
    })

from datasets import Dataset

apex_v3_train_dataset = Dataset.from_list(tokenized_rows)

print("\nDataset created successfully")
print("Examples:", len(apex_v3_train_dataset))
print("Columns:", apex_v3_train_dataset.column_names)

print("\nSequence lengths:")
for i, row in enumerate(apex_v3_train_dataset):
    print(
        f"Example {i+1:02d}: "
        f"sequence={len(row['input_ids'])}, "
        f"trainable={sum(x != -100 for x in row['labels'])}"
    )

print("\n" + "-" * 100)

seq_lengths = [len(x["input_ids"]) for x in apex_v3_train_dataset]
train_lengths = [
    sum(x != -100 for x in row["labels"])
    for row in apex_v3_train_dataset
]

print("Maximum sequence:", max(seq_lengths))
print("Average sequence:", round(sum(seq_lengths) / len(seq_lengths), 1))
print("Maximum trainable:", max(train_lengths))
print("Average trainable:", round(sum(train_lengths) / len(train_lengths), 1))

assert max(seq_lengths) <= MAX_LENGTH
assert all(x > 0 for x in train_lengths)

print("\nPASS: Complete responses preserved")
print("PASS: Response-only labels created")
print("PASS: No example exceeds 1024 tokens")

BUILDING CLEAN APEX V3 TRAINING DATASET

Dataset created successfully
Examples: 18
Columns: ['input_ids', 'attention_mask', 'labels']

Sequence lengths:
Example 01: sequence=789, trainable=257
Example 02: sequence=885, trainable=201
Example 03: sequence=669, trainable=138
Example 04: sequence=497, trainable=194
Example 05: sequence=720, trainable=207
Example 06: sequence=505, trainable=182
Example 07: sequence=856, trainable=227
Example 08: sequence=808, trainable=230
Example 09: sequence=469, trainable=164
Example 10: sequence=467, trainable=152
Example 11: sequence=444, trainable=162
Example 12: sequence=446, trainable=154
Example 13: sequence=465, trainable=180
Example 14: sequence=414, trainable=154
Example 15: sequence=405, trainable=150
Example 16: sequence=518, trainable=193
Example 17: sequence=405, trainable=146
Example 18: sequence=719, trainable=277

----------------------------------------------------------------------------------------------------
Maximum sequence: 885
Ave

In [145]:
print("=" * 100)
print("UPDATING TRAINER WITH CLEAN DATASET")
print("=" * 100)

# Replace the trainer's dataset with the newly rebuilt dataset
apex_v3_trainer.train_dataset = apex_v3_train_dataset

# Make sure the model is in training mode
apex_v3_model.train()

print("Trainer dataset:", type(apex_v3_trainer.train_dataset).__name__)
print("Training examples:", len(apex_v3_trainer.train_dataset))
print("Maximum sequence length:", max(
    len(row["input_ids"])
    for row in apex_v3_trainer.train_dataset
))

print("\nTrainer successfully updated.")

UPDATING TRAINER WITH CLEAN DATASET
Trainer dataset: Dataset
Training examples: 18
Maximum sequence length: 885

Trainer successfully updated.


In [146]:
import gc
import torch

print("=" * 100)
print("CLEANING GPU MEMORY")
print("=" * 100)

gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

free_before, total = torch.cuda.mem_get_info()

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Total VRAM: {total / 1024**3:.2f} GB")
print(f"Free VRAM:  {free_before / 1024**3:.2f} GB")
print(f"Allocated:  {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
print(f"Reserved:   {torch.cuda.memory_reserved() / 1024**3:.2f} GB")

CLEANING GPU MEMORY
GPU: Tesla T4
Total VRAM: 14.56 GB
Free VRAM:  1.76 GB
Allocated:  10.32 GB
Reserved:   12.65 GB


In [147]:
print("=" * 100)
print("APEX V3 — SINGLE-STEP MEMORY TEST")
print("=" * 100)

# Temporarily train for exactly one optimizer step
original_epochs = apex_v3_trainer.args.num_train_epochs
original_max_steps = apex_v3_trainer.args.max_steps

apex_v3_trainer.args.num_train_epochs = 1
apex_v3_trainer.args.max_steps = 1

apex_v3_model.train()

try:
    test_result = apex_v3_trainer.train()

    print("\n" + "=" * 100)
    print("SINGLE-STEP TEST PASSED")
    print("=" * 100)
    print("Training loss:", test_result.training_loss)
    print("Global step:", test_result.global_step)

except torch.cuda.OutOfMemoryError as e:
    print("\n" + "=" * 100)
    print("SINGLE-STEP TEST: OOM")
    print("=" * 100)
    print(str(e))

finally:
    # Restore original training configuration
    apex_v3_trainer.args.num_train_epochs = original_epochs
    apex_v3_trainer.args.max_steps = original_max_steps

    gc.collect()
    torch.cuda.empty_cache()

    print("\nTrainer configuration restored.")

APEX V3 — SINGLE-STEP MEMORY TEST


Step,Training Loss
1,8.771724



SINGLE-STEP TEST PASSED
Training loss: 8.771723747253418
Global step: 1

Trainer configuration restored.


In [148]:
print("=" * 100)
print("STARTING FULL APEX V3 TRAINING")
print("=" * 100)

# Make sure the clean dataset is attached
apex_v3_trainer.train_dataset = apex_v3_train_dataset

# Restore intended training configuration
apex_v3_trainer.args.num_train_epochs = APEX_V3_EPOCHS
apex_v3_trainer.args.max_steps = -1

apex_v3_model.train()

v3_train_result = apex_v3_trainer.train()

print("\n" + "=" * 100)
print("APEX V3 TRAINING COMPLETE")
print("=" * 100)

print("Final training loss:", v3_train_result.training_loss)
print("Training steps:", v3_train_result.global_step)
print(
    "Training runtime:",
    v3_train_result.metrics.get("train_runtime")
)

STARTING FULL APEX V3 TRAINING


Step,Training Loss
1,8.131836
2,8.155832
3,6.913171
4,7.239276
5,3.781390
6,6.800829
7,7.216550
8,5.988506
9,6.460954
10,3.129228



APEX V3 TRAINING COMPLETE
Final training loss: 6.078722953796387
Training steps: 15
Training runtime: 94.7866


In [149]:
import os
import torch

print("=" * 100)
print("SAVING APEX V3 ADAPTER")
print("=" * 100)

APEX_V3_OUTPUT = "./apex_model_v3_final"

os.makedirs(APEX_V3_OUTPUT, exist_ok=True)

apex_v3_model.save_pretrained(APEX_V3_OUTPUT)
apex_v3_tokenizer.save_pretrained(APEX_V3_OUTPUT)

print("\nSaved successfully.")
print("Output directory:", APEX_V3_OUTPUT)

print("\nSaved files:")
for name in sorted(os.listdir(APEX_V3_OUTPUT)):
    path = os.path.join(APEX_V3_OUTPUT, name)
    size_mb = os.path.getsize(path) / (1024 * 1024)
    print(f"{name:40s} {size_mb:8.2f} MB")

SAVING APEX V3 ADAPTER

Saved successfully.
Output directory: ./apex_model_v3_final

Saved files:
README.md                                    0.00 MB
adapter_config.json                          0.00 MB
adapter_model.safetensors                   45.04 MB
chat_template.jinja                          0.00 MB
tokenizer.json                              10.89 MB
tokenizer_config.json                        0.00 MB


In [150]:
import torch
from transformers import AutoModelForCausalLM
from peft import PeftModel

print("=" * 100)
print("LOADING CLEAN BASE + TRAINED APEX V3")
print("=" * 100)

# Clean base model
comparison_base = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

comparison_base.config.use_cache = True

# Attach trained adapter
comparison_apex = PeftModel.from_pretrained(
    comparison_base,
    APEX_V3_OUTPUT
)

comparison_apex.eval()

print("Base model:", type(comparison_base).__name__)
print("APEX model:", type(comparison_apex).__name__)
print("Adapter loaded:", comparison_apex.peft_config.keys())

print(
    "GPU allocated:",
    round(torch.cuda.memory_allocated() / 1024**3, 2),
    "GB"
)

print("\nPASS: Clean base and trained adapter loaded.")

LOADING CLEAN BASE + TRAINED APEX V3


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Base model: Qwen3ForCausalLM
APEX model: PeftModelForCausalLM
Adapter loaded: dict_keys(['default'])
GPU allocated: 12.86 GB

PASS: Clean base and trained adapter loaded.


In [151]:
print("=" * 100)
print("BASE vs APEX V3 — F001 BEHAVIOR TEST")
print("=" * 100)

prompt = """Machine:
DeltaWorks DX-200 Automated Press Brake

Observed fault:
F001 — DC Link Voltage Too High

Maintenance evidence:
F001 means the DC link voltage has exceeded its specified limit.

Documented possible causes:
- Braking resistor or cable failed/open.
- BG.DECEL setting too aggressive.
- Mains voltage above the permitted range.

Documented diagnostic information:
- Measure mains voltage at the main isolator terminals.
- Permitted mains range: 360–440 V AC.
- If mains is acceptable, apply LOTO and measure the braking resistor between R+ and RB.
- Nominal braking resistor value: 47 Ω ±10%.
- If the resistor is healthy, increase BG.DECEL by 25%.
- If the fault persists with normal mains and a healthy resistor, request a DC link capacitor capacitance test from DeltaWorks service.

Question:
Diagnose the fault and explain the correct diagnostic order. Do not assume a specific component has failed without evidence."""

def generate_test(model, prompt, max_new_tokens=300):
    messages = [
        {
            "role": "system",
            "content": (
                "You are APEX, an industrial troubleshooting reasoning model. "
                "Use maintenance evidence as the source of truth. "
                "Reason from evidence, distinguish facts from hypotheses, "
                "follow diagnostic order, respect safety requirements, "
                "and do not invent machine facts."
            )
        },
        {
            "role": "user",
            "content": prompt
        }
    ]

    text = apex_v3_tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False
    )

    inputs = apex_v3_tokenizer(
        text,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=None,
            top_p=None
        )

    generated = output_ids[0][inputs["input_ids"].shape[1]:]

    return apex_v3_tokenizer.decode(
        generated,
        skip_special_tokens=True
    )

print("\n" + "=" * 100)
print("CLEAN BASE OUTPUT")
print("=" * 100)

base_output = generate_test(comparison_base, prompt)

print(base_output)

print("\n" + "=" * 100)
print("TRAINED APEX V3 OUTPUT")
print("=" * 100)

apex_output = generate_test(comparison_apex, prompt)

print(apex_output)

print("\n" + "=" * 100)
print("COMPARISON COMPLETE")
print("=" * 100)

BASE vs APEX V3 — F001 BEHAVIOR TEST

CLEAN BASE OUTPUT
To diagnose the fault F001 — DC Link Voltage Too High on the DeltaWorks DX-200 Automated Press Brake, we must follow a structured diagnostic order based on the documented evidence and safety requirements. The goal is to identify the root cause without assuming a specific component has failed without evidence.

### **Diagnostic Order:**

1. **Verify Mains Voltage:**
   - **Action:** Measure the mains voltage at the main isolator terminals.
   - **Evidence:** The permitted mains range is 360–440 V AC.
   - **Reason:** The DC link voltage is influenced by the mains voltage. If the mains voltage is outside the permitted range, it could directly cause the DC link voltage to be too high.

2. **If Mains Voltage is Within Permitted Range:**
   - **Action:** Apply LOTO (Lockout/Tagout) to ensure safety.
   - **Action:** Measure the braking resistor between R+ and RB.
   - **Evidence:** The nominal braking resistor value is 47 Ω ±10%.
   - 

In [152]:
print("=" * 100)
print("MEASURING BASE vs APEX V3 DIFFERENCE")
print("=" * 100)

comparison_base.eval()
comparison_apex.eval()

# Tokenize the exact test prompt
messages = [
    {
        "role": "system",
        "content": (
            "You are APEX, an industrial troubleshooting reasoning model. "
            "Use maintenance evidence as the source of truth. "
            "Reason from evidence, distinguish facts from hypotheses, "
            "follow diagnostic order, respect safety requirements, "
            "and do not invent machine facts."
        )
    },
    {
        "role": "user",
        "content": prompt
    }
]

test_text = apex_v3_tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False
)

test_inputs = apex_v3_tokenizer(
    test_text,
    return_tensors="pt"
).to(comparison_base.device)

with torch.no_grad():
    base_logits = comparison_base(**test_inputs).logits[:, -1, :]
    apex_logits = comparison_apex(**test_inputs).logits[:, -1, :]

diff = (apex_logits.float() - base_logits.float()).abs()

print("Maximum logit difference:", diff.max().item())
print("Mean logit difference:", diff.mean().item())
print("Non-zero differences:", (diff > 1e-7).sum().item())

base_top = torch.topk(base_logits, 10, dim=-1)
apex_top = torch.topk(apex_logits, 10, dim=-1)

print("\nTop-10 next tokens — BASE:")
for token_id, score in zip(
    base_top.indices[0].tolist(),
    base_top.values[0].tolist()
):
    print(
        repr(apex_v3_tokenizer.decode([token_id])),
        f"{score:.4f}"
    )

print("\nTop-10 next tokens — APEX:")
for token_id, score in zip(
    apex_top.indices[0].tolist(),
    apex_top.values[0].tolist()
):
    print(
        repr(apex_v3_tokenizer.decode([token_id])),
        f"{score:.4f}"
    )

print("\n" + "=" * 100)
print("LOGIT COMPARISON COMPLETE")
print("=" * 100)

MEASURING BASE vs APEX V3 DIFFERENCE
Maximum logit difference: 0.0
Mean logit difference: 0.0
Non-zero differences: 0

Top-10 next tokens — BASE:
'To' 22.1250
'###' 22.1250
'The' 21.3750
'**' 20.5000
'Based' 19.8750
'I' 19.8750
'Given' 19.7500
'1' 19.5000
'F' 18.7500
'Let' 18.7500

Top-10 next tokens — APEX:
'To' 22.1250
'###' 22.1250
'The' 21.3750
'**' 20.5000
'Based' 19.8750
'I' 19.8750
'Given' 19.7500
'1' 19.5000
'F' 18.7500
'Let' 18.7500

LOGIT COMPARISON COMPLETE


In [153]:
import os
import torch
from safetensors.torch import load_file

print("=" * 100)
print("CHECKING SAVED APEX V3 ADAPTER WEIGHTS")
print("=" * 100)

adapter_path = os.path.join(
    APEX_V3_OUTPUT,
    "adapter_model.safetensors"
)

adapter_state = load_file(adapter_path)

print("Adapter tensors:", len(adapter_state))

total_elements = 0
nonzero_elements = 0
max_abs = 0.0
sum_abs = 0.0

for name, tensor in adapter_state.items():
    t = tensor.float()

    total_elements += t.numel()
    nonzero_elements += (t != 0).sum().item()
    max_abs = max(max_abs, t.abs().max().item())
    sum_abs += t.abs().sum().item()

    print(
        f"{name:70s} "
        f"shape={tuple(tensor.shape)} "
        f"max={t.abs().max().item():.6e}"
    )

print("\n" + "-" * 100)
print("Total elements:", total_elements)
print("Non-zero elements:", nonzero_elements)
print(
    "Non-zero percentage:",
    round(100 * nonzero_elements / total_elements, 4)
)
print("Maximum absolute weight:", max_abs)
print("Mean absolute weight:", sum_abs / total_elements)

print("\n" + "=" * 100)
print("ADAPTER WEIGHT CHECK COMPLETE")
print("=" * 100)

CHECKING SAVED APEX V3 ADAPTER WEIGHTS
Adapter tensors: 288
base_model.model.model.layers.0.self_attn.k_proj.lora_A.weight         shape=(16, 2560) max=2.020598e-02
base_model.model.model.layers.0.self_attn.k_proj.lora_B.weight         shape=(1024, 16) max=5.510297e-04
base_model.model.model.layers.0.self_attn.o_proj.lora_A.weight         shape=(16, 4096) max=1.608848e-02
base_model.model.model.layers.0.self_attn.o_proj.lora_B.weight         shape=(2560, 16) max=5.585675e-04
base_model.model.model.layers.0.self_attn.q_proj.lora_A.weight         shape=(16, 2560) max=2.019821e-02
base_model.model.model.layers.0.self_attn.q_proj.lora_B.weight         shape=(4096, 16) max=5.620072e-04
base_model.model.model.layers.0.self_attn.v_proj.lora_A.weight         shape=(16, 2560) max=2.021456e-02
base_model.model.model.layers.0.self_attn.v_proj.lora_B.weight         shape=(1024, 16) max=5.616719e-04
base_model.model.model.layers.1.self_attn.k_proj.lora_A.weight         shape=(16, 2560) max=2.018203

In [154]:
print("=" * 100)
print("CORRECT BASE vs APEX V3 LOGIT TEST")
print("=" * 100)

comparison_apex.eval()

# ---------------------------------------------------------
# Adapter OFF = clean Qwen3-4B behavior
# Adapter ON  = trained APEX V3 behavior
# ---------------------------------------------------------

with torch.no_grad():
    # Clean base
    with comparison_apex.disable_adapter():
        base_logits = comparison_apex(**test_inputs).logits[:, -1, :]

    # Trained APEX
    apex_logits = comparison_apex(**test_inputs).logits[:, -1, :]

diff = (apex_logits.float() - base_logits.float()).abs()

print("Maximum logit difference:", diff.max().item())
print("Mean logit difference:", diff.mean().item())
print("Changed logits (>1e-7):", (diff > 1e-7).sum().item())

# Compare top tokens
base_top = torch.topk(base_logits, 10, dim=-1)
apex_top = torch.topk(apex_logits, 10, dim=-1)

print("\n" + "-" * 100)
print("TOP-10 NEXT TOKENS — BASE / ADAPTER OFF")
print("-" * 100)

for token_id, score in zip(
    base_top.indices[0].tolist(),
    base_top.values[0].tolist()
):
    print(
        f"{repr(apex_v3_tokenizer.decode([token_id])):20s} "
        f"{score:.4f}"
    )

print("\n" + "-" * 100)
print("TOP-10 NEXT TOKENS — APEX V3 / ADAPTER ON")
print("-" * 100)

for token_id, score in zip(
    apex_top.indices[0].tolist(),
    apex_top.values[0].tolist()
):
    print(
        f"{repr(apex_v3_tokenizer.decode([token_id])):20s} "
        f"{score:.4f}"
    )

print("\n" + "=" * 100)
print("CORRECT LOGIT TEST COMPLETE")
print("=" * 100)

CORRECT BASE vs APEX V3 LOGIT TEST
Maximum logit difference: 11.75
Mean logit difference: 3.424964427947998
Changed logits (>1e-7): 151845

----------------------------------------------------------------------------------------------------
TOP-10 NEXT TOKENS — BASE / ADAPTER OFF
----------------------------------------------------------------------------------------------------
'**'                 32.2500
'###'                31.6250
'To'                 29.1250
'The'                27.8750
'Based'              24.3750
'Ap'                 22.7500
'Given'              22.3750
'##'                 22.2500
'AP'                 22.0000
'#'                  21.5000

----------------------------------------------------------------------------------------------------
TOP-10 NEXT TOKENS — APEX V3 / ADAPTER ON
----------------------------------------------------------------------------------------------------
'To'                 22.1250
'###'                22.1250
'The'                21.3

In [155]:
print("=" * 100)
print("APEX V3 — REAL GENERATION TEST")
print("=" * 100)

# Use the already-loaded APEX model: comparison_apex
comparison_apex.eval()

test_query = """The Delta DX-200 shows fault F001. The operator says it happens
during rapid backgauge movement. What should I check first, and how should I
distinguish between the possible causes?"""

messages = [
    {
        "role": "system",
        "content": (
            "You are APEX, an industrial troubleshooting reasoning model. "
            "Use the provided maintenance evidence as the source of truth. "
            "Reason from evidence, distinguish documented facts from hypotheses, "
            "compare plausible causes when necessary, follow documented diagnostic "
            "order, respect safety requirements, and do not invent machine facts."
        )
    },
    {
        "role": "user",
        "content": test_query
    }
]

prompt = apex_v3_tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = apex_v3_tokenizer(
    prompt,
    return_tensors="pt"
).to(comparison_apex.device)

with torch.no_grad():
    output_ids = comparison_apex.generate(
        **inputs,
        max_new_tokens=300,
        do_sample=False,
        temperature=None,
        top_p=None,
        use_cache=False
    )

generated = output_ids[0][inputs["input_ids"].shape[1]:]

print("\n" + "-" * 100)
print("APEX V3 OUTPUT")
print("-" * 100)

print(apex_v3_tokenizer.decode(
    generated,
    skip_special_tokens=True
))

print("\n" + "=" * 100)
print("GENERATION TEST COMPLETE")
print("=" * 100)

APEX V3 — REAL GENERATION TEST

----------------------------------------------------------------------------------------------------
APEX V3 OUTPUT
----------------------------------------------------------------------------------------------------
<think>
Okay, the user is dealing with a Delta DX-200 machine that's showing fault F001, and it happens during rapid backgauge movement. I need to figure out what to check first and how to differentiate between possible causes.

First, I should recall the maintenance evidence provided. The key points are the fault code F001 and the timing—rapid backgauge movement. The user is asking for the first steps and how to distinguish between causes.

From the maintenance evidence, I remember that F001 is typically related to the backgauge system. The possible causes could be related to the backgauge's mechanical components, sensors, or the control system. The operator mentioned rapid movement, so maybe there's an issue with the motor, encoder, or the

In [156]:
print("=" * 100)
print("APEX V3 — GENERATION TOKEN DIAGNOSTIC")
print("=" * 100)

comparison_apex.eval()

with torch.no_grad():
    output_ids = comparison_apex.generate(
        **inputs,
        max_new_tokens=50,
        do_sample=False,
        use_cache=False,
        return_dict_in_generate=True,
    )

generated_ids = output_ids.sequences[0][inputs["input_ids"].shape[1]:]

print("Generated token count:", len(generated_ids))
print("Generated token IDs:", generated_ids.tolist())

print("\nDecoded WITHOUT skipping special tokens:")
print(repr(apex_v3_tokenizer.decode(
    generated_ids,
    skip_special_tokens=False
)))

print("\nDecoded WITH special tokens skipped:")
print(repr(apex_v3_tokenizer.decode(
    generated_ids,
    skip_special_tokens=True
)))

print("\nTokenizer special tokens:")
print("EOS:", apex_v3_tokenizer.eos_token, apex_v3_tokenizer.eos_token_id)
print("BOS:", apex_v3_tokenizer.bos_token, apex_v3_tokenizer.bos_token_id)
print("PAD:", apex_v3_tokenizer.pad_token, apex_v3_tokenizer.pad_token_id)

print("\n" + "=" * 100)
print("TOKEN DIAGNOSTIC COMPLETE")
print("=" * 100)

APEX V3 — GENERATION TOKEN DIAGNOSTIC
Generated token count: 50
Generated token IDs: [151667, 198, 32313, 11, 279, 1196, 374, 14550, 448, 264, 24957, 30808, 12, 17, 15, 15, 5662, 429, 594, 9027, 14527, 434, 15, 15, 16, 11, 323, 432, 8573, 2337, 11048, 1182, 70, 19392, 7203, 13, 358, 1184, 311, 7071, 700, 1128, 311, 1779, 1156, 323, 1246, 311, 53163, 1948]

Decoded WITHOUT skipping special tokens:
"<think>\nOkay, the user is dealing with a Delta DX-200 machine that's showing fault F001, and it happens during rapid backgauge movement. I need to figure out what to check first and how to differentiate between"

Decoded WITH special tokens skipped:
"<think>\nOkay, the user is dealing with a Delta DX-200 machine that's showing fault F001, and it happens during rapid backgauge movement. I need to figure out what to check first and how to differentiate between"

Tokenizer special tokens:
EOS: <|im_end|> 151645
BOS: None None
PAD: <|endoftext|> 151643

TOKEN DIAGNOSTIC COMPLETE


In [157]:
print("=" * 100)
print("APEX V3 — FULL REASONING + ANSWER TEST")
print("=" * 100)

comparison_apex.eval()

with torch.no_grad():
    output_ids = comparison_apex.generate(
        **inputs,
        max_new_tokens=600,
        do_sample=False,
        use_cache=False,
        return_dict_in_generate=True,
    )

generated_ids = output_ids.sequences[0][inputs["input_ids"].shape[1]:]

generated_text = apex_v3_tokenizer.decode(
    generated_ids,
    skip_special_tokens=False
)

print("\n" + "-" * 100)
print("RAW APEX GENERATION")
print("-" * 100)

print(generated_text)

print("\n" + "-" * 100)
print("GENERATION STATS")
print("-" * 100)

print("Generated tokens:", len(generated_ids))
print("Contains <think>:", "<think>" in generated_text)
print("Contains </think>:", "</think>" in generated_text)
print("Contains <|im_end|>:", "<|im_end|>" in generated_text)

print("\n" + "=" * 100)
print("FULL GENERATION TEST COMPLETE")
print("=" * 100)

APEX V3 — FULL REASONING + ANSWER TEST

----------------------------------------------------------------------------------------------------
RAW APEX GENERATION
----------------------------------------------------------------------------------------------------
<think>
Okay, the user is dealing with a Delta DX-200 machine that's showing fault F001, and it happens during rapid backgauge movement. I need to figure out what to check first and how to differentiate between possible causes.

First, I should recall the maintenance evidence provided. The key points are the fault code F001 and the timing—rapid backgauge movement. The user is asking for the first steps and how to distinguish between causes.

From the maintenance evidence, I remember that F001 is typically related to the backgauge system. The possible causes could be related to the backgauge's mechanical components, sensors, or the control system. The operator mentioned rapid movement, so maybe there's an issue with the motor, en

In [158]:
print("=" * 100)
print("APEX V3 — EVIDENCE-GROUNDED GENERATION TEST")
print("=" * 100)

comparison_apex.eval()

evidence = """
DOCUMENT: DeltaWorks Industries — Delta DX-200 Automated Press Brake
DOCUMENT ID: DW-MAN-DX200-REV2
FIRMWARE: 5.0.4

FAULT F001 — DC Link Voltage Too High

Documented possible causes:
1. Braking resistor or cable open/failed.
2. BG.DECEL parameter set too low.
3. Mains voltage above the permitted range.

Documented diagnostic procedure:
1. Measure mains voltage at the main isolator terminals.
   Required range: 360–440 V AC.
2. Apply LOTO before electrical checks.
3. Measure the braking resistor between R+ and RB.
   Nominal value: 47 Ω ±10%.
4. If the resistor is healthy, increase BG.DECEL by 25%.
5. If the fault persists with normal mains voltage and a healthy resistor,
   request a DC link capacitor capacitance test from DeltaWorks service.

Safety:
Follow the documented isolation and LOTO requirements before electrical
measurements or component checks.
"""

test_query = """
The Delta DX-200 shows fault F001 during rapid backgauge movement.
Using ONLY the maintenance evidence provided above, explain:
1. what F001 means,
2. what should be checked first,
3. how to distinguish the documented possible causes,
4. what to do if the first checks are normal.

Do not introduce causes, components, measurements, or procedures that are
not present in the evidence.
"""

messages = [
    {
        "role": "system",
        "content": (
            "You are APEX, an industrial troubleshooting reasoning model. "
            "The maintenance evidence is the source of truth. "
            "Reason only from the supplied evidence. "
            "Distinguish documented facts from hypotheses. "
            "Do not invent machine facts, components, causes, measurements, "
            "or procedures. Give a concise evidence-grounded answer."
        )
    },
    {
        "role": "user",
        "content": (
            "MAINTENANCE EVIDENCE:\n"
            + evidence
            + "\n\nUSER QUERY:\n"
            + test_query
        )
    }
]

prompt = apex_v3_tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

test_inputs = apex_v3_tokenizer(
    prompt,
    return_tensors="pt"
).to(comparison_apex.device)

with torch.no_grad():
    output_ids = comparison_apex.generate(
        **test_inputs,
        max_new_tokens=500,
        do_sample=False,
        use_cache=False,
        return_dict_in_generate=True,
    )

generated_ids = output_ids.sequences[0][test_inputs["input_ids"].shape[1]:]

generated_text = apex_v3_tokenizer.decode(
    generated_ids,
    skip_special_tokens=False
)

print("\n" + "-" * 100)
print("APEX V3 OUTPUT")
print("-" * 100)

print(generated_text)

print("\n" + "-" * 100)
print("GENERATION STATS")
print("-" * 100)

print("Generated tokens:", len(generated_ids))
print("Contains <think>:", "<think>" in generated_text)
print("Contains </think>:", "</think>" in generated_text)
print("Contains <|im_end|>:", "<|im_end|>" in generated_text)

print("\n" + "=" * 100)
print("EVIDENCE-GROUNDED TEST COMPLETE")
print("=" * 100)

APEX V3 — EVIDENCE-GROUNDED GENERATION TEST

----------------------------------------------------------------------------------------------------
APEX V3 OUTPUT
----------------------------------------------------------------------------------------------------
<think>
Okay, let's tackle this query step by step. The user is asking about fault F001 on the Delta DX-200 automated press brake. The maintenance evidence provided includes the document ID, firmware version, the fault description, possible causes, diagnostic procedure, and safety notes.

First, I need to answer the four parts of the query based solely on the given evidence. Let me start by recalling the information from the document.

1. **What F001 means**: The document says "DC Link Voltage Too High." So, the fault is indicating that the DC link voltage is exceeding the acceptable range. The user needs to know the definition of the fault based on the document.

2. **What should be checked first**: The diagnostic procedure lis

In [159]:
print("=" * 100)
print("BASE vs APEX — EVIDENCE-GROUNDED GENERATION")
print("=" * 100)

comparison_apex.eval()

# Adapter OFF = clean Qwen3-4B
with comparison_apex.disable_adapter():
    with torch.no_grad():
        base_output = comparison_apex.generate(
            **test_inputs,
            max_new_tokens=250,
            do_sample=False,
            use_cache=False,
        )

# Adapter ON = trained APEX V3
with torch.no_grad():
    apex_output = comparison_apex.generate(
        **test_inputs,
        max_new_tokens=250,
        do_sample=False,
        use_cache=False,
    )

base_generated = base_output[0][test_inputs["input_ids"].shape[1]:]
apex_generated = apex_output[0][test_inputs["input_ids"].shape[1]:]

base_text = apex_v3_tokenizer.decode(
    base_generated,
    skip_special_tokens=False
)

apex_text = apex_v3_tokenizer.decode(
    apex_generated,
    skip_special_tokens=False
)

print("\n" + "-" * 100)
print("BASE QWEN3-4B — ADAPTER OFF")
print("-" * 100)
print(base_text)

print("\n" + "-" * 100)
print("APEX V3 — ADAPTER ON")
print("-" * 100)
print(apex_text)

print("\n" + "-" * 100)
print("COMPARISON")
print("-" * 100)

print("Base tokens:", len(base_generated))
print("APEX tokens:", len(apex_generated))

print("\nBase has </think>:", "</think>" in base_text)
print("APEX has </think>:", "</think>" in apex_text)

print("\nBase has <|im_end|>:", "<|im_end|>" in base_text)
print("APEX has <|im_end|>:", "<|im_end|>" in apex_text)

print("\n" + "=" * 100)
print("BASE vs APEX TEST COMPLETE")
print("=" * 100)

BASE vs APEX — EVIDENCE-GROUNDED GENERATION

----------------------------------------------------------------------------------------------------
BASE QWEN3-4B — ADAPTER OFF
----------------------------------------------------------------------------------------------------
<think>
Okay, let's tackle this query step by step. The user is asking about fault F001 on the Delta DX-200 Automated Press Brake. The maintenance evidence provided includes the document ID, firmware version, the fault description, possible causes, diagnostic procedure, and safety notes.

First, I need to answer the four parts of the query based solely on the given evidence. Let me start by recalling the information from the document.

1. **What F001 means**: The document says "DC Link Voltage Too High." So, the fault is indicating that the DC link voltage is exceeding the acceptable range. The user needs to know the meaning of the fault code, which is directly stated in the document.

2. **What should be checked fi

In [161]:
print("=" * 100)
print("APEX V3 — NON-THINKING MODE TEST")
print("=" * 100)

comparison_apex.eval()

messages_nonthinking = [
    {
        "role": "system",
        "content": (
            "You are APEX, an industrial troubleshooting reasoning model. "
            "The maintenance evidence is the source of truth. "
            "Reason only from the supplied evidence. "
            "Distinguish documented facts from hypotheses. "
            "Do not invent machine facts, causes, measurements, or procedures. "
            "Give a concise evidence-grounded troubleshooting answer."
        ),
    },
    {
        "role": "user",
        "content": (
            "MAINTENANCE EVIDENCE:\n"
            + evidence
            + "\n\nUSER QUERY:\n"
            + test_query
        ),
    },
]

prompt_nonthinking = apex_v3_tokenizer.apply_chat_template(
    messages_nonthinking,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False,
)

nonthinking_inputs = apex_v3_tokenizer(
    prompt_nonthinking,
    return_tensors="pt",
).to(comparison_apex.device)

with torch.no_grad():
    nonthinking_output = comparison_apex.generate(
        **nonthinking_inputs,
        max_new_tokens=350,
        do_sample=False,
        use_cache=False,
    )

generated_nonthinking = nonthinking_output[0][
    nonthinking_inputs["input_ids"].shape[1]:
]

nonthinking_text = apex_v3_tokenizer.decode(
    generated_nonthinking,
    skip_special_tokens=True,
)

print("\n" + "-" * 100)
print("APEX V3 — NON-THINKING OUTPUT")
print("-" * 100)

print(nonthinking_text)

print("\n" + "-" * 100)
print("STATS")
print("-" * 100)

print("Generated tokens:", len(generated_nonthinking))
print("Contains <think>:", "<think>" in nonthinking_text)
print("Contains </think>:", "</think>" in nonthinking_text)

print("\n" + "=" * 100)
print("NON-THINKING TEST COMPLETE")
print("=" * 100)

APEX V3 — NON-THINKING MODE TEST

----------------------------------------------------------------------------------------------------
APEX V3 — NON-THINKING OUTPUT
----------------------------------------------------------------------------------------------------
1. **Fault F001 (DC Link Voltage Too High)**: This fault indicates that the DC link voltage is exceeding the acceptable range, which could be due to the listed possible causes.

2. **What should be checked first**: The first step is to measure the mains voltage at the main isolator terminals to ensure it is within the required range of 360–440 V AC.

3. **How to distinguish the documented possible causes**: 
   - If the mains voltage is outside the range, the issue is likely due to mains voltage being too high.
   - If the mains voltage is within range, the next step is to measure the braking resistor between R+ and RB to check for an open/failed resistor.
   - If the resistor is healthy, then the BG.DECEL parameter should b

In [162]:
print("=" * 100)
print("APEX V3 — MULTI-EVIDENCE / FAULT DISAMBIGUATION TEST")
print("=" * 100)

comparison_apex.eval()

multi_evidence = """
DOCUMENT A — Delta DX-200 Automated Press Brake

FAULT F002 — Hydraulic Pressure Low

Documented possible causes:
1. Hydraulic oil level low.
2. Hydraulic pump inlet filter clogged.
3. Pressure transducer faulty.

Diagnostic guidance:
- Check hydraulic oil level.
- Inspect the pump inlet filter.
- Verify the pressure transducer if the first checks are normal.


DOCUMENT B — Delta DX-200 Automated Press Brake

FAULT F099 — General Hydraulic System Fault

Documented possible causes:
1. Hydraulic pressure outside the permitted operating condition.
2. Hydraulic system sensor fault.
3. Hydraulic control system fault.

Diagnostic guidance:
- Check the hydraulic pressure condition.
- Verify relevant hydraulic sensors.
- If the fault remains after the documented checks, escalate to DeltaWorks service.
"""

multi_query = """
The machine is reporting F002, but the operator also says that the hydraulic
pressure warning appeared immediately before F002.

Using ONLY the supplied evidence:

1. Explain what F002 represents.
2. Explain whether the pressure warning is enough to conclude that the pump
   or pressure transducer has failed.
3. Compare F002 with F099 and explain why the two codes should not simply be
   treated as the same fault.
4. Give the safest documented diagnostic sequence.
5. Clearly distinguish documented facts from conclusions that cannot yet be made.

Do not invent components, measurements, thresholds, or procedures.
"""

messages_multi = [
    {
        "role": "system",
        "content": (
            "You are APEX, an industrial troubleshooting reasoning model. "
            "Maintenance evidence is the source of truth. "
            "Reason across all supplied evidence. "
            "Compare competing interpretations when necessary. "
            "Distinguish documented facts from conclusions. "
            "Do not invent machine facts, causes, measurements, or procedures. "
            "Give a concise evidence-grounded troubleshooting answer."
        ),
    },
    {
        "role": "user",
        "content": (
            "MAINTENANCE EVIDENCE:\n"
            + multi_evidence
            + "\n\nUSER QUERY:\n"
            + multi_query
        ),
    },
]

prompt_multi = apex_v3_tokenizer.apply_chat_template(
    messages_multi,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False,
)

multi_inputs = apex_v3_tokenizer(
    prompt_multi,
    return_tensors="pt",
).to(comparison_apex.device)

with torch.no_grad():
    multi_output = comparison_apex.generate(
        **multi_inputs,
        max_new_tokens=400,
        do_sample=False,
        use_cache=False,
    )

multi_generated = multi_output[0][
    multi_inputs["input_ids"].shape[1]:
]

multi_text = apex_v3_tokenizer.decode(
    multi_generated,
    skip_special_tokens=True,
)

print("\n" + "-" * 100)
print("APEX V3 — MULTI-EVIDENCE OUTPUT")
print("-" * 100)

print(multi_text)

print("\n" + "-" * 100)
print("STATS")
print("-" * 100)

print("Generated tokens:", len(multi_generated))
print("Contains <think>:", "<think>" in multi_text)
print("Contains </think>:", "</think>" in multi_text)

print("\n" + "=" * 100)
print("MULTI-EVIDENCE TEST COMPLETE")
print("=" * 100)

APEX V3 — MULTI-EVIDENCE / FAULT DISAMBIGUATION TEST

----------------------------------------------------------------------------------------------------
APEX V3 — MULTI-EVIDENCE OUTPUT
----------------------------------------------------------------------------------------------------
1. **F002 represents a hydraulic pressure low condition.** This is explicitly stated in Document A as the fault code for "Hydraulic Pressure Low."

2. **The pressure warning alone is not sufficient to conclude that the pump or pressure transducer has failed.** The pressure warning is a symptom, not a definitive diagnosis. The documented possible causes for F002 include a low oil level, a clogged pump inlet filter, or a faulty pressure transducer. The operator's observation of the warning appearing before F002 suggests a possible issue, but without further diagnostic checks, it cannot be concluded that the pump or transducer has failed.

3. **F002 and F099 are distinct fault codes with different implicat

In [163]:
print("=" * 100)
print("APEX V3 — INSUFFICIENT EVIDENCE / REFUSAL TEST")
print("=" * 100)

comparison_apex.eval()

refusal_evidence = """
DOCUMENT: DeltaWorks Industries — Delta DX-200 Automated Press Brake

FAULT F001 — DC Link Voltage Too High

Documented possible causes:
1. Braking resistor or cable open/failed.
2. BG.DECEL parameter set too low.
3. Mains voltage above the permitted range.

The documentation does not establish which of these causes is present
without performing the documented diagnostic checks.
"""

refusal_query = """
The machine has fault F001. The operator insists that the braking resistor
is definitely failed and wants me to replace it immediately.

Using ONLY the supplied evidence, should the braking resistor be replaced
immediately? Explain what is known, what is not known, and what diagnostic
evidence is required before identifying the failed component.

Do not invent any additional test results.
"""

messages_refusal = [
    {
        "role": "system",
        "content": (
            "You are APEX, an industrial troubleshooting reasoning model. "
            "The supplied maintenance evidence is the source of truth. "
            "Do not treat an operator assumption as a confirmed diagnosis. "
            "Distinguish documented facts from hypotheses. "
            "If the evidence is insufficient to identify a failed component, "
            "say so explicitly and state the documented next diagnostic step. "
            "Do not invent machine facts, measurements, causes, or procedures."
        ),
    },
    {
        "role": "user",
        "content": (
            "MAINTENANCE EVIDENCE:\n"
            + refusal_evidence
            + "\n\nUSER QUERY:\n"
            + refusal_query
        ),
    },
]

prompt_refusal = apex_v3_tokenizer.apply_chat_template(
    messages_refusal,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False,
)

refusal_inputs = apex_v3_tokenizer(
    prompt_refusal,
    return_tensors="pt",
).to(comparison_apex.device)

with torch.no_grad():
    refusal_output = comparison_apex.generate(
        **refusal_inputs,
        max_new_tokens=300,
        do_sample=False,
        use_cache=False,
    )

refusal_generated = refusal_output[0][
    refusal_inputs["input_ids"].shape[1]:
]

refusal_text = apex_v3_tokenizer.decode(
    refusal_generated,
    skip_special_tokens=True,
)

print("\n" + "-" * 100)
print("APEX V3 — REFUSAL OUTPUT")
print("-" * 100)

print(refusal_text)

print("\n" + "-" * 100)
print("STATS")
print("-" * 100)

print("Generated tokens:", len(refusal_generated))
print("Contains <think>:", "<think>" in refusal_text)
print("Contains </think>:", "</think>" in refusal_text)

print("\n" + "=" * 100)
print("REFUSAL TEST COMPLETE")
print("=" * 100)

APEX V3 — INSUFFICIENT EVIDENCE / REFUSAL TEST

----------------------------------------------------------------------------------------------------
APEX V3 — REFUSAL OUTPUT
----------------------------------------------------------------------------------------------------
The operator's assertion that the braking resistor is "definitely failed" is an assumption, not a confirmed diagnosis. Based on the supplied evidence, the following is known and not known:

### **Known:**
- Fault F001 — DC Link Voltage Too High is present.
- The documented possible causes include:
  1. Braking resistor or cable open/failed.
  2. BG.DECEL parameter set too low.
  3. Mains voltage above the permitted range.

### **Not Known:**
- The exact cause of the fault is not determined.
- The operator's assumption that the braking resistor is "definitely failed" is not confirmed by documented diagnostic checks.
- The status of the braking resistor (open/failed) is not confirmed.
- The BG.DECEL parameter value is